# HOI-TRUST

## Dependency-safe hospital-onset infection surveillance prioritisation

**Evidence status:** retrospective model development and internal validation only.  
**Clinical use:** not authorised.  
**Primary analysis:** fold-contained stacking, dependency-component uncertainty, cross-fitted calibration, and prespecified sensitivity analyses.


## Public release notice

This distribution preserves the definitive code-cell sources but removes all saved execution outputs and interactive widget state. The removal prevents redistribution of restricted record-level derived outputs. The source data are not included; obtain authorised access and configure the paths described in the repository README before execution.


## Run instructions

This notebook contains the complete HOI-TRUST retrospective analysis pipeline.

1. Save this file as `MSc_Project_Private_Data/HOI_TRUST_submission_ready.ipynb`.
2. Use **Python 3.13.16**. The environment bootstrap cell installs the exact direct package versions and stops when a runtime restart is required.
3. After package installation, restart the runtime, reconnect the private Drive, and select **Runtime → Run all**.
4. The first run creates an environment- and code-specific modelling cache. Interrupted analyses may resume only from entries with matching data, split, feature, model, Python, package-lock, resolved-environment, and code fingerprints.
5. Keep `FAST_MODE = False` and `HOI_TRUST_PROFILE=definitive` for reported results.
6. A successful final audit writes the tables, figures, confidence intervals, sensitivity results, models, environment freeze, and run manifest.
7. After saving the completed notebook, run the manual hash cell once and do not save the notebook again. The external manifest then records the exact saved bytes.

The compatibility partition was inspected during development and is therefore non-confirmatory. The notebook supports retrospective research evaluation only and does not establish clinical readiness.


In [ ]:
# Exact environment lock and bootstrap

import hashlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import sys

PINNED_PYTHON_VERSION = "3.13.16"
PINNED_REQUIREMENTS = {
    "numpy": "2.2.6",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "scipy": "1.15.3",
    "matplotlib": "3.10.0",
    "seaborn": "0.13.2",
    "xgboost": "3.0.2",
    "plotly": "6.1.2",
    "ipywidgets": "8.1.7",
    "joblib": "1.5.1",
    "catboost": "1.2.8",
    "interpret": "0.7.2",
    "torch": "2.6.0",
}

REQUIREMENTS_LOCK_TEXT = "\n".join(
    f"{name}=={version}" for name, version in sorted(PINNED_REQUIREMENTS.items())
) + "\n"
PACKAGE_LOCK_SHA256 = hashlib.sha256(REQUIREMENTS_LOCK_TEXT.encode()).hexdigest()


def _installed_version(distribution):
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return None


if platform.python_version() != PINNED_PYTHON_VERSION:
    raise RuntimeError(
        f"This analysis is pinned to Python {PINNED_PYTHON_VERSION}; "
        f"the current interpreter is {platform.python_version()}. Create/select the pinned "
        "runtime before continuing. Do not change the constant merely to bypass this check."
    )

package_mismatches = {
    name: {"required": version, "installed": _installed_version(name)}
    for name, version in PINNED_REQUIREMENTS.items()
    if _installed_version(name) != version
}
if package_mismatches:
    lock_path = "/tmp/hoi_trust_requirements_lock.txt"
    with open(lock_path, "w", encoding="utf-8") as handle:
        handle.write(REQUIREMENTS_LOCK_TEXT)
    print(json.dumps(package_mismatches, indent=2))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--upgrade", "--requirement", lock_path,
    ])
    raise SystemExit(
        "Pinned packages were installed. Restart the runtime, then run the notebook from "
        "the beginning so no previously imported binary package remains in memory."
    )

print("Pinned direct environment is present.")
print("Direct package-lock SHA-256:", PACKAGE_LOCK_SHA256)


In [ ]:
# Fail-closed environment audit and full resolved-environment fingerprint

import importlib.metadata as importlib_metadata
import platform
import sys

import pandas as pd
from IPython.display import display


def installed_version(distribution):
    try:
        return importlib_metadata.version(distribution)
    except importlib_metadata.PackageNotFoundError:
        return None


def safe_version(distribution):
    return installed_version(distribution) or "not-installed"


environment_audit_df = pd.DataFrame([
    {
        "package": package,
        "pinned": pinned,
        "installed": installed_version(package),
    }
    for package, pinned in PINNED_REQUIREMENTS.items()
])
environment_audit_df["matches_pin"] = (
    environment_audit_df["pinned"] == environment_audit_df["installed"]
)

if platform.python_version() != PINNED_PYTHON_VERSION:
    raise RuntimeError("Python version changed after the bootstrap cell.")
if not environment_audit_df["matches_pin"].all():
    display(environment_audit_df)
    raise RuntimeError(
        "The runtime does not match the pinned direct environment. No modelling cell may run."
    )

FROZEN_ENVIRONMENT_LINES = sorted(
    f"{distribution.metadata.get('Name', 'unknown')}=={distribution.version}"
    for distribution in importlib_metadata.distributions()
)
FULL_ENVIRONMENT_TEXT = "\n".join(FROZEN_ENVIRONMENT_LINES) + "\n"
FULL_ENVIRONMENT_SHA256 = hashlib.sha256(FULL_ENVIRONMENT_TEXT.encode()).hexdigest()

display(environment_audit_df)
print("Python:", sys.version.replace("\n", " "))
print("Resolved environment SHA-256:", FULL_ENVIRONMENT_SHA256)


In [ ]:
# Project configuration

import os
import random
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display

RANDOM_STATE = 42
DEFINITIVE_SPLIT_SEED = 20260803

# Smoke mode is only for code-path checks. Reported results require False and
# HOI_TRUST_PROFILE=definitive.
FAST_MODE = False
STRICT_RELEASE_COUNTS = True
RUN_DASHBOARD = True

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_NAME = "HOI-TRUST"
PROJECT_VERSION = "3.1.0"
EXPECTED_CODE_SHA256 = "89a2d6e1875e10913bbdcd03f04c32c155034a4618b992e51fb236f51c27c8df"

print(PROJECT_NAME, PROJECT_VERSION)
print("Run mode:", "smoke test" if FAST_MODE else "full internal-validation analysis")
print("Code-version SHA-256:", EXPECTED_CODE_SHA256)


## 1. Data provenance and frozen cohort


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

BASE_DIR = Path(os.environ.get(
    "HOI_TRUST_DATA_DIR",
    "/content/drive/MyDrive/MSc_Project_Private_Data",
)).expanduser()
OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
METADATA_DIR = OUTPUT_DIR / "metadata"
NOTEBOOK_PATH = Path(os.environ.get(
    "HOI_TRUST_NOTEBOOK_PATH", BASE_DIR / "HOI_TRUST_submission_ready.ipynb"
)).expanduser()

for directory in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

OFFICIAL_RUNS = [
    "DS_E_coli", "ESBL_E_coli", "DS_K_pneumoniae",
    "ESBL_K_pneumoniae", "C_diff", "VSE_faecalis",
    "VRE_faecium", "MSSA", "MRSA", "DS_P_aeruginosa",
    "DR_P_aeruginosa",
]

EXPECTED_RELEASE_COUNTS = pd.DataFrame([
    {"analysis": "patient", "observations": 56_305, "cases": 16_933},
    {"analysis": "environmental", "observations": 43_403, "cases": 14_923},
])
STRICT_RELEASE_COUNTS = True


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


official_paths = {
    "environmental": BASE_DIR / "HO_infxn_environmental_analysis.csv",
    "patient": BASE_DIR / "HO_infxn_patient_analysis.csv",
}
combined_path = BASE_DIR / "HO_infxn_analysis.csv"

if all(path.exists() for path in official_paths.values()):
    source_files = []
    frames = []
    for pathway, path in official_paths.items():
        frame = pd.read_csv(path, low_memory=False)
        if "analysis" not in frame.columns:
            frame["analysis"] = pathway
        frames.append(frame)
        source_files.append(path)
    df_raw = pd.concat(frames, ignore_index=True)
elif combined_path.exists():
    source_files = [combined_path]
    df_raw = pd.read_csv(combined_path, low_memory=False)
else:
    expected = [str(path) for path in official_paths.values()] + [str(combined_path)]
    raise FileNotFoundError("No expected PhysioNet data file was found:\n" + "\n".join(expected))

RAW_COLUMNS = df_raw.columns.tolist()
RAW_COLUMN_COUNT = len(RAW_COLUMNS)

required = {
    "analysis", "run", "group_binary", "group_index", "deidentified_patient_id",
    "age", "sex", "any_surgery",
}
missing = sorted(required - set(df_raw.columns))
if missing:
    raise KeyError(f"Required source columns are missing: {missing}")

df_raw["analysis"] = df_raw["analysis"].astype("string").str.strip().str.lower()
unexpected_analyses = sorted(set(df_raw["analysis"].dropna()) - {"patient", "environmental"})
if unexpected_analyses:
    raise ValueError(f"Unexpected analysis pathway labels: {unexpected_analyses}")

df_raw["group_binary"] = pd.to_numeric(df_raw["group_binary"], errors="raise").astype(int)
if not set(df_raw["group_binary"].unique()).issubset({0, 1}):
    raise ValueError("group_binary contains values outside {0, 1}.")

excluded_run_summary_df = (
    df_raw.loc[~df_raw["run"].isin(OFFICIAL_RUNS)]
    .groupby(["analysis", "run"], dropna=False)
    .agg(observations=("group_binary", "size"), cases=("group_binary", "sum"))
    .reset_index()
)

# Primary analysis is frozen to the documented 11-target release.
df = df_raw.loc[df_raw["run"].isin(OFFICIAL_RUNS)].copy().reset_index(drop=True)
analysis_df = df

release_count_df = (
    df.groupby("analysis")
    .agg(observations=("group_binary", "size"), cases=("group_binary", "sum"))
    .reset_index()
    .sort_values("analysis")
    .reset_index(drop=True)
)
expected_sorted = EXPECTED_RELEASE_COUNTS.sort_values("analysis").reset_index(drop=True)

# Compare values strictly while ignoring harmless pandas dtype differences
# (for example, StringDtype for observed labels versus object dtype in the
# hand-written expected table). DataFrame.equals() incorrectly rejects that
# situation even when every displayed value is identical.
try:
    pd.testing.assert_frame_equal(
        release_count_df,
        expected_sorted,
        check_dtype=False,
        check_index_type=False,
        check_column_type=False,
    )
    counts_match = True
except AssertionError:
    counts_match = False

if STRICT_RELEASE_COUNTS and not counts_match:
    comparison = expected_sorted.merge(
        release_count_df, on="analysis", how="outer", suffixes=("_expected", "_observed")
    )
    display(comparison)
    raise AssertionError(
        "The primary analysis does not reproduce the documented PhysioNet v1.0.0 counts. "
        "Stop and resolve provenance before modelling."
    )

provenance_df = pd.DataFrame([
    {
        "file": path.name,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "loaded_utc": datetime.now(timezone.utc).isoformat(),
    }
    for path in source_files
])

provenance_df.to_csv(METADATA_DIR / "input_file_provenance.csv", index=False)
release_count_df.to_csv(METADATA_DIR / "official_release_count_check.csv", index=False)
excluded_run_summary_df.to_csv(METADATA_DIR / "excluded_nonrelease_runs.csv", index=False)

print(f"Raw file shape: {df_raw.shape}")
print(f"Frozen primary-analysis shape: {df.shape}")
print(f"Official pathogen targets: {df['run'].nunique()}")
display(release_count_df)
display(provenance_df)
if not excluded_run_summary_df.empty:
    print("Excluded non-release run values:")
    display(excluded_run_summary_df)


## 2. Dependency construction and feature governance


In [ ]:
from collections import defaultdict

TARGET = "group_binary"
OUTCOME_DISPLAY_NAME = "Recorded target-organism acquisition/detection during the documented follow-up window"
INTENDED_USE = "Retrospective hospital-onset infection surveillance prioritisation for human review"

df["age_clean"] = pd.to_numeric(
    df["age"].astype("string").str.replace(">", "", regex=False).str.replace("+", "", regex=False),
    errors="coerce",
)

design_columns = ["analysis", "run", "group_index", "deidentified_patient_id"]
if df[design_columns].isna().any().any():
    raise ValueError("A dependency-defining design identifier is missing.")

df["pathway_patient_id"] = (
    df["analysis"].astype(str) + "::" + df["deidentified_patient_id"].astype(str)
)
df["matched_set_id"] = (
    df["analysis"].astype(str)
    + "::" + df["run"].astype(str)
    + "::" + df["group_index"].astype(str)
)


class UnionFind:
    def __init__(self):
        self.parent = {}
        self.rank = defaultdict(int)

    def find(self, item):
        self.parent.setdefault(item, item)
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root == right_root:
            return
        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root
        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1


# A component links every repeated patient to every matched set in which they occur.
# Splitting these components prevents both patient and matched-set leakage.
component_labels = pd.Series(index=df.index, dtype="object")
for pathway, pathway_index in df.groupby("analysis").groups.items():
    uf = UnionFind()
    for patient_id, matched_id in df.loc[
        pathway_index, ["pathway_patient_id", "matched_set_id"]
    ].itertuples(index=False, name=None):
        uf.union("P::" + patient_id, "M::" + matched_id)

    roots = df.loc[pathway_index, "pathway_patient_id"].map(lambda value: uf.find("P::" + value))
    codes, _ = pd.factorize(roots, sort=True)
    component_labels.loc[pathway_index] = [
        f"{pathway}::component::{code:06d}" for code in codes
    ]

df["dependency_component"] = component_labels

# Source-faithful duplicate flag. Sensitivity analysis retains one exact source copy.
df["exact_duplicate_flag"] = df.duplicated(subset=RAW_COLUMNS, keep=False)
df["exact_duplicate_keep_flag"] = ~df.duplicated(subset=RAW_COLUMNS, keep="first")

antibiotic_cols = [column for column in RAW_COLUMNS if column.endswith("_0_60")]
antibiotic_class_cols = [column for column in antibiotic_cols if column != "any_abx_0_60"]
cp_cols = [column for column in RAW_COLUMNS if column.endswith("_cp")]
elix_cols = [column for column in RAW_COLUMNS if column.startswith("elix_")]

for column in antibiotic_cols + cp_cols + elix_cols + ["any_surgery"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Preserve the source value for audit, then use an explicit three-level modelling field.
# The single non-Male/Female value is never silently forced into a binary category.
df["sex_source_value"] = df["sex"].astype("string").str.strip()
sex_key = df["sex_source_value"].str.casefold()
df["sex"] = sex_key.map({"male": "Male", "female": "Female"}).fillna("Unknown").astype("string")
observed_nonstandard_sex = int((~sex_key.isin(["male", "female"])).sum())
if observed_nonstandard_sex != 1:
    raise AssertionError(
        f"Expected one source sex value outside Male/Female; observed {observed_nonstandard_sex}. "
        "Stop and inspect release drift rather than silently changing the rule."
    )
sex_value_audit_df = (
    df.groupby(["analysis", "sex_source_value"], dropna=False)
    .size().reset_index(name="observations")
)
sex_value_audit_df["modelled_as"] = np.where(
    sex_value_audit_df["sex_source_value"].astype("string").str.casefold().isin(["male", "female"]),
    sex_value_audit_df["sex_source_value"],
    "Unknown",
)
sex_value_audit_df.to_csv(TABLE_DIR / "sex_source_value_audit_private.csv", index=False)

# Resolve the inconsistent summary predictor from the class-specific fields. The source
# any_abx value is retained for audit but is never used as a model input.
if antibiotic_class_cols:
    class_exposure = (
        df[antibiotic_class_cols].fillna(0).gt(0).any(axis=1).astype(float)
    )
    df["any_abx_0_60_resolved"] = class_exposure
else:
    raise AssertionError("No class-specific antibiotic exposure fields were found.")

if "any_abx_0_60" in df.columns:
    source_summary = df["any_abx_0_60"].fillna(0).gt(0)
    df["antibiotic_summary_discordance"] = source_summary.ne(class_exposure.astype(bool))
else:
    df["antibiotic_summary_discordance"] = False

model_antibiotic_cols = antibiotic_class_cols + ["any_abx_0_60_resolved"]
basic_cols = ["age_clean", "sex", "any_surgery"]

FEATURE_ROLE_EXCLUSIONS = {
    "group": "Text form of the target",
    "group_binary": "Prediction target",
    "group_index": "Matched-study design identifier",
    "deidentified_patient_id": "Patient identifier used only for dependency control",
    "pathway_patient_id": "Derived patient identifier used only for dependency control",
    "matched_set_id": "Derived matched-set identifier used only for dependency control",
    "dependency_component": "Derived dependency identifier used only for partitioning",
    "time_to_infxn": "Case-only/post-outcome timing variable",
    "matching_duration": "Retrospective matching/design variable",
    "deidentified_year": "Patient-specific random date anchor; no common calendar meaning",
    "deidentified_month": "Patient-specific random date anchor; no common calendar meaning",
    "any_abx_0_60": "Inconsistent source summary retained for audit; replaced by class-derived summary",
    "sex_source_value": "Private source-value audit only; model uses Male/Female/Unknown",
}

integrity_audit_df = pd.DataFrame([
    {"check": "Rows in frozen primary analysis", "result": len(df)},
    {"check": "Original source attributes", "result": RAW_COLUMN_COUNT},
    {"check": "Exact duplicate rows (all copies flagged)", "result": int(df["exact_duplicate_flag"].sum())},
    {"check": "Rows after retaining one exact source copy", "result": int(df["exact_duplicate_keep_flag"].sum())},
    {"check": "Missing numeric age", "result": int(df["age_clean"].isna().sum())},
    {"check": "Age below 18", "result": int(df["age_clean"].lt(18).sum())},
    {"check": "Age above 110", "result": int(df["age_clean"].gt(110).sum())},
    {"check": "Source sex outside Male/Female", "result": observed_nonstandard_sex},
    {"check": "Modelled sex Unknown", "result": int(df["sex"].eq("Unknown").sum())},
    {"check": "any_abx versus class-specific discordance", "result": int(df["antibiotic_summary_discordance"].sum())},
    {"check": "Composite matched sets", "result": int(df["matched_set_id"].nunique())},
    {"check": "Dependency components", "result": int(df["dependency_component"].nunique())},
])

integrity_audit_df.to_csv(TABLE_DIR / "data_integrity_audit.csv", index=False)
display(integrity_audit_df)
display(sex_value_audit_df)


In [ ]:
# Feature-role audit and modelling schema checks
feature_group_summary_df = pd.DataFrame([
    {"feature_group": "Basic demographic/clinical", "n_features": len(basic_cols), "features": ", ".join(basic_cols)},
    {"feature_group": "Prior antibiotic exposure", "n_features": len(antibiotic_cols), "features": ", ".join(antibiotic_cols)},
    {"feature_group": "Colonisation pressure", "n_features": len(cp_cols), "features": ", ".join(cp_cols)},
    {"feature_group": "Elixhauser variables", "n_features": len(elix_cols), "features": ", ".join(elix_cols)},
    {"feature_group": "Pathogen context", "n_features": 1, "features": "run"},
])

excluded_feature_roles_df = pd.DataFrame([
    {"column": column, "exclusion_reason": reason, "present": column in df.columns}
    for column, reason in FEATURE_ROLE_EXCLUSIONS.items()
])

if not antibiotic_cols:
    raise AssertionError("No antibiotic exposure fields ending in _0_60 were found.")
if not cp_cols:
    raise AssertionError("No colonisation-pressure fields ending in _cp were found.")
if not elix_cols:
    raise AssertionError("No Elixhauser fields beginning elix_ were found.")

feature_group_summary_df.to_csv(TABLE_DIR / "feature_group_summary.csv", index=False)
excluded_feature_roles_df.to_csv(TABLE_DIR / "excluded_predictor_role_audit.csv", index=False)
display(feature_group_summary_df)
display(excluded_feature_roles_df)


## 3. Matched-design and source-release audit


In [ ]:
import warnings

matched_set_df = (
    df.groupby(["analysis", "run", "group_index", "matched_set_id"], dropna=False)
    .agg(
        observations=(TARGET, "size"),
        cases=(TARGET, "sum"),
        unique_patients=("pathway_patient_id", "nunique"),
    )
    .reset_index()
)
matched_set_df["controls"] = matched_set_df["observations"] - matched_set_df["cases"]
matched_set_df["valid_one_case"] = matched_set_df["cases"].eq(1)
matched_set_df["valid_control_count"] = matched_set_df["controls"].between(1, 3)
matched_set_df["documented_structure_valid"] = (
    matched_set_df["valid_one_case"] & matched_set_df["valid_control_count"]
)
matched_set_df["irregularity_reason"] = np.select(
    [
        ~matched_set_df["valid_one_case"] & ~matched_set_df["valid_control_count"],
        ~matched_set_df["valid_one_case"],
        ~matched_set_df["valid_control_count"],
    ],
    [
        "Case-count and control-count deviation",
        "Case-count deviation",
        "Control-count deviation",
    ],
    default="Nominal documented structure",
)

invalid_matched_sets_df = matched_set_df.loc[
    ~matched_set_df["documented_structure_valid"]
].copy()
invalid_matched_sets_df.to_csv(
    TABLE_DIR / "invalid_composite_matched_sets.csv", index=False
)

EXPECTED_IRREGULAR_MATCHED_SETS = 6
if len(invalid_matched_sets_df) != EXPECTED_IRREGULAR_MATCHED_SETS:
    raise AssertionError(
        f"Expected {EXPECTED_IRREGULAR_MATCHED_SETS} irregular matched sets in the frozen release; "
        f"observed {len(invalid_matched_sets_df)}. Stop and inspect provenance drift."
    )

# Preserve every official release row. Attach a transparent audit flag rather than
# silently deleting or relabelling source records.
matched_valid_lookup = matched_set_df.set_index("matched_set_id")[
    "documented_structure_valid"
]
matched_reason_lookup = matched_set_df.set_index("matched_set_id")[
    "irregularity_reason"
]
df["matched_set_design_valid"] = (
    df["matched_set_id"].map(matched_valid_lookup).fillna(False).astype(bool)
)
df["matched_set_irregularity_reason"] = (
    df["matched_set_id"].map(matched_reason_lookup).fillna("Unresolved matched-set identifier")
)
analysis_df = df

MATCHED_SET_IRREGULARITY_POLICY = (
    "Retain official-release rows; flag deviations; use matched-set and patient "
    "dependencies for group-disjoint partitioning; do not use conditional matched-set inference."
)

matched_design_diagnostics_df = (
    matched_set_df.groupby("analysis")
    .agg(
        matched_sets=("matched_set_id", "nunique"),
        median_set_size=("observations", "median"),
        maximum_set_size=("observations", "max"),
        sets_without_exactly_one_case=("valid_one_case", lambda values: int((~values).sum())),
        sets_outside_one_to_three_controls=("valid_control_count", lambda values: int((~values).sum())),
        irregular_sets=("documented_structure_valid", lambda values: int((~values).sum())),
    )
    .reset_index()
)
matched_design_diagnostics_df["irregular_set_fraction"] = (
    matched_design_diagnostics_df["irregular_sets"]
    / matched_design_diagnostics_df["matched_sets"]
)

patient_multiplicity_df = (
    df.groupby(["analysis", "pathway_patient_id"])
    .size().reset_index(name="observations_per_patient")
)
pathway_identity_df = (
    df.groupby("analysis")
    .agg(
        observations=(TARGET, "size"),
        cases=(TARGET, "sum"),
        pathway_specific_patients=("pathway_patient_id", "nunique"),
        matched_sets=("matched_set_id", "nunique"),
        dependency_components=("dependency_component", "nunique"),
    )
    .reset_index()
)

# Recompute the irregular-set count by pathway directly for clarity and robustness.
irregular_by_pathway = (
    invalid_matched_sets_df.groupby("analysis")["matched_set_id"]
    .nunique().rename("irregular_matched_sets")
)
pathway_identity_df = pathway_identity_df.drop(
    columns="irregular_matched_sets", errors="ignore"
).merge(irregular_by_pathway, on="analysis", how="left")
pathway_identity_df["irregular_matched_sets"] = (
    pathway_identity_df["irregular_matched_sets"].fillna(0).astype(int)
)

irregular_set_fraction_overall = (
    len(invalid_matched_sets_df) / len(matched_set_df) if len(matched_set_df) else 0.0
)

dataset_overview_df = pd.DataFrame([
    {"characteristic": "Primary-analysis observations", "value": f"{len(df):,}"},
    {"characteristic": "Original source attributes", "value": f"{RAW_COLUMN_COUNT:,}"},
    {"characteristic": "Pathogen targets", "value": f"{df['run'].nunique():,}"},
    {"characteristic": "Patient-pathway unique IDs", "value": f"{df.loc[df.analysis.eq('patient'), 'pathway_patient_id'].nunique():,}"},
    {"characteristic": "Environmental-pathway unique IDs", "value": f"{df.loc[df.analysis.eq('environmental'), 'pathway_patient_id'].nunique():,}"},
    {"characteristic": "Cross-pathway unique patients", "value": "Not identifiable: IDs were generated independently by pathway"},
    {"characteristic": "Composite matched sets", "value": f"{df['matched_set_id'].nunique():,}"},
    {"characteristic": "Matched sets outside nominal structure", "value": f"{len(invalid_matched_sets_df):,} ({irregular_set_fraction_overall:.4%})"},
    {"characteristic": "Matched-set irregularity policy", "value": MATCHED_SET_IRREGULARITY_POLICY},
    {"characteristic": "Exact duplicated rows", "value": f"{df.duplicated(subset=RAW_COLUMNS).sum():,}"},
])

display(dataset_overview_df)
display(pathway_identity_df)
display(matched_design_diagnostics_df)

if not invalid_matched_sets_df.empty:
    print("Source-release matched sets outside the nominal documented structure:")
    display(invalid_matched_sets_df)
    print(
        f"AUDITED SOURCE DEVIATION: {len(invalid_matched_sets_df):,} composite matched sets "
        "fall outside the nominal one-case/one-to-three-control structure. They remain in "
        "the primary release-faithful cohort and are removed in a prespecified sensitivity."
    )
else:
    print("All composite matched sets follow the nominal documented structure.")

dataset_overview_df.to_csv(TABLE_DIR / "table_dataset_overview_corrected.csv", index=False)
pathway_identity_df.to_csv(TABLE_DIR / "pathway_identity_summary.csv", index=False)
matched_design_diagnostics_df.to_csv(TABLE_DIR / "table_matched_design_diagnostics_corrected.csv", index=False)
matched_set_df.to_csv(TABLE_DIR / "appendix_composite_matched_set_details.csv", index=False)
patient_multiplicity_df.to_csv(TABLE_DIR / "appendix_patient_observation_multiplicity_corrected.csv", index=False)

# Deliberately no fatal assertion here: the source release is retained and audited.
# The subsequent component-based split still prevents matched-set and repeated-patient leakage.


## 4. Aggregate exploratory analysis


In [ ]:
# Aggregate-only descriptive outputs; no row-level source records are displayed.
eda_df = df.copy()

binary_elix_cols = []
for column in elix_cols:
    values = set(pd.to_numeric(eda_df[column], errors="coerce").dropna().unique().tolist())
    if values.issubset({0, 1}):
        binary_elix_cols.append(column)

eda_df["antibiotic_classes_exposed"] = (
    eda_df[antibiotic_class_cols].fillna(0).gt(0).sum(axis=1)
    if antibiotic_class_cols else 0
)
eda_df["colonisation_pressure_mean"] = eda_df[cp_cols].mean(axis=1, skipna=True)
eda_df["colonisation_pressure_max"] = eda_df[cp_cols].max(axis=1, skipna=True)
eda_df["elixhauser_comorbidity_count"] = (
    eda_df[binary_elix_cols].fillna(0).gt(0).sum(axis=1)
    if binary_elix_cols else np.nan
)

pathway_target_summary_df = (
    eda_df.groupby("analysis")[TARGET]
    .agg(observations="size", cases="sum", case_proportion="mean")
    .reset_index()
)
pathogen_pathway_summary_df = (
    eda_df.groupby(["analysis", "run"])
    .agg(observations=(TARGET, "size"), cases=(TARGET, "sum"),
         pathway_specific_patients=("pathway_patient_id", "nunique"))
    .reset_index()
)
pathogen_pathway_summary_df["controls"] = (
    pathogen_pathway_summary_df["observations"] - pathogen_pathway_summary_df["cases"]
)
pathogen_pathway_summary_df["case_proportion"] = (
    pathogen_pathway_summary_df["cases"] / pathogen_pathway_summary_df["observations"]
)

age_summary_df = (
    eda_df.groupby(["analysis", TARGET])["age_clean"]
    .agg(n="count", median="median", q1=lambda x: x.quantile(.25),
         q3=lambda x: x.quantile(.75), minimum="min", maximum="max")
    .reset_index()
)
sex_summary_df = (
    eda_df.assign(sex_display=eda_df["sex"].fillna("Missing"))
    .groupby(["analysis", "sex_display", TARGET]).size().reset_index(name="observations")
)
missingness_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).to_numpy(),
    "missing_n": df.isna().sum().to_numpy(),
    "missing_pct": (df.isna().mean() * 100).to_numpy(),
    "n_unique": df.nunique(dropna=True).to_numpy(),
}).sort_values(["missing_pct", "column"], ascending=[False, True])

for table, filename in [
    (pathway_target_summary_df, "pathway_target_summary.csv"),
    (pathogen_pathway_summary_df, "pathogen_pathway_summary.csv"),
    (age_summary_df, "age_summary_by_pathway_and_outcome.csv"),
    (sex_summary_df, "sex_summary_by_pathway_and_outcome.csv"),
    (missingness_df, "full_missingness_and_type_audit.csv"),
]:
    table.to_csv(TABLE_DIR / filename, index=False)

display(pathway_target_summary_df)
display(pathogen_pathway_summary_df)
display(age_summary_df)
display(missingness_df.head(20))


In [ ]:
# Publication-quality aggregate dataset figures
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved:", path)

# 1. Case composition in the matched analytic sample
plot_df = pathway_target_summary_df.copy()
fig, ax = plt.subplots(figsize=(7.2, 4.3))
ax.bar(plot_df["analysis"], plot_df["case_proportion"] * 100)
ax.set_ylabel("Cases in matched analytic sample (%)")
ax.set_xlabel("Analysis pathway")
ax.set_title("Outcome composition by HOI-TRUST pathway")
ax.set_ylim(0, max(40, plot_df["case_proportion"].max() * 120))
for idx, value in enumerate(plot_df["case_proportion"] * 100):
    ax.text(idx, value + 0.7, f"{value:.1f}%", ha="center")
fig.tight_layout(); save_figure(fig, "fig_dataset_pathway_case_composition.png"); plt.show()

# 2. Pathogen sample sizes
pivot = pathogen_pathway_summary_df.pivot(index="run", columns="analysis", values="observations").fillna(0)
fig, ax = plt.subplots(figsize=(9.5, 5.8))
pivot.loc[pivot.sum(axis=1).sort_values().index].plot.barh(ax=ax)
ax.set_xlabel("Observations")
ax.set_ylabel("Pathogen target")
ax.set_title("Documented 11-target cohort size by pathway")
ax.legend(title="Pathway")
fig.tight_layout(); save_figure(fig, "fig_dataset_pathogen_pathway_counts.png"); plt.show()

# 3. Age distribution (aggregate histograms only)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for pathway in ["patient", "environmental"]:
    values = eda_df.loc[eda_df["analysis"].eq(pathway), "age_clean"].dropna()
    ax.hist(values, bins=20, alpha=0.45, density=True, label=pathway)
ax.set_xlabel("Recorded age")
ax.set_ylabel("Density")
ax.set_title("Age distribution by analysis pathway")
ax.legend()
fig.tight_layout(); save_figure(fig, "fig_dataset_age_distribution.png"); plt.show()

# 4. Highest missingness fields
missing_plot = missingness_df.loc[missingness_df["missing_pct"].gt(0)].head(20).sort_values("missing_pct")
if not missing_plot.empty:
    fig, ax = plt.subplots(figsize=(9.5, 6.2))
    ax.barh(missing_plot["column"], missing_plot["missing_pct"])
    ax.set_xlabel("Missing values (%)")
    ax.set_title("Highest observed missingness (audit only)")
    fig.tight_layout(); save_figure(fig, "fig_dataset_missingness_top20.png"); plt.show()


## 5. Temporal and observation-bias boundaries


In [ ]:
temporal_validity_df = pd.DataFrame([
    {
        "candidate_analysis": "Calendar-time trend or temporal validation",
        "status": "Not valid with released dates",
        "reason": (
            "The release applies patient-specific random date anchors. De-identified year/month "
            "do not preserve a common hospital calendar and are excluded from modelling and validation."
        ),
        "action": "No temporal plot, temporal split, seasonality claim, or drift claim is produced.",
    }
])
display(temporal_validity_df)
temporal_validity_df.to_csv(TABLE_DIR / "temporal_validity_boundary.csv", index=False)


In [ ]:
observation_bias_boundary_df = pd.DataFrame([
    {
        "candidate_analysis": "Observation/testing propensity model and IPW",
        "status": "Not identifiable from the released matched analytic tables",
        "reason": (
            "The release contains selected matched cases and controls, not a denominator of all eligible "
            "patient-time or testing opportunities with an observed/not-observed indicator."
        ),
        "action": (
            "Treat recorded detection as the operational target, discuss testing dependence explicitly, "
            "and do not claim correction of observation bias."
        ),
    }
])
observation_bias_boundary_df.to_csv(TABLE_DIR / "observation_bias_identifiability_boundary.csv", index=False)
display(observation_bias_boundary_df)


## 6. Dependency-disjoint analytical partitions


In [ ]:
from itertools import combinations
from sklearn.model_selection import GroupShuffleSplit

PATHWAY_SPECS = {
    "patient": {
        "label": "Patient analysis",
        "features": basic_cols + model_antibiotic_cols + elix_cols + ["run"],
    },
    "environmental": {
        "label": "Environmental analysis",
        "features": basic_cols + cp_cols + elix_cols + ["run"],
    },
}

# New seed for the definitive post-audit split. Register it before any result is viewed.
DEFINITIVE_SPLIT_SEED = 20260803

PARTITION_ORDER = [
    "development", "probability_calibration", "operating_point",
    "conformal_calibration", "final_test",
]


def choose_balanced_component_holdout(frame, fraction, seed, trials=250):
    """Choose one group-disjoint split close to the requested size and case mix."""
    splitter = GroupShuffleSplit(n_splits=trials, test_size=fraction, random_state=seed)
    target_prevalence = frame[TARGET].mean()
    target_run_mix = frame["run"].value_counts(normalize=True)
    best = None

    for train_pos, holdout_pos in splitter.split(
        frame, frame[TARGET], groups=frame["dependency_component"]
    ):
        holdout = frame.iloc[holdout_pos]
        if holdout[TARGET].nunique() < 2:
            continue
        observed_fraction = len(holdout) / len(frame)
        run_mix = holdout["run"].value_counts(normalize=True).reindex(target_run_mix.index, fill_value=0)
        missing_run_penalty = (run_mix.eq(0)).mean()
        score = (
            abs(observed_fraction - fraction)
            + 2.0 * abs(holdout[TARGET].mean() - target_prevalence)
            + 0.25 * (run_mix - target_run_mix).abs().sum()
            + 2.0 * missing_run_penalty
        )
        if best is None or score < best[0]:
            best = (score, train_pos, holdout_pos)

    if best is None:
        raise RuntimeError("No valid dependency-component holdout split was found.")
    return best[1], best[2]


def make_five_way_partitions(frame, seed):
    labels = pd.Series(index=frame.index, dtype="object", name="partition")
    remaining = frame.copy()

    # Percentages of the complete pathway: 20%, 10%, 10%, 10%, 50%.
    for label, fraction, offset in [
        ("final_test", 0.20, 0),
        ("conformal_calibration", 0.125, 1),
        ("operating_point", 1 / 7, 2),
        ("probability_calibration", 1 / 6, 3),
    ]:
        keep_pos, take_pos = choose_balanced_component_holdout(
            remaining, fraction=fraction, seed=seed + offset
        )
        labels.loc[remaining.iloc[take_pos].index] = label
        remaining = remaining.iloc[keep_pos].copy()

    labels.loc[remaining.index] = "development"
    return labels


PATHWAY_DATA = {}
partition_summary_rows = []
partition_integrity_rows = []

for pathway_key, spec in PATHWAY_SPECS.items():
    frame = df.loc[df["analysis"].eq(pathway_key)].copy()
    needed = spec["features"] + [
        TARGET, "pathway_patient_id", "matched_set_id", "dependency_component",
        "exact_duplicate_flag",
    ]
    missing = sorted(set(needed) - set(frame.columns))
    if missing:
        raise KeyError(f"{pathway_key} is missing columns: {missing}")

    frame["partition"] = make_five_way_partitions(
        frame, seed=DEFINITIVE_SPLIT_SEED + (0 if pathway_key == "patient" else 1000)
    )
    PATHWAY_DATA[pathway_key] = {"frame": frame, **spec}

    for partition in PARTITION_ORDER:
        subset = frame.loc[frame["partition"].eq(partition)]
        partition_summary_rows.append({
            "pathway": pathway_key,
            "partition": partition,
            "observations": len(subset),
            "proportion": len(subset) / len(frame),
            "cases": (np.nan if partition == "final_test" else int(subset[TARGET].sum())),
            "case_proportion": (np.nan if partition == "final_test" else float(subset[TARGET].mean())),
            "dependency_components": subset["dependency_component"].nunique(),
            "final_labels_blinded_in_development_report": partition == "final_test",
        })

    for left, right in combinations(PARTITION_ORDER, 2):
        left_df = frame.loc[frame["partition"].eq(left)]
        right_df = frame.loc[frame["partition"].eq(right)]
        component_overlap = set(left_df["dependency_component"]) & set(right_df["dependency_component"])
        patient_overlap = set(left_df["pathway_patient_id"]) & set(right_df["pathway_patient_id"])
        matched_overlap = set(left_df["matched_set_id"]) & set(right_df["matched_set_id"])
        partition_integrity_rows.append({
            "pathway": pathway_key, "left": left, "right": right,
            "component_overlap": len(component_overlap),
            "patient_overlap": len(patient_overlap),
            "matched_set_overlap": len(matched_overlap),
        })

partition_summary_df = pd.DataFrame(partition_summary_rows)
partition_integrity_df = pd.DataFrame(partition_integrity_rows)
display(partition_summary_df)
display(partition_integrity_df)
partition_summary_df.to_csv(TABLE_DIR / "five_way_partition_summary_blinded.csv", index=False)
partition_integrity_df.to_csv(TABLE_DIR / "five_way_partition_integrity.csv", index=False)

if partition_integrity_df[["component_overlap", "patient_overlap", "matched_set_overlap"]].to_numpy().any():
    raise AssertionError("At least one dependency crossed a modelling partition.")


In [ ]:
# Partition visualisation and feature leakage assertions
all_declared_features = set()
for registry in PATHWAY_DATA.values():
    all_declared_features.update(registry["features"])
forbidden_present = sorted(all_declared_features & set(FEATURE_ROLE_EXCLUSIONS))
if forbidden_present:
    raise AssertionError(f"Forbidden predictor fields entered the feature registry: {forbidden_present}")

fig, ax = plt.subplots(figsize=(9.2, 4.8))
partition_plot = partition_summary_df.copy()
partition_plot["percent"] = partition_plot["proportion"] * 100
for pathway, group in partition_plot.groupby("pathway"):
    ordered = group.set_index("partition").reindex(PARTITION_ORDER)
    ax.plot(PARTITION_ORDER, ordered["percent"], marker="o", label=pathway)
ax.set_ylabel("Pathway observations (%)")
ax.set_title("Definitive dependency-component-disjoint partitions")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Pathway")
fig.tight_layout(); save_figure(fig, "fig_partition_proportions.png"); plt.show()


## 7. Balanced grouped validation and metric utilities


In [ ]:
from sklearn.utils.validation import check_consistent_length


class BalancedComponentKFold:
    # Deterministic binary-outcome cross-validation with intact dependency groups.
    # Repeated greedy assignment balances observations, events, non-events, and
    # component count. Every accepted validation fold contains both classes.

    def __init__(self, n_splits=5, *, shuffle=True, random_state=None, n_trials=64):
        if n_splits < 2:
            raise ValueError("n_splits must be at least 2.")
        self.n_splits = int(n_splits)
        self.shuffle = bool(shuffle)
        self.random_state = random_state
        self.n_trials = int(n_trials)

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    @staticmethod
    def _loss(fold_stats, targets):
        scaled = (fold_stats - targets) / np.maximum(targets, 1.0)
        weights = np.array([1.0, 1.5, 1.5, 0.20])
        return float(np.sum((scaled ** 2) * weights))

    def split(self, X, y, groups):
        y_array = np.asarray(y, dtype=int)
        group_array = np.asarray(groups)
        check_consistent_length(y_array, group_array)
        if pd.isna(group_array).any():
            raise ValueError("Dependency groups contain missing values.")
        if not set(np.unique(y_array)).issubset({0, 1}):
            raise ValueError("BalancedComponentKFold requires a binary outcome.")

        group_frame = pd.DataFrame({"group": group_array, "y": y_array})
        stats = group_frame.groupby("group", sort=False)["y"].agg(["size", "sum"])
        stats = stats.rename(columns={"size": "n", "sum": "events"})
        stats["nonevents"] = stats["n"] - stats["events"]
        stats["groups"] = 1.0

        if len(stats) < self.n_splits:
            raise ValueError("Fewer dependency groups than folds.")
        if stats["events"].sum() < self.n_splits or stats["nonevents"].sum() < self.n_splits:
            raise ValueError("Too few events or non-events to populate every fold.")

        values = stats[["n", "events", "nonevents", "groups"]].to_numpy(float)
        targets = values.sum(axis=0) / self.n_splits
        base_seed = 0 if self.random_state is None else int(self.random_state)
        best = None

        for trial in range(max(1, self.n_trials if self.shuffle else 1)):
            rng = np.random.default_rng(base_seed + trial)
            priority = (
                values[:, 0] / max(targets[0], 1.0)
                + np.abs(values[:, 1] - values[:, 2]) / max(targets[1] + targets[2], 1.0)
            )
            order = np.lexsort((rng.random(len(stats)), -priority))
            fold_stats = np.zeros((self.n_splits, 4), dtype=float)
            assignment = np.full(len(stats), -1, dtype=int)

            for position in order:
                candidate_losses = []
                for fold in range(self.n_splits):
                    proposed = fold_stats.copy()
                    proposed[fold] += values[position]
                    candidate_losses.append(self._loss(proposed, targets))
                minimum = min(candidate_losses)
                eligible = np.flatnonzero(np.isclose(candidate_losses, minimum))
                fold = int(rng.choice(eligible))
                assignment[position] = fold
                fold_stats[fold] += values[position]

            valid = np.all(fold_stats[:, 1] > 0) and np.all(fold_stats[:, 2] > 0)
            if not valid:
                continue
            loss = self._loss(fold_stats, targets)
            if best is None or loss < best[0]:
                best = (loss, assignment.copy(), fold_stats.copy())

        if best is None:
            raise RuntimeError(
                "No valid balanced allocation was found. Reduce n_splits or inspect component sizes."
            )

        group_to_fold = dict(zip(stats.index.to_list(), best[1]))
        row_folds = np.array([group_to_fold[group] for group in group_array], dtype=int)
        row_positions = np.arange(len(y_array))
        for fold in range(self.n_splits):
            validation = row_positions[row_folds == fold]
            training = row_positions[row_folds != fold]
            if set(group_array[training]) & set(group_array[validation]):
                raise AssertionError("Dependency-group leakage detected.")
            yield training, validation


def component_split_audit(frame, splitter, pathway, repeat):
    rows = []
    for fold, (training, validation) in enumerate(
        splitter.split(frame, frame[TARGET], frame["dependency_component"])
    ):
        subset = frame.iloc[validation]
        rows.append({
            "pathway": pathway,
            "repeat": repeat,
            "fold": fold,
            "observations": len(subset),
            "events": int(subset[TARGET].sum()),
            "nonevents": int((1 - subset[TARGET]).sum()),
            "prevalence": float(subset[TARGET].mean()),
            "dependency_components": int(subset["dependency_component"].nunique()),
            "pathogens": int(subset["run"].nunique()),
        })
    return pd.DataFrame(rows)


# Small deterministic self-test for the resampler.
_test_groups = np.repeat(np.arange(30), np.tile([2, 3, 4, 5, 6], 6))
_test_y = np.concatenate([
    np.r_[1, np.zeros(size - 1, dtype=int)] for size in np.tile([2, 3, 4, 5, 6], 6)
])
_test_splitter = BalancedComponentKFold(5, random_state=RANDOM_STATE, n_trials=32)
_test_splits = list(_test_splitter.split(np.zeros(len(_test_y)), _test_y, _test_groups))
assert len(_test_splits) == 5
assert all(_test_y[valid].sum() > 0 and (1 - _test_y[valid]).sum() > 0 for _, valid in _test_splits)
del _test_groups, _test_y, _test_splitter, _test_splits


In [ ]:
# Validation metrics and calibration helper

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score


def expected_calibration_error(y_true, probability, bins=10):
    y_true = np.asarray(y_true)
    probability = np.asarray(probability)
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(probability, edges, right=True) - 1, 0, bins - 1)
    total = len(y_true)
    return float(sum(
        (index == bin_id).sum() / total
        * abs(y_true[index == bin_id].mean() - probability[index == bin_id].mean())
        for bin_id in range(bins) if (index == bin_id).any()
    ))


def calibration_intercept_slope(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
    logit_probability = np.log(probability / (1 - probability)).reshape(-1, 1)
    calibration_model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)
    calibration_model.fit(logit_probability, y_true)
    return float(calibration_model.intercept_[0]), float(calibration_model.coef_[0, 0])


def probability_metrics(y_true, probability):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prevalence = y_true.mean()
    brier = brier_score_loss(y_true, probability)
    null_brier = prevalence * (1 - prevalence)
    auprc = average_precision_score(y_true, probability)
    intercept, slope = calibration_intercept_slope(y_true, probability)
    return {
        "AUROC": roc_auc_score(y_true, probability),
        "AUPRC": auprc,
        "AUPRC_lift": auprc / prevalence if prevalence > 0 else np.nan,
        "Brier": brier,
        "Brier_skill": 1 - brier / null_brier if null_brier > 0 else np.nan,
        "ECE_10": expected_calibration_error(y_true, probability, bins=10),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
        "prevalence": prevalence,
    }


class PlattLogitCalibrator:
    def __init__(self):
        self.model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)

    @staticmethod
    def _logit(probability):
        probability = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
        return np.log(probability / (1 - probability)).reshape(-1, 1)

    def fit(self, raw_probability, y_true):
        self.model.fit(self._logit(raw_probability), np.asarray(y_true, dtype=int))
        return self

    def predict(self, raw_probability):
        return self.model.predict_proba(self._logit(raw_probability))[:, 1]


## 8. Definitive analysis configuration


In [ ]:
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
import copy
import hashlib
import json
import math
import shutil
import time

V2_RUN_PROFILE = os.environ.get(
    "HOI_TRUST_PROFILE", "smoke" if FAST_MODE else "definitive"
)  # smoke, balanced, or definitive
V2_RUN_ADVANCED_MODELS = True
V2_RUN_MULTITASK = True
V2_RUN_PAIRWISE_RANKER = True
V2_RUN_STACK = True
V2_RUN_LEGACY_COMPATIBILITY = True
V2_RUN_EXTERNAL_VALIDATION = True
V2_RUN_SHADOW_MODE = True

PROFILE_SETTINGS = {
    "smoke": {
        "outer_repeats": 1, "outer_splits": 3, "inner_splits": 2,
        "tree_count": 100, "cat_iterations": 150, "ebm_rounds": 1000,
        "multitask_epochs": 12, "bootstrap": 100,
        "sensitivity_outer_repeats": 1,
    },
    "balanced": {
        "outer_repeats": 1, "outer_splits": 5, "inner_splits": 3,
        "tree_count": 300, "cat_iterations": 400, "ebm_rounds": 5000,
        "multitask_epochs": 40, "bootstrap": 500,
        "sensitivity_outer_repeats": 1,
    },
    "definitive": {
        "outer_repeats": 3, "outer_splits": 5, "inner_splits": 3,
        "tree_count": 500, "cat_iterations": 700, "ebm_rounds": 12000,
        "multitask_epochs": 80, "bootstrap": 2000,
        "sensitivity_outer_repeats": 1,
    },
}
if V2_RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown HOI_TRUST_PROFILE: {V2_RUN_PROFILE}")
V2_SETTINGS = PROFILE_SETTINGS[V2_RUN_PROFILE]

if V2_RUN_PROFILE == "definitive" and FAST_MODE:
    raise RuntimeError("FAST_MODE cannot be enabled for the definitive profile.")

CACHE_SCHEMA_VERSION = "strict-fold-contained-stack-v4"
STRICT_STACK_BASE_FAMILIES = tuple(V2_MODEL_GRIDS) if "V2_MODEL_GRIDS" in globals() else ()


def calculate_notebook_code_sha256(path, embedded_hash):
    """Hash code-cell sources while normalising the hash's own literal value."""
    payload = json.loads(path.read_text(encoding="utf-8"))
    chunks = []
    for cell in payload.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        text = "".join(cell.get("source", [])).replace(embedded_hash, "<SELF>")
        chunks.append(text.rstrip() + "\n")
    return hashlib.sha256("\n# ---- CELL ----\n".join(chunks).encode()).hexdigest()


if not NOTEBOOK_PATH.exists():
    raise FileNotFoundError(
        f"The definitive notebook must be saved at {NOTEBOOK_PATH} before modelling. "
        "Copy/rename it there or set HOI_TRUST_NOTEBOOK_PATH to the exact saved file."
    )
OBSERVED_CODE_SHA256 = calculate_notebook_code_sha256(NOTEBOOK_PATH, EXPECTED_CODE_SHA256)
if OBSERVED_CODE_SHA256 != EXPECTED_CODE_SHA256:
    raise RuntimeError(
        "Notebook code does not match its embedded code-version hash. Regenerate the "
        "strict notebook after any source edit; existing modelling caches are invalid."
    )

ANALYSIS_FINGERPRINT = hashlib.sha256(json.dumps({
    "project_version": PROJECT_VERSION,
    "source_sha256": sorted(provenance_df["sha256"].astype(str).tolist()),
    "split_seed": DEFINITIVE_SPLIT_SEED,
    "python": PINNED_PYTHON_VERSION,
    "package_lock_sha256": PACKAGE_LOCK_SHA256,
    "full_environment_sha256": FULL_ENVIRONMENT_SHA256,
    "code_sha256": EXPECTED_CODE_SHA256,
}, sort_keys=True).encode()).hexdigest()

V2_DIR = OUTPUT_DIR / "hoi_trust_analysis_v4"
V2_TABLE_DIR = V2_DIR / "tables"
V2_FIGURE_DIR = V2_DIR / "figures"
V2_MODEL_DIR = V2_DIR / "models"
V2_PROTOCOL_DIR = V2_DIR / "protocols"
V2_CACHE_DIR = V2_DIR / f"cache_{ANALYSIS_FINGERPRINT[:16]}"
for directory in [V2_DIR, V2_TABLE_DIR, V2_FIGURE_DIR, V2_MODEL_DIR, V2_PROTOCOL_DIR, V2_CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# One-time, recoverable removal of every pre-existing modelling cache from active use.
# Legacy directories are moved, not deleted. The new strict cache survives reruns.
CACHE_ARCHIVE_ROOT = OUTPUT_DIR / "cache_archive_before_analysis_v4"
CACHE_INITIALISED_MARKER = V2_DIR / f"analysis_cache_initialised_{ANALYSIS_FINGERPRINT[:16]}.json"
if not CACHE_INITIALISED_MARKER.exists():
    CACHE_ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    legacy_cache_dirs = []
    for candidate in OUTPUT_DIR.rglob("*"):
        if not candidate.is_dir() or "cache" not in candidate.name.casefold():
            continue
        if candidate.resolve() in {V2_CACHE_DIR.resolve(), CACHE_ARCHIVE_ROOT.resolve()}:
            continue
        if V2_CACHE_DIR.resolve() in [candidate.resolve(), *[parent.resolve() for parent in candidate.parents]]:
            continue
        if CACHE_ARCHIVE_ROOT.resolve() in [parent.resolve() for parent in candidate.parents]:
            continue
        legacy_cache_dirs.append(candidate)
    legacy_cache_dirs = sorted(set(legacy_cache_dirs), key=lambda path: len(path.parts))
    legacy_cache_dirs = [
        candidate for candidate in legacy_cache_dirs
        if not any(parent in legacy_cache_dirs for parent in candidate.parents)
    ]
    archived = []
    for legacy in legacy_cache_dirs:
        if not legacy.exists():
            continue
        relative = "__".join(legacy.relative_to(OUTPUT_DIR).parts)
        destination = CACHE_ARCHIVE_ROOT / relative
        if destination.exists():
            destination = CACHE_ARCHIVE_ROOT / f"{relative}_{int(time.time())}"
        shutil.move(str(legacy), str(destination))
        archived.append({"source": str(legacy), "archive": str(destination)})
    CACHE_INITIALISED_MARKER.write_text(json.dumps({
        "cache_schema": CACHE_SCHEMA_VERSION,
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "legacy_caches_archived": archived,
        "policy": "Earlier modelling caches are excluded from active use; resumption requires an exact fingerprint match.",
    }, indent=2), encoding="utf-8")


def stable_hash(payload, length=24):
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode()
    ).hexdigest()[:length]


def strict_cache_key(kind, **payload):
    return stable_hash({
        "kind": kind,
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "analysis_fingerprint": ANALYSIS_FINGERPRINT,
        "python": PINNED_PYTHON_VERSION,
        "package_lock_sha256": PACKAGE_LOCK_SHA256,
        "full_environment_sha256": FULL_ENVIRONMENT_SHA256,
        "code_sha256": EXPECTED_CODE_SHA256,
        **payload,
    })


def atomic_write_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)


def atomic_write_csv(path, frame):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def atomic_joblib_dump(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    joblib.dump(value, temporary)
    temporary.replace(path)


def validated_cached_fold(csv_path, json_path):
    if not (csv_path.exists() and json_path.exists()):
        return None
    metadata = json.loads(json_path.read_text(encoding="utf-8"))
    if metadata.get("analysis_fingerprint") != ANALYSIS_FINGERPRINT:
        return None
    if metadata.get("cache_schema_version") != CACHE_SCHEMA_VERSION:
        return None
    if metadata.get("code_sha256") != EXPECTED_CODE_SHA256:
        return None
    return pd.read_csv(csv_path), metadata


(V2_PROTOCOL_DIR / "requirements_direct_lock.txt").write_text(
    REQUIREMENTS_LOCK_TEXT, encoding="utf-8"
)
(V2_PROTOCOL_DIR / "requirements_frozen_full.txt").write_text(
    FULL_ENVIRONMENT_TEXT, encoding="utf-8"
)

print("Analysis profile:", V2_RUN_PROFILE)
print("Settings:", V2_SETTINGS)
print("Strict analysis fingerprint:", ANALYSIS_FINGERPRINT)
print("Verified notebook code hash:", OBSERVED_CODE_SHA256)
print("Exact-match modelling cache is active; earlier caches remain archived.")


## 9. Partition governance and split audit


In [ ]:
V2_PARTITIONS = {}
for pathway_key, registry in PATHWAY_DATA.items():
    frame = registry["frame"].copy()
    V2_PARTITIONS[pathway_key] = {
        "development": frame.loc[frame["partition"].eq("development")].copy(),
        "probability_calibration": frame.loc[frame["partition"].eq("probability_calibration")].copy(),
        "operating_point": frame.loc[frame["partition"].eq("operating_point")].copy(),
        "conformal_calibration": frame.loc[frame["partition"].eq("conformal_calibration")].copy(),
        "legacy_final_test": frame.loc[frame["partition"].eq("final_test")].copy(),
    }

v2_partition_governance_df = pd.DataFrame([
    {
        "pathway": pathway,
        "partition": partition,
        "observations": len(frame),
        "dependency_components": frame["dependency_component"].nunique(),
        "permitted_v2_use": {
            "development": "Repeated nested model development and cross-fitted stack",
            "probability_calibration": "Final probability calibration only",
            "operating_point": "Capacity and selective-abstention policy only",
            "conformal_calibration": "Pathway-level component conformal calibration only",
            "legacy_final_test": "Compatibility reporting; not model selection",
        }[partition],
    }
    for pathway, partitions in V2_PARTITIONS.items()
    for partition, frame in partitions.items()
])
v2_partition_governance_df.to_csv(V2_TABLE_DIR / "v2_partition_governance.csv", index=False)
display(v2_partition_governance_df)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

v2_split_audit_frames = []
for pathway, partitions in V2_PARTITIONS.items():
    development = partitions["development"]
    for repeat in range(V2_SETTINGS["outer_repeats"]):
        splitter = BalancedComponentKFold(
            V2_SETTINGS["outer_splits"],
            random_state=RANDOM_STATE + repeat * 1000,
            n_trials=128,
        )
        v2_split_audit_frames.append(
            component_split_audit(development, splitter, pathway, repeat)
        )

v2_split_audit_df = pd.concat(v2_split_audit_frames, ignore_index=True)
if (v2_split_audit_df[["events", "nonevents"]] <= 0).any().any():
    raise RuntimeError("At least one outer validation fold lacks an outcome class.")
v2_split_audit_df.to_csv(V2_TABLE_DIR / "development_split_audit.csv", index=False)

figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.barplot(data=v2_split_audit_df, x="fold", y="observations", hue="pathway", ax=axes[0])
sns.pointplot(data=v2_split_audit_df, x="fold", y="prevalence", hue="pathway", dodge=0.25, ax=axes[1])
axes[0].set_title("Observations per validation fold")
axes[1].set_title("Outcome prevalence per validation fold")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
figure.tight_layout()
figure.savefig(V2_FIGURE_DIR / "development_split_balance.png", dpi=300, bbox_inches="tight")
plt.show()
display(v2_split_audit_df)


## 10. Clinically structured feature engineering


In [ ]:
PATHOGEN_TO_CP = {
    "DS_E_coli": "DS_Entero_cp",
    "DS_K_pneumoniae": "DS_Entero_cp",
    "ESBL_E_coli": "ESBL_cp",
    "ESBL_K_pneumoniae": "ESBL_cp",
    "C_diff": "CDiff_cp",
    "VSE_faecalis": "VSE_cp",
    "VRE_faecium": "VRE_cp",
    "MSSA": "MSSA_cp",
    "MRSA": "MRSA_cp",
    "DS_P_aeruginosa": "DS_PsA_cp",
    "DR_P_aeruginosa": "DR_PsA_cp",
}


def safe_entropy_rows(values):
    matrix = np.asarray(values, dtype=float)
    matrix = np.where(np.isfinite(matrix) & (matrix > 0), matrix, 0.0)
    totals = matrix.sum(axis=1, keepdims=True)
    probabilities = np.divide(matrix, totals, out=np.zeros_like(matrix), where=totals > 0)
    logp = np.zeros_like(probabilities)
    positive = probabilities > 0
    logp[positive] = np.log(probabilities[positive])
    return -(probabilities * logp).sum(axis=1)


def engineer_v2_features(frame, pathway):
    engineered = frame.copy()
    engineered["age_clean"] = pd.to_numeric(engineered["age_clean"], errors="coerce")
    engineered["age_squared_scaled"] = (engineered["age_clean"] / 10.0) ** 2
    engineered["age_missing"] = engineered["age_clean"].isna().astype(int)
    engineered["age_band_v2"] = pd.cut(
        engineered["age_clean"], [-np.inf, 39, 59, 69, 79, np.inf],
        labels=["<40", "40-59", "60-69", "70-79", "80+"],
    ).astype("string").fillna("Missing")

    # Construct missingness indicators in one concat to avoid DataFrame fragmentation.
    binary_elix = []
    missing_indicators = {}
    for column in elix_cols:
        numeric = pd.to_numeric(engineered[column], errors="coerce")
        observed = set(numeric.dropna().unique().tolist())
        if observed.issubset({0, 1}):
            binary_elix.append(column)
        missing_indicators[column + "__missing"] = numeric.isna().astype(int)
    if missing_indicators:
        engineered = pd.concat([
            engineered,
            pd.DataFrame(missing_indicators, index=engineered.index),
        ], axis=1)

    engineered["elix_comorbidity_count"] = (
        engineered[binary_elix].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).sum(axis=1)
        if binary_elix else 0
    )
    cancer_columns = [c for c in elix_cols if "CANCER" in c]
    vulnerability_columns = [
        c for c in elix_cols
        if any(token in c for token in [
            "DEMENTIA", "WGHTLOSS", "RENLFL", "CANCER", "HF", "LUNG_CHRONIC"
        ])
    ]
    engineered["elix_any_cancer"] = (
        engineered[cancer_columns].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).any(axis=1).astype(int)
        if cancer_columns else 0
    )
    engineered["elix_vulnerability_burden"] = (
        engineered[vulnerability_columns].apply(pd.to_numeric, errors="coerce").fillna(0).gt(0).sum(axis=1)
        if vulnerability_columns else 0
    )

    interaction_data = {}
    if pathway == "patient":
        abx_numeric = engineered[antibiotic_class_cols].apply(
            pd.to_numeric, errors="coerce"
        ).fillna(0)
        engineered["antibiotic_classes_exposed"] = abx_numeric.gt(0).sum(axis=1)
        engineered["antibiotic_total_courses"] = abx_numeric.clip(lower=0).sum(axis=1)
        engineered["antibiotic_max_class_courses"] = abx_numeric.max(axis=1)
        engineered["antibiotic_diversity_entropy"] = safe_entropy_rows(abx_numeric.to_numpy())
        for run in OFFICIAL_RUNS:
            run_flag = engineered["run"].eq(run).astype(float)
            for column in model_antibiotic_cols:
                interaction_data[f"INT__{run}__{column}"] = (
                    pd.to_numeric(engineered[column], errors="coerce").fillna(0) * run_flag
                )
    elif pathway == "environmental":
        cp_numeric = engineered[cp_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
        engineered["cp_mean_all"] = cp_numeric.mean(axis=1)
        engineered["cp_max_all"] = cp_numeric.max(axis=1)
        engineered["cp_sd_all"] = cp_numeric.std(axis=1)
        engineered["cp_entropy"] = safe_entropy_rows(cp_numeric.to_numpy())
        engineered["target_cp"] = 0.0
        for pathogen, cp_column in PATHOGEN_TO_CP.items():
            if cp_column in engineered.columns:
                mask = engineered["run"].eq(pathogen)
                engineered.loc[mask, "target_cp"] = pd.to_numeric(
                    engineered.loc[mask, cp_column], errors="coerce"
                ).fillna(0.0)
        engineered["target_cp"] = pd.to_numeric(
            engineered["target_cp"], errors="coerce"
        ).fillna(0.0)
        engineered["target_cp_minus_mean"] = engineered["target_cp"] - engineered["cp_mean_all"]
        engineered["target_cp_share"] = engineered["target_cp"] / (
            engineered[cp_cols].sum(axis=1) + 1e-8
        )
        for run in OFFICIAL_RUNS:
            run_flag = engineered["run"].eq(run).astype(float)
            for column in cp_cols:
                interaction_data[f"INT__{run}__{column}"] = (
                    pd.to_numeric(engineered[column], errors="coerce").fillna(0) * run_flag
                )
    else:
        raise ValueError(f"Unknown pathway: {pathway}")

    if interaction_data:
        engineered = pd.concat([
            engineered,
            pd.DataFrame(interaction_data, index=engineered.index),
        ], axis=1)
    engineered.replace([np.inf, -np.inf], np.nan, inplace=True)
    return engineered


V2_ENGINEERED = {}
V2_FEATURE_SETS = {}
for pathway_key in PATHWAY_DATA:
    complete = pd.concat(V2_PARTITIONS[pathway_key].values(), axis=0).sort_index()
    engineered = engineer_v2_features(complete, pathway_key)
    V2_ENGINEERED[pathway_key] = engineered

    interaction_cols = [c for c in engineered.columns if c.startswith("INT__")]
    missing_indicator_cols = [c for c in engineered.columns if c.endswith("__missing")]
    common_compact = basic_cols + elix_cols + [
        "age_squared_scaled", "age_missing", "age_band_v2",
        "elix_comorbidity_count", "elix_any_cancer", "elix_vulnerability_burden", "run",
    ]
    if pathway_key == "patient":
        compact = common_compact + model_antibiotic_cols + [
            "antibiotic_classes_exposed", "antibiotic_total_courses",
            "antibiotic_max_class_courses", "antibiotic_diversity_entropy",
        ]
    else:
        compact = common_compact + cp_cols + [
            "cp_mean_all", "cp_max_all", "cp_sd_all", "cp_entropy",
            "target_cp", "target_cp_minus_mean", "target_cp_share",
        ]
    compact = list(dict.fromkeys([c for c in compact if c in engineered.columns]))
    full = list(dict.fromkeys(compact + interaction_cols + missing_indicator_cols))
    without_pathogen_context = [
        column for column in full
        if column != "run" and not column.startswith("INT__")
    ]
    V2_FEATURE_SETS[pathway_key] = {
        "compact": compact,
        "full": full,
        "without_pathogen_context": without_pathogen_context,
    }

v2_feature_registry_df = pd.DataFrame([
    {"pathway": p, "feature_set": name, "n_features": len(features), "features": ", ".join(features)}
    for p, sets in V2_FEATURE_SETS.items() for name, features in sets.items()
])
v2_feature_registry_df.to_csv(V2_TABLE_DIR / "v2_feature_registry.csv", index=False)
display(v2_feature_registry_df[["pathway", "feature_set", "n_features"]])


## 11. Capacity and ranking metrics


In [ ]:
DEFAULT_CAPACITY_FRACTIONS = [0.01, 0.02, 0.05, 0.10, 0.20]


def capacity_curve(y_true, score, fractions=DEFAULT_CAPACITY_FRACTIONS):
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    prevalence = y_true.mean()
    rows = []
    ordered = np.argsort(-score, kind="stable")
    for fraction in fractions:
        nominal_n = max(1, int(np.ceil(len(y_true) * fraction)))
        boundary = score[ordered[nominal_n - 1]]
        selected = score >= boundary
        actual_n = int(selected.sum())
        tp = int(y_true[selected].sum())
        precision = tp / actual_n if actual_n else np.nan
        recall = tp / y_true.sum() if y_true.sum() else np.nan
        rows.append({
            "capacity_fraction": fraction,
            "nominal_n": nominal_n,
            "actual_n_including_ties": actual_n,
            "precision_at_capacity": precision,
            "recall_at_capacity": recall,
            "enrichment_at_capacity": precision / prevalence if prevalence > 0 else np.nan,
            "number_needed_to_review": 1 / precision if precision > 0 else np.inf,
            "work_saved_vs_review_all": 1 - actual_n / len(y_true),
        })
    return pd.DataFrame(rows)


def capacity_selection_score(y_true, score, pathogen, primary_fraction=0.05):
    frame = pd.DataFrame({
        "y": np.asarray(y_true, dtype=int),
        "score": np.asarray(score, dtype=float),
        "run": np.asarray(pathogen),
    })
    selected = np.zeros(len(frame), dtype=bool)
    for _, positions in frame.groupby("run", sort=False).indices.items():
        positions = np.asarray(positions)
        local_score = frame.iloc[positions]["score"].to_numpy()
        count = max(1, int(np.ceil(len(positions) * primary_fraction)))
        boundary = np.sort(local_score)[-count]
        selected[positions] = local_score >= boundary
    events = frame["y"].sum()
    true_positives = frame.loc[selected, "y"].sum()
    precision = frame.loc[selected, "y"].mean() if selected.any() else 0.0
    recall = true_positives / events if events else 0.0
    prevalence = frame["y"].mean()
    enrichment = precision / prevalence if prevalence > 0 else 0.0
    pathogen_lifts = []
    for _, subset in frame.groupby("run", sort=False):
        if subset["y"].nunique() < 2:
            continue
        local_prevalence = subset["y"].mean()
        pathogen_lifts.append(
            average_precision_score(subset["y"], subset["score"]) / local_prevalence
        )
    auprc_lift = float(np.mean(pathogen_lifts)) if pathogen_lifts else 0.0
    return float(
        0.50 * min(auprc_lift / 2.0, 1.0)
        + 0.30 * recall
        + 0.20 * min(enrichment / 2.0, 1.0)
    )


def matched_set_ranking_metrics(frame, score_column):
    valid = frame.loc[frame["matched_set_design_valid"]].copy()
    rows = []
    for matched_set, group in valid.groupby("matched_set_id"):
        if group[TARGET].sum() != 1 or len(group) < 2:
            continue
        ordered = group.sort_values(score_column, ascending=False, kind="stable")
        case_positions = np.flatnonzero(ordered[TARGET].to_numpy() == 1)
        if not len(case_positions):
            continue
        rank = int(case_positions[0] + 1)
        rows.append({
            "matched_set_id": matched_set,
            "case_rank": rank,
            "case_ranked_first": rank == 1,
            "reciprocal_rank": 1.0 / rank,
            "set_size": len(group),
        })
    details = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "matched_sets_evaluated": len(details),
        "case_ranked_first_fraction": details["case_ranked_first"].mean() if len(details) else np.nan,
        "mean_reciprocal_rank": details["reciprocal_rank"].mean() if len(details) else np.nan,
        "median_case_rank": details["case_rank"].median() if len(details) else np.nan,
    }])
    return summary, details


## 12. Candidate model registry

The registry compares penalised logistic regression, Extra Trees, XGBoost, CatBoost and Explainable Boosting Machines under identical dependency-grouped resampling. Ranking, multi-task and stacking branches are evaluated under the same development-only evidence boundary.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from catboost import CatBoostClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from xgboost import XGBClassifier, XGBRanker


def v2_one_hot():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_v2_preprocessor(features):
    categorical = [f for f in features if f in {"sex", "run", "age_band_v2"}]
    numeric = [f for f in features if f not in categorical]
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=False)),
            ("scaler", StandardScaler()),
        ]), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", v2_one_hot()),
        ]), categorical),
    ], sparse_threshold=0)


@dataclass(frozen=True)
class V2ModelSpec:
    name: str
    feature_set: str
    params: dict


def make_v2_classifier(spec, features, seed):
    trees = V2_SETTINGS["tree_count"]
    if spec.name == "Logistic regression":
        model = LogisticRegression(
            max_iter=5000, C=spec.params.get("C", 1.0),
            class_weight=spec.params.get("class_weight"), random_state=seed,
        )
    elif spec.name == "Extra Trees":
        model = ExtraTreesClassifier(
            n_estimators=trees,
            min_samples_leaf=spec.params.get("min_samples_leaf", 20),
            max_features=spec.params.get("max_features", "sqrt"),
            class_weight=spec.params.get("class_weight"),
            n_jobs=-1, random_state=seed,
        )
    elif spec.name == "XGBoost":
        model = XGBClassifier(
            n_estimators=trees,
            max_depth=spec.params.get("max_depth", 3),
            learning_rate=spec.params.get("learning_rate", 0.04),
            min_child_weight=spec.params.get("min_child_weight", 10),
            subsample=0.85, colsample_bytree=0.85,
            reg_lambda=spec.params.get("reg_lambda", 1.0),
            reg_alpha=spec.params.get("reg_alpha", 0.0),
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=seed,
        )
    elif spec.name == "CatBoost":
        model = CatBoostClassifier(
            iterations=V2_SETTINGS["cat_iterations"],
            depth=spec.params.get("depth", 6),
            learning_rate=spec.params.get("learning_rate", 0.04),
            l2_leaf_reg=spec.params.get("l2_leaf_reg", 5.0),
            loss_function="Logloss", eval_metric="PRAUC",
            random_seed=seed, verbose=False, allow_writing_files=False,
            thread_count=-1,
        )
    elif spec.name == "Explainable Boosting Machine":
        model = ExplainableBoostingClassifier(
            interactions=spec.params.get("interactions", 10),
            outer_bags=spec.params.get("outer_bags", 8),
            learning_rate=spec.params.get("learning_rate", 0.025),
            max_rounds=V2_SETTINGS["ebm_rounds"],
            min_samples_leaf=spec.params.get("min_samples_leaf", 8),
            n_jobs=-2, random_state=seed,
        )
    else:
        raise KeyError(spec.name)
    return Pipeline([
        ("preprocessor", make_v2_preprocessor(features)),
        ("model", model),
    ])


V2_MODEL_GRIDS = {
    "Logistic regression": [
        V2ModelSpec("Logistic regression", "compact", {"C": 0.1}),
        V2ModelSpec("Logistic regression", "compact", {"C": 1.0}),
    ],
    "Extra Trees": [
        V2ModelSpec("Extra Trees", "full", {"min_samples_leaf": 10, "max_features": "sqrt"}),
        V2ModelSpec("Extra Trees", "full", {"min_samples_leaf": 25, "max_features": 0.5}),
    ],
    "XGBoost": [
        V2ModelSpec("XGBoost", "full", {"max_depth": 3, "min_child_weight": 10, "learning_rate": 0.035}),
        V2ModelSpec("XGBoost", "full", {"max_depth": 4, "min_child_weight": 20, "learning_rate": 0.025, "reg_lambda": 3.0}),
    ],
    "CatBoost": [
        V2ModelSpec("CatBoost", "full", {"depth": 6, "learning_rate": 0.04, "l2_leaf_reg": 5.0}),
        V2ModelSpec("CatBoost", "full", {"depth": 8, "learning_rate": 0.025, "l2_leaf_reg": 10.0}),
    ],
    "Explainable Boosting Machine": [
        V2ModelSpec("Explainable Boosting Machine", "compact", {"interactions": 10, "learning_rate": 0.025}),
        V2ModelSpec("Explainable Boosting Machine", "compact", {"interactions": 20, "learning_rate": 0.015}),
    ],
}


## 13. Repeated nested component-grouped validation


In [ ]:
def probability_and_capacity_metrics(y_true, probability):
    base = probability_metrics(y_true, probability)
    primary_capacity = capacity_curve(y_true, probability, [0.05]).iloc[0].to_dict()
    return {**base, **{f"cap5_{k}": v for k, v in primary_capacity.items()}}


def ranking_and_capacity_metrics(y_true, score):
    """Metrics valid for an arbitrary monotone ranking score."""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    prevalence = float(y_true.mean())
    primary_capacity = capacity_curve(y_true, score, [0.05]).iloc[0].to_dict()
    auprc = average_precision_score(y_true, score)
    return {
        "AUROC": roc_auc_score(y_true, score),
        "AUPRC": auprc,
        "AUPRC_lift": auprc / prevalence if prevalence > 0 else np.nan,
        "Brier": np.nan,
        "Brier_skill": np.nan,
        "ECE_10": np.nan,
        "calibration_intercept": np.nan,
        "calibration_slope": np.nan,
        "prevalence": prevalence,
        **{f"cap5_{k}": v for k, v in primary_capacity.items()},
    }


def inner_score_spec(frame, pathway, spec, seed, n_splits, feature_sets=None):
    feature_sets = V2_FEATURE_SETS[pathway] if feature_sets is None else feature_sets
    features = feature_sets[spec.feature_set]
    splitter = BalancedComponentKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for fold, (train_pos, valid_pos) in enumerate(
        splitter.split(frame, frame[TARGET], frame["dependency_component"])
    ):
        train = frame.iloc[train_pos]
        valid = frame.iloc[valid_pos]
        estimator = make_v2_classifier(spec, features, seed + fold)
        estimator.fit(train[features], train[TARGET])
        probability = estimator.predict_proba(valid[features])[:, 1]
        scores.append(capacity_selection_score(valid[TARGET], probability, valid["run"]))
    return float(np.mean(scores))


def repeated_nested_evaluate_model_family(pathway, family_name):
    development_indices = V2_PARTITIONS[pathway]["development"].index
    frame = V2_ENGINEERED[pathway].loc[development_indices].copy()
    specs = V2_MODEL_GRIDS[family_name]
    prediction_rows = []
    fold_rows = []

    for repeat in range(V2_SETTINGS["outer_repeats"]):
        outer_seed = RANDOM_STATE + repeat * 1000
        outer = BalancedComponentKFold(
            n_splits=V2_SETTINGS["outer_splits"], shuffle=True, random_state=outer_seed
        )
        for outer_fold, (train_pos, valid_pos) in enumerate(
            outer.split(frame, frame[TARGET], frame["dependency_component"])
        ):
            cache_key = strict_cache_key(
                "standalone_nested_family",
                pathway=pathway,
                family=family_name,
                repeat=repeat,
                outer_fold=outer_fold,
                profile=V2_RUN_PROFILE,
                outer_seed=outer_seed,
                row_index_hash=stable_hash(frame.index.astype(str).tolist()),
                feature_registry_hash=stable_hash(V2_FEATURE_SETS[pathway]),
                specs=[asdict(spec) for spec in specs],
            )
            cache_file = V2_CACHE_DIR / f"nested_{cache_key}.csv"
            meta_file = V2_CACHE_DIR / f"nested_{cache_key}.json"
            cached = validated_cached_fold(cache_file, meta_file)
            if cached is not None:
                cached_predictions, cached_metadata = cached
                prediction_rows.extend(cached_predictions.to_dict("records"))
                fold_rows.append(cached_metadata)
                continue

            outer_train = frame.iloc[train_pos]
            outer_valid = frame.iloc[valid_pos]
            if set(outer_train["dependency_component"]) & set(outer_valid["dependency_component"]):
                raise AssertionError("Outer-fold dependency leakage detected.")

            inner_results = []
            for spec in specs:
                score = inner_score_spec(
                    outer_train, pathway, spec,
                    seed=outer_seed + outer_fold * 100,
                    n_splits=V2_SETTINGS["inner_splits"],
                )
                inner_results.append((score, spec))
            inner_results.sort(key=lambda item: item[0], reverse=True)
            best_inner_score, best_spec = inner_results[0]
            features = V2_FEATURE_SETS[pathway][best_spec.feature_set]
            estimator = make_v2_classifier(best_spec, features, outer_seed + outer_fold)
            estimator.fit(outer_train[features], outer_train[TARGET])
            probability = estimator.predict_proba(outer_valid[features])[:, 1]

            fold_record = {
                "pathway": pathway,
                "family": family_name,
                "repeat": repeat,
                "outer_fold": outer_fold,
                "best_inner_score": best_inner_score,
                "best_spec": asdict(best_spec),
                "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "code_sha256": EXPECTED_CODE_SHA256,
                **probability_and_capacity_metrics(outer_valid[TARGET], probability),
            }
            current_predictions = pd.DataFrame({
                "source_index": outer_valid.index,
                "pathway": pathway,
                "family": family_name,
                "repeat": repeat,
                "outer_fold": outer_fold,
                "y_true": outer_valid[TARGET].to_numpy(),
                "probability": probability,
                "dependency_component": outer_valid["dependency_component"].to_numpy(),
            })
            atomic_write_csv(cache_file, current_predictions)
            atomic_write_json(meta_file, fold_record)
            prediction_rows.extend(current_predictions.to_dict("records"))
            fold_rows.append(fold_record)

    return pd.DataFrame(prediction_rows), pd.DataFrame(fold_rows)


V2_NESTED_PREDICTIONS = {}
V2_NESTED_FOLDS = {}
if V2_RUN_ADVANCED_MODELS:
    for pathway in PATHWAY_DATA:
        for family in V2_MODEL_GRIDS:
            print(f"Nested standalone evaluation: {pathway} / {family}")
            predictions, folds = repeated_nested_evaluate_model_family(pathway, family)
            key = (pathway, family)
            V2_NESTED_PREDICTIONS[key] = predictions
            V2_NESTED_FOLDS[key] = folds
            atomic_write_csv(V2_TABLE_DIR / f"nested_oof_{pathway}_{family.replace(' ', '_')}.csv", predictions)
            atomic_write_csv(V2_TABLE_DIR / f"nested_folds_{pathway}_{family.replace(' ', '_')}.csv", folds)

    v2_nested_summary_df = pd.concat([
        folds.assign(pathway=pathway, family=family)
        for (pathway, family), folds in V2_NESTED_FOLDS.items()
    ], ignore_index=True)
    display(v2_nested_summary_df.groupby(["pathway", "family"])[
        ["AUROC", "AUPRC", "Brier", "cap5_precision_at_capacity", "cap5_recall_at_capacity", "cap5_enrichment_at_capacity"]
    ].agg(["mean", "std"]).round(4))
else:
    print("Advanced nested model evaluation skipped by configuration.")


## 14. Matched-set ranking, multi-task learning and stacking


In [ ]:
def ranker_preprocessor(features):
    return make_v2_preprocessor(features)


def fit_xgb_pairwise_ranker(train_frame, features, params, seed):
    valid = train_frame.loc[train_frame["matched_set_design_valid"]].copy()
    valid = valid.sort_values("matched_set_id", kind="stable")
    preprocessor = ranker_preprocessor(features)
    matrix = preprocessor.fit_transform(valid[features])
    group_sizes = valid.groupby("matched_set_id", sort=False).size().to_numpy()
    ranker = XGBRanker(
        objective=params.get("objective", "rank:pairwise"),
        n_estimators=V2_SETTINGS["tree_count"],
        max_depth=params.get("max_depth", 3),
        learning_rate=params.get("learning_rate", 0.035),
        min_child_weight=params.get("min_child_weight", 5),
        subsample=0.85, colsample_bytree=0.85,
        reg_lambda=params.get("reg_lambda", 2.0),
        tree_method="hist", n_jobs=-1, random_state=seed,
    )
    ranker.fit(matrix, valid[TARGET].to_numpy(), group=group_sizes)
    return {"preprocessor": preprocessor, "ranker": ranker, "features": features}


def predict_xgb_pairwise_ranker(bundle, frame):
    matrix = bundle["preprocessor"].transform(frame[bundle["features"]])
    return np.asarray(bundle["ranker"].predict(matrix), dtype=float)


PAIRWISE_GRID = [
    {"objective": "rank:pairwise", "max_depth": 3, "learning_rate": 0.035, "min_child_weight": 5},
    {"objective": "rank:ndcg", "max_depth": 4, "learning_rate": 0.025, "min_child_weight": 10},
]


def evaluate_pairwise_ranker(pathway):
    indices = V2_PARTITIONS[pathway]["development"].index
    frame = V2_ENGINEERED[pathway].loc[indices].copy()
    features = V2_FEATURE_SETS[pathway]["full"]
    rows, fold_rows = [], []
    for repeat in range(V2_SETTINGS["outer_repeats"]):
        seed = RANDOM_STATE + repeat * 1000
        outer = BalancedComponentKFold(V2_SETTINGS["outer_splits"], shuffle=True, random_state=seed)
        for fold, (train_pos, valid_pos) in enumerate(outer.split(frame, frame[TARGET], frame["dependency_component"])):
            cache_key = strict_cache_key(
                "pairwise_outer_fold", pathway=pathway, repeat=repeat, outer_fold=fold,
                profile=V2_RUN_PROFILE, grid=PAIRWISE_GRID,
                row_index_hash=stable_hash(frame.index.astype(str).tolist()),
            )
            cache_file = V2_CACHE_DIR / f"pairwise_{cache_key}.csv"
            meta_file = V2_CACHE_DIR / f"pairwise_{cache_key}.json"
            cached = validated_cached_fold(cache_file, meta_file)
            if cached is not None:
                cached_predictions, cached_metadata = cached
                rows.extend(cached_predictions.to_dict("records"))
                fold_rows.append(cached_metadata)
                continue
            train, valid = frame.iloc[train_pos], frame.iloc[valid_pos]
            # Small, fully group-disjoint inner comparison of ranking objectives.
            inner = BalancedComponentKFold(V2_SETTINGS["inner_splits"], shuffle=True, random_state=seed + fold)
            candidate_scores = []
            for parameter_index, params in enumerate(PAIRWISE_GRID):
                scores = []
                for inner_fold, (it, iv) in enumerate(inner.split(train, train[TARGET], train["dependency_component"])):
                    bundle = fit_xgb_pairwise_ranker(train.iloc[it], features, params, seed + inner_fold)
                    pred = predict_xgb_pairwise_ranker(bundle, train.iloc[iv])
                    scores.append(capacity_selection_score(train.iloc[iv][TARGET], pred, train.iloc[iv]["run"]))
                candidate_scores.append((np.mean(scores), params))
            best_score, best_params = sorted(candidate_scores, key=lambda x: x[0], reverse=True)[0]
            bundle = fit_xgb_pairwise_ranker(train, features, best_params, seed + fold)
            pred = predict_xgb_pairwise_ranker(bundle, valid)
            current = valid[[TARGET, "matched_set_id", "matched_set_design_valid", "dependency_component"]].copy()
            current["pairwise_score"] = pred
            rank_summary, _ = matched_set_ranking_metrics(current, "pairwise_score")
            fold_record = {
                "pathway": pathway, "repeat": repeat, "outer_fold": fold,
                "best_inner_score": best_score, "best_params": json.dumps(best_params),
                "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "code_sha256": EXPECTED_CODE_SHA256,
                **ranking_and_capacity_metrics(valid[TARGET], pred),
                **rank_summary.iloc[0].to_dict(),
            }
            current_predictions = pd.DataFrame({
                "source_index": valid.index, "pathway": pathway, "family": "Pairwise ranker",
                "repeat": repeat, "outer_fold": fold, "y_true": valid[TARGET].to_numpy(),
                "probability": pred, "dependency_component": valid["dependency_component"].to_numpy(),
            })
            atomic_write_csv(cache_file, current_predictions)
            atomic_write_json(meta_file, fold_record)
            fold_rows.append(fold_record)
            rows.extend(current_predictions.to_dict("records"))
    return pd.DataFrame(rows), pd.DataFrame(fold_rows)


V2_PAIRWISE_PREDICTIONS = {}
V2_PAIRWISE_FOLDS = {}
if V2_RUN_PAIRWISE_RANKER:
    for pathway in PATHWAY_DATA:
        preds, folds = evaluate_pairwise_ranker(pathway)
        V2_PAIRWISE_PREDICTIONS[pathway] = preds
        V2_PAIRWISE_FOLDS[pathway] = folds
        preds.to_csv(V2_TABLE_DIR / f"pairwise_ranker_oof_{pathway}.csv", index=False)
        folds.to_csv(V2_TABLE_DIR / f"pairwise_ranker_folds_{pathway}.csv", index=False)
        display(folds.describe(include="all"))


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class SharedTrunkPathogenNet(nn.Module):
    def __init__(self, input_dim, n_pathogens, hidden=(128, 64), dropout=0.20):
        super().__init__()
        layers = []
        previous = input_dim
        for width in hidden:
            layers.extend([nn.Linear(previous, width), nn.ReLU(), nn.BatchNorm1d(width), nn.Dropout(dropout)])
            previous = width
        self.trunk = nn.Sequential(*layers)
        self.heads = nn.Linear(previous, n_pathogens)

    def forward(self, x, pathogen_index):
        shared = self.trunk(x)
        all_logits = self.heads(shared)
        return all_logits.gather(1, pathogen_index.reshape(-1, 1)).squeeze(1)


def multitask_features(pathway):
    return [f for f in V2_FEATURE_SETS[pathway]["compact"] if f != "run"]


def fit_multitask_model(train_frame, pathway, seed, hidden=(128, 64), dropout=0.20, weight_decay=1e-4):
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    features = multitask_features(pathway)
    preprocessor = make_v2_preprocessor(features)

    splitter = BalancedComponentKFold(
        n_splits=7, random_state=seed, n_trials=64
    )

    fit_pos, stop_pos = next(splitter.split(
        train_frame, train_frame[TARGET], train_frame["dependency_component"]
    ))
    fit_frame, stop_frame = train_frame.iloc[fit_pos], train_frame.iloc[stop_pos]
    X_fit = preprocessor.fit_transform(fit_frame[features]).astype("float32")
    X_stop = preprocessor.transform(stop_frame[features]).astype("float32")
    pathogen_map = {name: idx for idx, name in enumerate(OFFICIAL_RUNS)}
    p_fit = fit_frame["run"].map(pathogen_map).to_numpy(dtype="int64")
    p_stop = stop_frame["run"].map(pathogen_map).to_numpy(dtype="int64")
    y_fit = fit_frame[TARGET].to_numpy(dtype="float32")
    y_stop = stop_frame[TARGET].to_numpy(dtype="float32")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SharedTrunkPathogenNet(X_fit.shape[1], len(OFFICIAL_RUNS), hidden=hidden, dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    dataset = TensorDataset(torch.from_numpy(X_fit), torch.from_numpy(p_fit), torch.from_numpy(y_fit))
    loader_generator = torch.Generator().manual_seed(seed)

    loader = DataLoader(
        dataset, batch_size=512, shuffle=True, drop_last=False,
        generator=loader_generator,
    )

    best_state, best_loss, stale = None, np.inf, 0
    for epoch in range(V2_SETTINGS["multitask_epochs"]):
        model.train()
        for xb, pb, yb in loader:
            xb, pb, yb = xb.to(device), pb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb, pb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            stop_logits = model(torch.from_numpy(X_stop).to(device), torch.from_numpy(p_stop).to(device))
            stop_loss = float(criterion(stop_logits, torch.from_numpy(y_stop).to(device)).cpu())
        if stop_loss < best_loss - 1e-4:
            best_loss = stop_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= 8:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.cpu().eval()
    return {
        "preprocessor": preprocessor, "model": model, "features": features,
        "pathogen_map": pathogen_map, "input_dim": int(X_fit.shape[1]),
        "hidden": hidden, "dropout": dropout,
        "best_internal_loss": best_loss,
    }


def predict_multitask_model(bundle, frame):
    X = bundle["preprocessor"].transform(frame[bundle["features"]]).astype("float32")
    p = frame["run"].map(bundle["pathogen_map"]).to_numpy(dtype="int64")
    with torch.no_grad():
        logits = bundle["model"](torch.from_numpy(X), torch.from_numpy(p))
        return torch.sigmoid(logits).numpy()


MULTITASK_GRID = [
    {"hidden": (128, 64), "dropout": 0.20, "weight_decay": 1e-4},
    {"hidden": (192, 96), "dropout": 0.30, "weight_decay": 5e-4},
]


def evaluate_multitask(pathway):
    indices = V2_PARTITIONS[pathway]["development"].index
    frame = V2_ENGINEERED[pathway].loc[indices].copy()
    rows, fold_rows = [], []
    for repeat in range(V2_SETTINGS["outer_repeats"]):
        seed = RANDOM_STATE + repeat * 1000
        outer = BalancedComponentKFold(V2_SETTINGS["outer_splits"], shuffle=True, random_state=seed)
        for fold, (train_pos, valid_pos) in enumerate(outer.split(frame, frame[TARGET], frame["dependency_component"])):
            cache_key = strict_cache_key(
                "multitask_outer_fold", pathway=pathway, repeat=repeat, outer_fold=fold,
                profile=V2_RUN_PROFILE, grid=MULTITASK_GRID,
                row_index_hash=stable_hash(frame.index.astype(str).tolist()),
            )
            cache_file = V2_CACHE_DIR / f"multitask_{cache_key}.csv"
            meta_file = V2_CACHE_DIR / f"multitask_{cache_key}.json"
            cached = validated_cached_fold(cache_file, meta_file)
            if cached is not None:
                cached_predictions, cached_metadata = cached
                rows.extend(cached_predictions.to_dict("records"))
                fold_rows.append(cached_metadata)
                continue
            train, valid = frame.iloc[train_pos], frame.iloc[valid_pos]
            # One disjoint tuning split inside the outer training set to control compute.
            tune_splitter = BalancedComponentKFold(
                n_splits=5, random_state=seed + fold, n_trials=64
            )

            tune_train_pos, tune_valid_pos = next(tune_splitter.split(
                train, train[TARGET], train["dependency_component"]
            ))
            tune_train, tune_valid = train.iloc[tune_train_pos], train.iloc[tune_valid_pos]
            candidate_scores = []
            for config_index, config in enumerate(MULTITASK_GRID):
                bundle = fit_multitask_model(tune_train, pathway, seed, **config)
                pred = predict_multitask_model(bundle, tune_valid)
                candidate_scores.append((capacity_selection_score(tune_valid[TARGET], pred, tune_valid["run"]), config))
            best_score, best_config = sorted(candidate_scores, key=lambda x: x[0], reverse=True)[0]
            bundle = fit_multitask_model(train, pathway, seed + fold, **best_config)
            pred = predict_multitask_model(bundle, valid)
            fold_record = {
                "pathway": pathway, "repeat": repeat, "outer_fold": fold,
                "best_inner_score": best_score, "best_config": json.dumps(best_config),
                "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "code_sha256": EXPECTED_CODE_SHA256,
                **probability_and_capacity_metrics(valid[TARGET], pred),
            }
            current_predictions = pd.DataFrame({
                "source_index": valid.index, "pathway": pathway, "family": "Multi-task neural network",
                "repeat": repeat, "outer_fold": fold, "y_true": valid[TARGET].to_numpy(),
                "probability": pred, "dependency_component": valid["dependency_component"].to_numpy(),
            })
            atomic_write_csv(cache_file, current_predictions)
            atomic_write_json(meta_file, fold_record)
            fold_rows.append(fold_record)
            rows.extend(current_predictions.to_dict("records"))
    return pd.DataFrame(rows), pd.DataFrame(fold_rows)


V2_MULTITASK_PREDICTIONS = {}
V2_MULTITASK_FOLDS = {}
if V2_RUN_MULTITASK:
    for pathway in PATHWAY_DATA:
        predictions, folds = evaluate_multitask(pathway)
        V2_MULTITASK_PREDICTIONS[pathway] = predictions
        V2_MULTITASK_FOLDS[pathway] = folds
        predictions.to_csv(V2_TABLE_DIR / f"multitask_oof_{pathway}.csv", index=False)
        folds.to_csv(V2_TABLE_DIR / f"multitask_folds_{pathway}.csv", index=False)
        display(folds.describe(include="all"))


In [ ]:
# Strict fold-contained stacking
#
# Each outer validation component remains untouched while every base-model choice,
# level-one prediction, meta-model fit, and internal calibrator fit occurs inside the
# corresponding outer training fold. Pairwise and neural models remain standalone
# comparators because their score scales/estimands differ; the strict probability stack
# uses all five conventional probabilistic families.


def make_meta_model(seed):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=0.25, max_iter=3000, random_state=seed)),
    ])


def choose_family_spec_and_oof(train_frame, pathway, family, seed, feature_sets):
    specs = V2_MODEL_GRIDS[family]
    splitter = BalancedComponentKFold(
        V2_SETTINGS["inner_splits"], shuffle=True, random_state=seed, n_trials=128
    )
    splits = list(splitter.split(
        train_frame, train_frame[TARGET], train_frame["dependency_component"]
    ))
    candidates = []
    for spec_index, spec in enumerate(specs):
        features = feature_sets[spec.feature_set]
        oof = np.full(len(train_frame), np.nan, dtype=float)
        fold_scores = []
        for inner_fold, (fit_pos, valid_pos) in enumerate(splits):
            fit_frame = train_frame.iloc[fit_pos]
            valid_frame = train_frame.iloc[valid_pos]
            estimator = make_v2_classifier(spec, features, seed + spec_index * 100 + inner_fold)
            estimator.fit(fit_frame[features], fit_frame[TARGET])
            prediction = estimator.predict_proba(valid_frame[features])[:, 1]
            oof[valid_pos] = prediction
            fold_scores.append(capacity_selection_score(
                valid_frame[TARGET], prediction, valid_frame["run"], primary_fraction=0.05
            ))
        if not np.isfinite(oof).all():
            raise RuntimeError(f"Incomplete inner OOF predictions for {pathway}/{family}.")
        candidates.append({
            "mean_inner_score": float(np.mean(fold_scores)),
            "spec": spec,
            "oof": oof,
        })
    return max(candidates, key=lambda item: item["mean_inner_score"])


def crossfit_meta_probability(level_one, y, groups, seed):
    prediction = np.full(len(level_one), np.nan, dtype=float)
    splitter = BalancedComponentKFold(
        V2_SETTINGS["inner_splits"], shuffle=True, random_state=seed, n_trials=128
    )
    for fold, (fit_pos, valid_pos) in enumerate(splitter.split(level_one, y, groups)):
        meta_model = make_meta_model(seed + fold)
        meta_model.fit(level_one.iloc[fit_pos], np.asarray(y)[fit_pos])
        prediction[valid_pos] = meta_model.predict_proba(level_one.iloc[valid_pos])[:, 1]
    if not np.isfinite(prediction).all():
        raise RuntimeError("Meta-level cross-fitting did not produce one prediction per row.")
    return prediction


def fit_strict_stack_training_bundle(train_frame, pathway, seed, feature_sets):
    level_one = pd.DataFrame(index=train_frame.index)
    selected = {}
    for family_index, family in enumerate(V2_MODEL_GRIDS):
        choice = choose_family_spec_and_oof(
            train_frame, pathway, family, seed + family_index * 10_000, feature_sets
        )
        level_one[family] = choice["oof"]
        selected[family] = choice

    crossfit_meta = crossfit_meta_probability(
        level_one.reset_index(drop=True),
        train_frame[TARGET].to_numpy(),
        train_frame["dependency_component"].to_numpy(),
        seed + 800_000,
    )
    calibrator = PlattLogitCalibrator().fit(crossfit_meta, train_frame[TARGET])
    meta_model = make_meta_model(seed + 900_000)
    meta_model.fit(level_one, train_frame[TARGET])

    base_models = {}
    for family_index, family in enumerate(V2_MODEL_GRIDS):
        spec = selected[family]["spec"]
        features = feature_sets[spec.feature_set]
        estimator = make_v2_classifier(spec, features, seed + 950_000 + family_index)
        estimator.fit(train_frame[features], train_frame[TARGET])
        base_models[family] = {
            "estimator": estimator,
            "features": features,
            "spec": spec,
        }

    return {
        "pathway": pathway,
        "base_columns": list(V2_MODEL_GRIDS),
        "base_models": base_models,
        "meta_model": meta_model,
        "internal_calibrator": calibrator,
        "selected_inner_scores": {
            family: selected[family]["mean_inner_score"] for family in V2_MODEL_GRIDS
        },
        "selected_specs": {
            family: asdict(selected[family]["spec"]) for family in V2_MODEL_GRIDS
        },
        "level_one_training": level_one,
        "crossfit_meta_training_probability": crossfit_meta,
    }


def predict_strict_stack_bundle(bundle, frame, calibrated=False):
    level_one = pd.DataFrame(index=frame.index)
    for family in bundle["base_columns"]:
        model_bundle = bundle["base_models"][family]
        level_one[family] = model_bundle["estimator"].predict_proba(
            frame[model_bundle["features"]]
        )[:, 1]
    raw_probability = bundle["meta_model"].predict_proba(
        level_one[bundle["base_columns"]]
    )[:, 1]
    if calibrated:
        return bundle["internal_calibrator"].predict(raw_probability)
    return raw_probability


def strict_outer_stack_evaluation(
    pathway,
    frame,
    scenario="primary",
    outer_repeats=None,
    feature_sets=None,
    build_final_bundle=True,
):
    outer_repeats = V2_SETTINGS["outer_repeats"] if outer_repeats is None else int(outer_repeats)
    feature_sets = V2_FEATURE_SETS[pathway] if feature_sets is None else feature_sets
    prediction_rows = []
    fold_rows = []

    for repeat in range(outer_repeats):
        outer_seed = RANDOM_STATE + repeat * 1000
        outer = BalancedComponentKFold(
            V2_SETTINGS["outer_splits"], shuffle=True, random_state=outer_seed, n_trials=128
        )
        for outer_fold, (train_pos, valid_pos) in enumerate(
            outer.split(frame, frame[TARGET], frame["dependency_component"])
        ):
            cache_key = strict_cache_key(
                "strict_outer_stack",
                pathway=pathway,
                scenario=scenario,
                repeat=repeat,
                outer_fold=outer_fold,
                profile=V2_RUN_PROFILE,
                row_index_hash=stable_hash(frame.index.astype(str).tolist()),
                row_component_hash=stable_hash(frame["dependency_component"].astype(str).tolist()),
                feature_sets=feature_sets,
                model_grids={name: [asdict(spec) for spec in specs] for name, specs in V2_MODEL_GRIDS.items()},
            )
            cache_file = V2_CACHE_DIR / f"strict_stack_{cache_key}.csv"
            meta_file = V2_CACHE_DIR / f"strict_stack_{cache_key}.json"
            cached = validated_cached_fold(cache_file, meta_file)
            if cached is not None:
                cached_predictions, cached_metadata = cached
                prediction_rows.extend(cached_predictions.to_dict("records"))
                fold_rows.append(cached_metadata)
                continue

            outer_train = frame.iloc[train_pos].copy()
            outer_valid = frame.iloc[valid_pos].copy()
            overlap = set(outer_train["dependency_component"]) & set(outer_valid["dependency_component"])
            if overlap:
                raise AssertionError("A strict stack outer fold contains dependency leakage.")

            bundle = fit_strict_stack_training_bundle(
                outer_train,
                pathway,
                seed=outer_seed + outer_fold * 100_000,
                feature_sets=feature_sets,
            )
            base_valid = pd.DataFrame(index=outer_valid.index)
            for family in bundle["base_columns"]:
                model_bundle = bundle["base_models"][family]
                base_valid[family] = model_bundle["estimator"].predict_proba(
                    outer_valid[model_bundle["features"]]
                )[:, 1]

            raw_stack = bundle["meta_model"].predict_proba(
                base_valid[bundle["base_columns"]]
            )[:, 1]
            calibrated_stack = bundle["internal_calibrator"].predict(raw_stack)
            best_single_family = max(
                bundle["selected_inner_scores"], key=bundle["selected_inner_scores"].get
            )
            best_single = base_valid[best_single_family].to_numpy()

            current_predictions = pd.DataFrame({
                "source_index": outer_valid.index,
                "pathway": pathway,
                "scenario": scenario,
                "repeat": repeat,
                "outer_fold": outer_fold,
                "y_true": outer_valid[TARGET].to_numpy(),
                "dependency_component": outer_valid["dependency_component"].to_numpy(),
                "run": outer_valid["run"].to_numpy(),
                "probability": raw_stack,
                "calibrated_probability": calibrated_stack,
                "best_single_probability": best_single,
                "best_single_family": best_single_family,
            })
            for family in bundle["base_columns"]:
                current_predictions[f"base__{family}"] = base_valid[family].to_numpy()

            fold_record = {
                "pathway": pathway,
                "scenario": scenario,
                "repeat": repeat,
                "outer_fold": outer_fold,
                "outer_train_n": len(outer_train),
                "outer_valid_n": len(outer_valid),
                "outer_train_components": outer_train["dependency_component"].nunique(),
                "outer_valid_components": outer_valid["dependency_component"].nunique(),
                "component_overlap": 0,
                "best_single_family": best_single_family,
                "selected_inner_scores": bundle["selected_inner_scores"],
                "selected_specs": bundle["selected_specs"],
                "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "code_sha256": EXPECTED_CODE_SHA256,
                **{f"raw_{key}": value for key, value in ranking_and_capacity_metrics(
                    outer_valid[TARGET], raw_stack
                ).items()},
                **{f"calibrated_{key}": value for key, value in probability_metrics(
                    outer_valid[TARGET], calibrated_stack
                ).items()},
            }
            atomic_write_csv(cache_file, current_predictions)
            atomic_write_json(meta_file, fold_record)
            prediction_rows.extend(current_predictions.to_dict("records"))
            fold_rows.append(fold_record)

    final_bundle = None
    if build_final_bundle:
        bundle_key = strict_cache_key(
            "strict_final_stack_bundle",
            pathway=pathway,
            scenario=scenario,
            profile=V2_RUN_PROFILE,
            row_index_hash=stable_hash(frame.index.astype(str).tolist()),
            feature_sets=feature_sets,
        )
        bundle_file = V2_CACHE_DIR / f"strict_final_stack_{bundle_key}.joblib"
        bundle_meta = V2_CACHE_DIR / f"strict_final_stack_{bundle_key}.json"
        if bundle_file.exists() and bundle_meta.exists():
            metadata = json.loads(bundle_meta.read_text(encoding="utf-8"))
            if (
                metadata.get("analysis_fingerprint") == ANALYSIS_FINGERPRINT
                and metadata.get("code_sha256") == EXPECTED_CODE_SHA256
                and metadata.get("cache_schema_version") == CACHE_SCHEMA_VERSION
            ):
                final_bundle = joblib.load(bundle_file)
        if final_bundle is None:
            final_bundle = fit_strict_stack_training_bundle(
                frame, pathway, seed=RANDOM_STATE + 7_000_000, feature_sets=feature_sets
            )
            atomic_joblib_dump(bundle_file, final_bundle)
            atomic_write_json(bundle_meta, {
                "analysis_fingerprint": ANALYSIS_FINGERPRINT,
                "cache_schema_version": CACHE_SCHEMA_VERSION,
                "code_sha256": EXPECTED_CODE_SHA256,
                "pathway": pathway,
                "scenario": scenario,
            })

    return pd.DataFrame(prediction_rows), pd.DataFrame(fold_rows), final_bundle


V2_STACK_PREDICTIONS = {}
V2_STACK_FOLDS = {}
STRICT_BEST_SINGLE_PREDICTIONS = {}
STRICT_STACK_FINAL_BUNDLES = {}
V2_STACK_META_MODELS = {}
V2_STACK_BASE_COLUMNS = {}

if V2_RUN_STACK:
    for pathway in PATHWAY_DATA:
        development = V2_ENGINEERED[pathway].loc[
            V2_PARTITIONS[pathway]["development"].index
        ].copy()
        predictions, folds, final_bundle = strict_outer_stack_evaluation(
            pathway, development, scenario="primary", build_final_bundle=True
        )
        V2_STACK_PREDICTIONS[pathway] = predictions
        V2_STACK_FOLDS[pathway] = folds
        STRICT_STACK_FINAL_BUNDLES[pathway] = final_bundle
        V2_STACK_META_MODELS[pathway] = final_bundle["meta_model"]
        V2_STACK_BASE_COLUMNS[pathway] = final_bundle["base_columns"]
        STRICT_BEST_SINGLE_PREDICTIONS[pathway] = predictions[
            ["source_index", "pathway", "repeat", "outer_fold", "y_true",
             "dependency_component", "run", "best_single_probability", "best_single_family"]
        ].rename(columns={"best_single_probability": "probability"})

        atomic_write_csv(V2_TABLE_DIR / f"strict_stack_oof_{pathway}.csv", predictions)
        atomic_write_csv(V2_TABLE_DIR / f"strict_stack_outer_folds_{pathway}.csv", folds)
        atomic_write_csv(
            V2_TABLE_DIR / f"strict_fold_selected_best_single_oof_{pathway}.csv",
            STRICT_BEST_SINGLE_PREDICTIONS[pathway],
        )
        display(folds[[
            "pathway", "repeat", "outer_fold", "outer_train_n", "outer_valid_n",
            "component_overlap", "best_single_family", "raw_AUROC", "raw_AUPRC",
            "calibrated_Brier", "calibrated_ECE_10", "calibrated_calibration_intercept",
            "calibrated_calibration_slope",
        ]].round(4))


## 15. Internal candidate comparison and sensitivity analysis


In [ ]:
def candidate_prediction_registry():
    registry = {}
    for (pathway, family), predictions in V2_NESTED_PREDICTIONS.items():
        registry[(pathway, family)] = predictions.copy()
    for pathway, predictions in V2_PAIRWISE_PREDICTIONS.items():
        registry[(pathway, "Pairwise ranker")] = predictions.copy()
    for pathway, predictions in V2_MULTITASK_PREDICTIONS.items():
        registry[(pathway, "Multi-task neural network")] = predictions.copy()
    for pathway, predictions in V2_STACK_PREDICTIONS.items():
        registry[(pathway, "Strict nested stack")] = predictions.copy()
    return registry


def within_pathogen_capacity(y_true, score, pathogen, fraction=0.05):
    frame = pd.DataFrame({"y": np.asarray(y_true, int), "score": score, "run": pathogen})
    selected = np.zeros(len(frame), dtype=bool)
    nominal_total = 0
    for _, positions in frame.groupby("run", sort=False).indices.items():
        positions = np.asarray(positions)
        local_scores = frame.iloc[positions]["score"].to_numpy(float)
        count = max(1, int(np.ceil(len(positions) * fraction)))
        nominal_total += count
        boundary = np.sort(local_scores)[-count]
        selected[positions] = local_scores >= boundary
    events = int(frame["y"].sum())
    true_positives = int(frame.loc[selected, "y"].sum())
    precision = true_positives / selected.sum() if selected.any() else np.nan
    prevalence = frame["y"].mean()
    return {
        "capacity_fraction": float(fraction),
        "nominal_n": nominal_total,
        "actual_n_including_ties": int(selected.sum()),
        "true_positives": true_positives,
        "precision_at_capacity": precision,
        "recall_at_capacity": true_positives / events if events else np.nan,
        "enrichment_at_capacity": precision / prevalence if prevalence > 0 else np.nan,
        "number_needed_to_review": 1 / precision if precision > 0 else np.inf,
        "work_saved_vs_review_all": 1 - selected.mean(),
        "selected_fraction": float(selected.mean()),
    }


def pooled_repeat_metrics(pathway, candidate, predictions):
    rows = []
    development = V2_ENGINEERED[pathway].loc[V2_PARTITIONS[pathway]["development"].index]
    run_lookup = development["run"]
    for repeat, repeat_predictions in predictions.groupby("repeat", sort=True):
        if repeat_predictions["source_index"].duplicated().any():
            repeat_predictions = repeat_predictions.groupby("source_index", as_index=False).agg(
                y_true=("y_true", "first"),
                probability=("probability", "mean"),
                dependency_component=("dependency_component", "first"),
            )
        repeat_predictions = repeat_predictions.copy()
        repeat_predictions["run"] = repeat_predictions["source_index"].map(run_lookup)
        if repeat_predictions["run"].isna().any():
            raise RuntimeError("A development prediction could not be linked to its pathogen.")

        y_true = repeat_predictions["y_true"].to_numpy(int)
        score = repeat_predictions["probability"].to_numpy(float)
        overall_auroc = roc_auc_score(y_true, score)
        overall_auprc = average_precision_score(y_true, score)
        per_pathogen = []
        for pathogen, subset in repeat_predictions.groupby("run"):
            if subset["y_true"].nunique() < 2:
                continue
            local_prevalence = subset["y_true"].mean()
            per_pathogen.append({
                "run": pathogen,
                "AUROC": roc_auc_score(subset["y_true"], subset["probability"]),
                "AUPRC_lift": average_precision_score(
                    subset["y_true"], subset["probability"]
                ) / local_prevalence,
            })
        per_pathogen = pd.DataFrame(per_pathogen)
        capacity = within_pathogen_capacity(
            y_true, score, repeat_predictions["run"].to_numpy(), fraction=0.05
        )
        rows.append({
            "pathway": pathway,
            "candidate": candidate,
            "repeat": int(repeat),
            "n": len(repeat_predictions),
            "AUROC": overall_auroc,
            "AUPRC": overall_auprc,
            "AUPRC_lift": overall_auprc / y_true.mean(),
            "macro_pathogen_AUROC": per_pathogen["AUROC"].mean(),
            "macro_pathogen_AUPRC_lift": per_pathogen["AUPRC_lift"].mean(),
            **capacity,
        })
    return pd.DataFrame(rows)


V2_CANDIDATE_PREDICTIONS = candidate_prediction_registry()
v2_candidate_repeat_metrics_df = pd.concat([
    pooled_repeat_metrics(pathway, candidate, predictions)
    for (pathway, candidate), predictions in V2_CANDIDATE_PREDICTIONS.items()
], ignore_index=True)

v2_candidate_summary_df = (
    v2_candidate_repeat_metrics_df.groupby(["pathway", "candidate"], as_index=False)
    .agg(
        repeats=("repeat", "nunique"),
        mean_AUROC=("AUROC", "mean"),
        sd_AUROC=("AUROC", "std"),
        mean_AUPRC=("AUPRC", "mean"),
        sd_AUPRC=("AUPRC", "std"),
        mean_AUPRC_lift=("AUPRC_lift", "mean"),
        mean_macro_pathogen_AUROC=("macro_pathogen_AUROC", "mean"),
        mean_macro_pathogen_AUPRC_lift=("macro_pathogen_AUPRC_lift", "mean"),
        mean_precision_at_5pct=("precision_at_capacity", "mean"),
        mean_recall_at_5pct=("recall_at_capacity", "mean"),
        mean_enrichment_at_5pct=("enrichment_at_capacity", "mean"),
    )
)
for column in [
    "mean_macro_pathogen_AUPRC_lift", "mean_recall_at_5pct", "mean_enrichment_at_5pct"
]:
    maximum = v2_candidate_summary_df.groupby("pathway")[column].transform("max")
    v2_candidate_summary_df[f"scaled_{column}"] = np.divide(
        v2_candidate_summary_df[column], maximum,
        out=np.zeros(len(v2_candidate_summary_df), dtype=float),
        where=maximum.to_numpy() > 0,
    )
v2_candidate_summary_df["selection_score"] = (
    0.50 * v2_candidate_summary_df["scaled_mean_macro_pathogen_AUPRC_lift"]
    + 0.30 * v2_candidate_summary_df["scaled_mean_recall_at_5pct"]
    + 0.20 * v2_candidate_summary_df["scaled_mean_enrichment_at_5pct"]
)
v2_candidate_summary_df = v2_candidate_summary_df.sort_values(
    ["pathway", "selection_score", "mean_macro_pathogen_AUROC"],
    ascending=[True, False, False],
)
V2_INTERNAL_WINNERS = (
    v2_candidate_summary_df.groupby("pathway", as_index=False).first()[
        ["pathway", "candidate", "selection_score"]
    ]
)
V2_FINAL_CANDIDATES = dict(zip(V2_INTERNAL_WINNERS["pathway"], V2_INTERNAL_WINNERS["candidate"]))
V2_BEST_SINGLE_CANDIDATES = {}
for pathway in PATHWAY_DATA:
    eligible_single = v2_candidate_summary_df.loc[
        v2_candidate_summary_df["pathway"].eq(pathway)
        & ~v2_candidate_summary_df["candidate"].eq("Strict nested stack")
    ].sort_values(["selection_score", "mean_macro_pathogen_AUROC"], ascending=[False, False])
    if eligible_single.empty:
        raise RuntimeError(f"No single-model comparator is available for {pathway}.")
    V2_BEST_SINGLE_CANDIDATES[pathway] = eligible_single.iloc[0]["candidate"]

atomic_write_csv(V2_TABLE_DIR / "candidate_metrics_by_repeat.csv", v2_candidate_repeat_metrics_df)
atomic_write_csv(V2_TABLE_DIR / "candidate_summary.csv", v2_candidate_summary_df)
atomic_write_csv(V2_TABLE_DIR / "internal_model_selection.csv", V2_INTERNAL_WINNERS)
atomic_write_json(V2_PROTOCOL_DIR / "final_candidate_registry.json", V2_FINAL_CANDIDATES)
atomic_write_json(V2_PROTOCOL_DIR / "best_single_candidate_registry.json", V2_BEST_SINGLE_CANDIDATES)

figure, axis = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=v2_candidate_summary_df,
    x="mean_macro_pathogen_AUPRC_lift",
    y="candidate",
    hue="pathway",
    ax=axis,
)
axis.set_title("Internal candidate comparison")
axis.set_xlabel("Mean pathogen-macro AUPRC lift")
axis.set_ylabel("")
axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
figure.savefig(V2_FIGURE_DIR / "candidate_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
display(v2_candidate_summary_df.round(4))
display(V2_INTERNAL_WINNERS)


### Dependency-component uncertainty and paired model comparison

Confidence intervals resample whole dependency components with replacement. Each source observation is represented once by averaging its repeated outer-fold predictions. The 10% capacity metrics are named generically in computation and explicitly labelled as 10% only in final tables. The paired comparison evaluates the strict nested stack against the strongest single candidate selected by the prespecified internal rule, using identical bootstrap samples. Its interval is therefore explicitly selection-conditioned.


In [ ]:
BOOTSTRAP_CAPACITY_FRACTION = 0.10
BOOTSTRAP_METRICS = ["AUROC", "AUPRC", "precision", "recall", "enrichment"]


def average_repeated_oof(predictions, score_column="probability"):
    required = ["source_index", "y_true", "dependency_component", "run", score_column]
    missing = sorted(set(required) - set(predictions.columns))
    if missing:
        raise KeyError(f"OOF predictions are missing: {missing}")
    return predictions.groupby("source_index", as_index=False).agg(
        y_true=("y_true", "first"),
        dependency_component=("dependency_component", "first"),
        run=("run", "first"),
        score=(score_column, "mean"),
    )


def bootstrap_metric_vector(frame, score_column="score", capacity_fraction=0.10):
    if frame["y_true"].nunique() < 2:
        return None
    y = frame["y_true"].to_numpy(int)
    score = frame[score_column].to_numpy(float)
    capacity = within_pathogen_capacity(y, score, frame["run"], capacity_fraction)
    return {
        "AUROC": roc_auc_score(y, score),
        "AUPRC": average_precision_score(y, score),
        "precision": capacity["precision_at_capacity"],
        "recall": capacity["recall_at_capacity"],
        "enrichment": capacity["enrichment_at_capacity"],
    }


def component_bootstrap_draw(frame, replicate, seed):
    components = frame["dependency_component"].drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed + replicate)
    drawn = rng.choice(components, size=len(components), replace=True)
    pieces = []
    for draw_index, component in enumerate(drawn):
        piece = frame.loc[frame["dependency_component"].eq(component)].copy()
        piece["bootstrap_component_copy"] = draw_index
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True)


def cached_component_bootstrap(frame, pathway, model_label, score_column, replicates, seed):
    key = strict_cache_key(
        "component_bootstrap",
        pathway=pathway,
        model=model_label,
        replicates=replicates,
        seed=seed,
        capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        row_hash=stable_hash(frame["source_index"].astype(str).tolist()),
        score_hash=stable_hash(np.round(frame[score_column].to_numpy(float), 12).tolist()),
    )
    cache_file = V2_CACHE_DIR / f"bootstrap_{key}.csv"
    rows = pd.read_csv(cache_file).to_dict("records") if cache_file.exists() else []
    completed = {int(row["replicate"]) for row in rows}
    for replicate in range(replicates):
        if replicate in completed:
            continue
        sample = component_bootstrap_draw(frame, replicate, seed)
        metrics = bootstrap_metric_vector(sample, score_column, BOOTSTRAP_CAPACITY_FRACTION)
        if metrics is not None:
            rows.append({"replicate": replicate, **metrics})
        if (replicate + 1) % 50 == 0:
            atomic_write_csv(cache_file, pd.DataFrame(rows).sort_values("replicate"))
    result = pd.DataFrame(rows).sort_values("replicate").reset_index(drop=True)
    atomic_write_csv(cache_file, result)
    return result


def summarise_bootstrap(point, replicates, pathway, model_label):
    rows = []
    for metric in BOOTSTRAP_METRICS:
        values = replicates[metric].dropna().to_numpy(float)
        rows.append({
            "pathway": pathway,
            "model": model_label,
            "metric": metric,
            "capacity_fraction_for_queue_metrics": (
                BOOTSTRAP_CAPACITY_FRACTION if metric in {"precision", "recall", "enrichment"} else np.nan
            ),
            "estimate": point[metric],
            "ci_95_lower": np.quantile(values, 0.025),
            "ci_95_upper": np.quantile(values, 0.975),
            "bootstrap_replicates": len(values),
            "resampling_unit": "dependency_component",
        })
    return pd.DataFrame(rows)


def cached_paired_bootstrap(frame, pathway, replicates, seed):
    key = strict_cache_key(
        "paired_stack_vs_single_bootstrap",
        pathway=pathway,
        replicates=replicates,
        seed=seed,
        capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        row_hash=stable_hash(frame["source_index"].astype(str).tolist()),
        score_hash=stable_hash(np.round(frame[["stack_score", "single_score"]].to_numpy(float), 12).tolist()),
    )
    cache_file = V2_CACHE_DIR / f"paired_bootstrap_{key}.csv"
    rows = pd.read_csv(cache_file).to_dict("records") if cache_file.exists() else []
    completed = {int(row["replicate"]) for row in rows}
    for replicate in range(replicates):
        if replicate in completed:
            continue
        sample = component_bootstrap_draw(frame, replicate, seed)
        stack = bootstrap_metric_vector(sample, "stack_score", BOOTSTRAP_CAPACITY_FRACTION)
        single = bootstrap_metric_vector(sample, "single_score", BOOTSTRAP_CAPACITY_FRACTION)
        if stack is not None and single is not None:
            rows.append({
                "replicate": replicate,
                **{f"difference_{metric}": stack[metric] - single[metric] for metric in BOOTSTRAP_METRICS},
            })
        if (replicate + 1) % 50 == 0:
            atomic_write_csv(cache_file, pd.DataFrame(rows).sort_values("replicate"))
    result = pd.DataFrame(rows).sort_values("replicate").reset_index(drop=True)
    atomic_write_csv(cache_file, result)
    return result


bootstrap_summary_frames = []
paired_summary_rows = []
crossfitted_calibration_rows = []

for pathway in PATHWAY_DATA:
    stack = average_repeated_oof(V2_STACK_PREDICTIONS[pathway], "probability")
    best_single_candidate = V2_BEST_SINGLE_CANDIDATES[pathway]
    single_predictions = V2_CANDIDATE_PREDICTIONS[(pathway, best_single_candidate)].copy()
    if "run" not in single_predictions.columns:
        run_lookup = V2_ENGINEERED[pathway].loc[
            V2_PARTITIONS[pathway]["development"].index, "run"
        ]
        single_predictions["run"] = single_predictions["source_index"].map(run_lookup)
    single = average_repeated_oof(single_predictions, "probability")
    paired = stack.rename(columns={"score": "stack_score"}).merge(
        single[["source_index", "score"]].rename(columns={"score": "single_score"}),
        on="source_index", how="inner", validate="one_to_one",
    )

    for model_label, frame, score_column, seed_offset in [
        ("Strict nested stack", paired, "stack_score", 0),
        (f"Best single model: {best_single_candidate}", paired, "single_score", 10_000),
    ]:
        point = bootstrap_metric_vector(frame, score_column, BOOTSTRAP_CAPACITY_FRACTION)
        bootstrap_replicates = cached_component_bootstrap(
            frame, pathway, model_label, score_column,
            V2_SETTINGS["bootstrap"], RANDOM_STATE + seed_offset,
        )
        bootstrap_summary_frames.append(
            summarise_bootstrap(point, bootstrap_replicates, pathway, model_label)
        )
        safe_model_label = "".join(
            character if character.isalnum() else "_" for character in model_label
        ).strip("_")
        atomic_write_csv(
            V2_TABLE_DIR / f"component_bootstrap_replicates_{pathway}_{safe_model_label}.csv",
            bootstrap_replicates,
        )

    paired_replicates = cached_paired_bootstrap(
        paired, pathway, V2_SETTINGS["bootstrap"], RANDOM_STATE + 20_000
    )
    point_stack = bootstrap_metric_vector(paired, "stack_score", BOOTSTRAP_CAPACITY_FRACTION)
    point_single = bootstrap_metric_vector(paired, "single_score", BOOTSTRAP_CAPACITY_FRACTION)
    for metric in BOOTSTRAP_METRICS:
        values = paired_replicates[f"difference_{metric}"].dropna().to_numpy(float)
        paired_summary_rows.append({
            "pathway": pathway,
            "comparison": f"Strict nested stack minus best single model ({best_single_candidate})",
            "best_single_model": best_single_candidate,
            "metric": metric,
            "capacity_fraction_for_queue_metrics": (
                BOOTSTRAP_CAPACITY_FRACTION if metric in {"precision", "recall", "enrichment"} else np.nan
            ),
            "stack_estimate": point_stack[metric],
            "single_estimate": point_single[metric],
            "paired_difference": point_stack[metric] - point_single[metric],
            "paired_ci_95_lower": np.quantile(values, 0.025),
            "paired_ci_95_upper": np.quantile(values, 0.975),
            "bootstrap_replicates": len(values),
            "resampling_unit": "dependency_component",
            "interpretation": "Selection-conditioned internal comparison; independent validation remains required",
        })
    atomic_write_csv(
        V2_TABLE_DIR / f"paired_bootstrap_replicates_stack_vs_single_{pathway}.csv",
        paired_replicates,
    )

    # Honest internal calibration: each outer validation score was calibrated by a
    # calibrator trained on cross-fitted predictions from that outer training fold.
    for repeat, repeat_frame in V2_STACK_PREDICTIONS[pathway].groupby("repeat"):
        metrics = probability_metrics(
            repeat_frame["y_true"], repeat_frame["calibrated_probability"]
        )
        crossfitted_calibration_rows.append({
            "pathway": pathway,
            "repeat": int(repeat),
            "n": len(repeat_frame),
            "components": repeat_frame["dependency_component"].nunique(),
            "assessment": "Outer-fold predictions; calibrator fitted only within outer training components",
            **metrics,
        })

component_bootstrap_ci_df = pd.concat(bootstrap_summary_frames, ignore_index=True)
paired_stack_vs_single_ci_df = pd.DataFrame(paired_summary_rows)
crossfitted_internal_calibration_df = pd.DataFrame(crossfitted_calibration_rows)

atomic_write_csv(V2_TABLE_DIR / "component_bootstrap_95ci.csv", component_bootstrap_ci_df)
atomic_write_csv(V2_TABLE_DIR / "paired_stack_vs_best_single_95ci.csv", paired_stack_vs_single_ci_df)
atomic_write_csv(
    V2_TABLE_DIR / "crossfitted_internal_calibration_by_repeat.csv",
    crossfitted_internal_calibration_df,
)
display(component_bootstrap_ci_df.round(4))
display(paired_stack_vs_single_ci_df.round(4))
display(crossfitted_internal_calibration_df.round(4))


In [ ]:
# Dependency-component bootstrap uncertainty,
# paired stack-versus-single comparisons,
# and cross-fitted internal calibration assessment.

BOOTSTRAP_CAPACITY_FRACTION = 0.10
BOOTSTRAP_METRICS = [
    "AUROC",
    "AUPRC",
    "precision",
    "recall",
    "enrichment",
]
BOOTSTRAP_CHECKPOINT_EVERY = 25


def average_repeated_oof(predictions, score_column="probability"):
    required = [
        "source_index",
        "y_true",
        "dependency_component",
        "run",
        score_column,
    ]
    missing = sorted(set(required) - set(predictions.columns))
    if missing:
        raise KeyError(f"OOF predictions are missing: {missing}")

    result = (
        predictions
        .groupby("source_index", as_index=False)
        .agg(
            y_true=("y_true", "first"),
            dependency_component=("dependency_component", "first"),
            run=("run", "first"),
            score=(score_column, "mean"),
        )
    )

    if result["source_index"].duplicated().any():
        raise RuntimeError(
            "Repeated-OOF averaging did not produce one row per source_index."
        )

    return result


def bootstrap_metric_vector(
    frame,
    score_column="score",
    capacity_fraction=0.10,
):
    if frame["y_true"].nunique() < 2:
        return None

    y = frame["y_true"].to_numpy(dtype=int)
    score = frame[score_column].to_numpy(dtype=float)

    capacity = within_pathogen_capacity(
        y,
        score,
        frame["run"],
        capacity_fraction,
    )

    return {
        "AUROC": roc_auc_score(y, score),
        "AUPRC": average_precision_score(y, score),
        "precision": capacity["precision_at_capacity"],
        "recall": capacity["recall_at_capacity"],
        "enrichment": capacity["enrichment_at_capacity"],
    }


def prepare_component_bootstrap_index(frame):
    """
    Index the integer row positions belonging to each dependency component.

    This is performed once for each bootstrap job. It avoids repeatedly
    scanning the complete DataFrame for every sampled component in every
    bootstrap replicate.
    """
    required = {"dependency_component", "y_true", "run"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise KeyError(
            f"Bootstrap frame is missing required columns: {missing}"
        )

    if frame["dependency_component"].isna().any():
        raise RuntimeError(
            "Dependency-component bootstrap cannot proceed with missing "
            "component identifiers."
        )

    components = (
        frame["dependency_component"]
        .drop_duplicates()
        .to_numpy()
    )

    grouped_positions = frame.groupby(
        "dependency_component",
        sort=False,
        observed=True,
    ).indices

    component_positions = {
        component: np.asarray(
            grouped_positions[component],
            dtype=np.int64,
        )
        for component in components
    }

    indexed_rows = sum(
        len(positions)
        for positions in component_positions.values()
    )

    if indexed_rows != len(frame):
        raise RuntimeError(
            "Component index does not account for every bootstrap row: "
            f"{indexed_rows:,} indexed versus {len(frame):,} expected."
        )

    return components, component_positions


def component_bootstrap_draw(
    frame,
    replicate,
    seed,
    component_index=None,
):
    """
    Sample dependency components with replacement and retain every row
    belonging to each sampled component.

    The random-number construction and component ordering match the
    original implementation. The removed bootstrap_component_copy column
    was not used by any downstream metric.
    """
    if component_index is None:
        component_index = prepare_component_bootstrap_index(frame)

    components, component_positions = component_index

    rng = np.random.default_rng(seed + replicate)
    drawn_components = rng.choice(
        components,
        size=len(components),
        replace=True,
    )

    sampled_positions = np.concatenate([
        component_positions[component]
        for component in drawn_components
    ])

    return frame.iloc[sampled_positions]


def load_bootstrap_checkpoint(cache_file):
    if not cache_file.exists():
        return pd.DataFrame()

    checkpoint = pd.read_csv(cache_file)

    if checkpoint.empty:
        return checkpoint

    if "replicate" not in checkpoint.columns:
        raise RuntimeError(
            f"Invalid bootstrap checkpoint without replicate column: "
            f"{cache_file}"
        )

    checkpoint["replicate"] = pd.to_numeric(
        checkpoint["replicate"],
        errors="raise",
    ).astype(int)

    checkpoint = (
        checkpoint
        .drop_duplicates(subset=["replicate"], keep="last")
        .sort_values("replicate")
        .reset_index(drop=True)
    )

    return checkpoint


def write_bootstrap_checkpoint(cache_file, rows):
    checkpoint = pd.DataFrame(rows)

    if checkpoint.empty:
        return checkpoint

    checkpoint["replicate"] = pd.to_numeric(
        checkpoint["replicate"],
        errors="raise",
    ).astype(int)

    checkpoint = (
        checkpoint
        .drop_duplicates(subset=["replicate"], keep="last")
        .sort_values("replicate")
        .reset_index(drop=True)
    )

    atomic_write_csv(cache_file, checkpoint)
    return checkpoint


def cached_component_bootstrap(
    frame,
    pathway,
    model_label,
    score_column,
    replicates,
    seed,
):
    key = strict_cache_key(
        "component_bootstrap",
        pathway=pathway,
        model=model_label,
        replicates=replicates,
        seed=seed,
        capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        row_hash=stable_hash(
            frame["source_index"].astype(str).tolist()
        ),
        score_hash=stable_hash(
            np.round(
                frame[score_column].to_numpy(dtype=float),
                12,
            ).tolist()
        ),
    )

    cache_file = V2_CACHE_DIR / f"bootstrap_{key}.csv"
    checkpoint = load_bootstrap_checkpoint(cache_file)

    if checkpoint.empty:
        rows = []
        completed = set()
    else:
        rows = checkpoint.to_dict("records")
        completed = set(
            checkpoint["replicate"].astype(int).tolist()
        )

    print(
        f"[{pathway} | {model_label}] "
        f"loaded {len(completed):,}/{replicates:,} "
        "component-bootstrap replicates.",
        flush=True,
    )

    if len(completed) >= replicates:
        print(
            f"[{pathway} | {model_label}] cache complete.",
            flush=True,
        )
        return checkpoint

    component_index = prepare_component_bootstrap_index(frame)

    for replicate in range(replicates):
        if replicate in completed:
            continue

        sample = component_bootstrap_draw(
            frame=frame,
            replicate=replicate,
            seed=seed,
            component_index=component_index,
        )

        metrics = bootstrap_metric_vector(
            sample,
            score_column=score_column,
            capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        )

        if metrics is not None:
            rows.append({
                "replicate": replicate,
                **metrics,
            })
            completed.add(replicate)

        if (
            (replicate + 1) % BOOTSTRAP_CHECKPOINT_EVERY == 0
            or replicate == replicates - 1
        ):
            checkpoint = write_bootstrap_checkpoint(
                cache_file,
                rows,
            )
            rows = checkpoint.to_dict("records")

            print(
                f"[{pathway} | {model_label}] "
                f"{len(completed):,}/{replicates:,} complete.",
                flush=True,
            )

    result = write_bootstrap_checkpoint(cache_file, rows)

    print(
        f"[{pathway} | {model_label}] finished with "
        f"{len(result):,} valid replicates.",
        flush=True,
    )

    return result


def summarise_bootstrap(
    point,
    replicates,
    pathway,
    model_label,
):
    rows = []

    for metric in BOOTSTRAP_METRICS:
        values = (
            replicates[metric]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(values) == 0:
            raise RuntimeError(
                f"No valid bootstrap values were obtained for "
                f"{pathway}, {model_label}, {metric}."
            )

        rows.append({
            "pathway": pathway,
            "model": model_label,
            "metric": metric,
            "capacity_fraction_for_queue_metrics": (
                BOOTSTRAP_CAPACITY_FRACTION
                if metric in {
                    "precision",
                    "recall",
                    "enrichment",
                }
                else np.nan
            ),
            "estimate": point[metric],
            "ci_95_lower": np.quantile(values, 0.025),
            "ci_95_upper": np.quantile(values, 0.975),
            "bootstrap_replicates": len(values),
            "resampling_unit": "dependency_component",
        })

    return pd.DataFrame(rows)


def cached_paired_bootstrap(
    frame,
    pathway,
    replicates,
    seed,
):
    key = strict_cache_key(
        "paired_stack_vs_single_bootstrap",
        pathway=pathway,
        replicates=replicates,
        seed=seed,
        capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        row_hash=stable_hash(
            frame["source_index"].astype(str).tolist()
        ),
        score_hash=stable_hash(
            np.round(
                frame[
                    ["stack_score", "single_score"]
                ].to_numpy(dtype=float),
                12,
            ).tolist()
        ),
    )

    cache_file = V2_CACHE_DIR / f"paired_bootstrap_{key}.csv"
    checkpoint = load_bootstrap_checkpoint(cache_file)

    if checkpoint.empty:
        rows = []
        completed = set()
    else:
        rows = checkpoint.to_dict("records")
        completed = set(
            checkpoint["replicate"].astype(int).tolist()
        )

    print(
        f"[{pathway} | paired stack-versus-single] "
        f"loaded {len(completed):,}/{replicates:,} replicates.",
        flush=True,
    )

    if len(completed) >= replicates:
        print(
            f"[{pathway} | paired stack-versus-single] "
            "cache complete.",
            flush=True,
        )
        return checkpoint

    component_index = prepare_component_bootstrap_index(frame)

    for replicate in range(replicates):
        if replicate in completed:
            continue

        sample = component_bootstrap_draw(
            frame=frame,
            replicate=replicate,
            seed=seed,
            component_index=component_index,
        )

        stack_metrics = bootstrap_metric_vector(
            sample,
            score_column="stack_score",
            capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        )

        single_metrics = bootstrap_metric_vector(
            sample,
            score_column="single_score",
            capacity_fraction=BOOTSTRAP_CAPACITY_FRACTION,
        )

        if (
            stack_metrics is not None
            and single_metrics is not None
        ):
            rows.append({
                "replicate": replicate,
                **{
                    f"difference_{metric}": (
                        stack_metrics[metric]
                        - single_metrics[metric]
                    )
                    for metric in BOOTSTRAP_METRICS
                },
            })
            completed.add(replicate)

        if (
            (replicate + 1) % BOOTSTRAP_CHECKPOINT_EVERY == 0
            or replicate == replicates - 1
        ):
            checkpoint = write_bootstrap_checkpoint(
                cache_file,
                rows,
            )
            rows = checkpoint.to_dict("records")

            print(
                f"[{pathway} | paired stack-versus-single] "
                f"{len(completed):,}/{replicates:,} complete.",
                flush=True,
            )

    result = write_bootstrap_checkpoint(cache_file, rows)

    print(
        f"[{pathway} | paired stack-versus-single] finished "
        f"with {len(result):,} valid replicates.",
        flush=True,
    )

    return result


bootstrap_summary_frames = []
paired_summary_rows = []
crossfitted_calibration_rows = []

for pathway in PATHWAY_DATA:
    print(
        "\n"
        + "=" * 78
        + f"\nBOOTSTRAP ANALYSIS: {pathway.upper()}"
        + "\n"
        + "=" * 78,
        flush=True,
    )

    stack = average_repeated_oof(
        V2_STACK_PREDICTIONS[pathway],
        "probability",
    )

    best_single_candidate = V2_BEST_SINGLE_CANDIDATES[pathway]

    single_predictions = V2_CANDIDATE_PREDICTIONS[
        (pathway, best_single_candidate)
    ].copy()

    if "run" not in single_predictions.columns:
        run_lookup = V2_ENGINEERED[pathway].loc[
            V2_PARTITIONS[pathway]["development"].index,
            "run",
        ]
        single_predictions["run"] = (
            single_predictions["source_index"].map(run_lookup)
        )

    single = average_repeated_oof(
        single_predictions,
        "probability",
    )

    paired = (
        stack
        .rename(columns={"score": "stack_score"})
        .merge(
            single[["source_index", "score"]].rename(
                columns={"score": "single_score"}
            ),
            on="source_index",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(paired) != len(stack) or len(paired) != len(single):
        raise RuntimeError(
            f"{pathway}: stack and single-model prediction rows "
            "did not align completely."
        )

    for (
        model_label,
        frame,
        score_column,
        seed_offset,
    ) in [
        (
            "Strict nested stack",
            paired,
            "stack_score",
            0,
        ),
        (
            f"Best single model: {best_single_candidate}",
            paired,
            "single_score",
            10_000,
        ),
    ]:
        point = bootstrap_metric_vector(
            frame,
            score_column,
            BOOTSTRAP_CAPACITY_FRACTION,
        )

        bootstrap_replicates = cached_component_bootstrap(
            frame=frame,
            pathway=pathway,
            model_label=model_label,
            score_column=score_column,
            replicates=V2_SETTINGS["bootstrap"],
            seed=RANDOM_STATE + seed_offset,
        )

        bootstrap_summary_frames.append(
            summarise_bootstrap(
                point=point,
                replicates=bootstrap_replicates,
                pathway=pathway,
                model_label=model_label,
            )
        )

        safe_model_label = "".join(
            character if character.isalnum() else "_"
            for character in model_label
        ).strip("_")

        atomic_write_csv(
            V2_TABLE_DIR
            / (
                f"component_bootstrap_replicates_"
                f"{pathway}_{safe_model_label}.csv"
            ),
            bootstrap_replicates,
        )

    paired_replicates = cached_paired_bootstrap(
        frame=paired,
        pathway=pathway,
        replicates=V2_SETTINGS["bootstrap"],
        seed=RANDOM_STATE + 20_000,
    )

    point_stack = bootstrap_metric_vector(
        paired,
        "stack_score",
        BOOTSTRAP_CAPACITY_FRACTION,
    )

    point_single = bootstrap_metric_vector(
        paired,
        "single_score",
        BOOTSTRAP_CAPACITY_FRACTION,
    )

    for metric in BOOTSTRAP_METRICS:
        values = (
            paired_replicates[f"difference_{metric}"]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(values) == 0:
            raise RuntimeError(
                f"No valid paired-bootstrap differences were "
                f"obtained for {pathway}, {metric}."
            )

        paired_summary_rows.append({
            "pathway": pathway,
            "comparison": (
                "Strict nested stack minus best single model "
                f"({best_single_candidate})"
            ),
            "best_single_model": best_single_candidate,
            "metric": metric,
            "capacity_fraction_for_queue_metrics": (
                BOOTSTRAP_CAPACITY_FRACTION
                if metric in {
                    "precision",
                    "recall",
                    "enrichment",
                }
                else np.nan
            ),
            "stack_estimate": point_stack[metric],
            "single_estimate": point_single[metric],
            "paired_difference": (
                point_stack[metric]
                - point_single[metric]
            ),
            "paired_ci_95_lower": np.quantile(
                values,
                0.025,
            ),
            "paired_ci_95_upper": np.quantile(
                values,
                0.975,
            ),
            "bootstrap_replicates": len(values),
            "resampling_unit": "dependency_component",
            "interpretation": (
                "Selection-conditioned internal comparison; "
                "independent validation remains required"
            ),
        })

    atomic_write_csv(
        V2_TABLE_DIR
        / (
            f"paired_bootstrap_replicates_"
            f"stack_vs_single_{pathway}.csv"
        ),
        paired_replicates,
    )

    # Each outer-validation prediction is calibrated by a calibrator
    # fitted only using cross-fitted predictions from the corresponding
    # outer-training components.
    for repeat, repeat_frame in (
        V2_STACK_PREDICTIONS[pathway]
        .groupby("repeat")
    ):
        metrics = probability_metrics(
            repeat_frame["y_true"],
            repeat_frame["calibrated_probability"],
        )

        crossfitted_calibration_rows.append({
            "pathway": pathway,
            "repeat": int(repeat),
            "n": len(repeat_frame),
            "components": (
                repeat_frame["dependency_component"].nunique()
            ),
            "assessment": (
                "Outer-fold predictions; calibrator fitted only "
                "within outer-training components"
            ),
            **metrics,
        })


component_bootstrap_ci_df = pd.concat(
    bootstrap_summary_frames,
    ignore_index=True,
)

paired_stack_vs_single_ci_df = pd.DataFrame(
    paired_summary_rows
)

crossfitted_internal_calibration_df = pd.DataFrame(
    crossfitted_calibration_rows
)

atomic_write_csv(
    V2_TABLE_DIR / "component_bootstrap_95ci.csv",
    component_bootstrap_ci_df,
)

atomic_write_csv(
    V2_TABLE_DIR / "paired_stack_vs_best_single_95ci.csv",
    paired_stack_vs_single_ci_df,
)

atomic_write_csv(
    V2_TABLE_DIR
    / "crossfitted_internal_calibration_by_repeat.csv",
    crossfitted_internal_calibration_df,
)

print(
    "\nAll dependency-component bootstrap analyses completed.",
    flush=True,
)

display(component_bootstrap_ci_df.round(4))
display(paired_stack_vs_single_ci_df.round(4))
display(crossfitted_internal_calibration_df.round(4))

# Main-text figure: absolute estimates and paired stack-minus-single differences.
metric_labels = {
    "AUROC": "AUROC",
    "AUPRC": "AUPRC",
    "precision": "Precision at 10%",
    "recall": "Recall at 10%",
    "enrichment": "Enrichment at 10%",
}
absolute_metrics = ["AUROC", "AUPRC", "precision", "recall"]
comparison_metrics = ["AUROC", "AUPRC", "precision", "recall", "enrichment"]
pathway_colours = {"patient": "#D97706", "environmental": "#2563A6"}
model_markers = {"Strict nested stack": "o", "single": "s"}

figure, axes = plt.subplots(2, 2, figsize=(12, 7.6), constrained_layout=True)
for row, pathway in enumerate(["patient", "environmental"]):
    absolute_axis = axes[row, 0]
    absolute_subset = component_bootstrap_ci_df.loc[
        component_bootstrap_ci_df["pathway"].eq(pathway)
        & component_bootstrap_ci_df["metric"].isin(absolute_metrics)
    ].copy()
    model_names = absolute_subset["model"].drop_duplicates().tolist()
    offsets = np.linspace(-0.12, 0.12, max(1, len(model_names)))
    for model_index, model_name in enumerate(model_names):
        model_subset = (
            absolute_subset.loc[absolute_subset["model"].eq(model_name)]
            .set_index("metric").reindex(absolute_metrics)
        )
        estimates = model_subset["estimate"].to_numpy(dtype=float)
        lower = model_subset["ci_95_lower"].to_numpy(dtype=float)
        upper = model_subset["ci_95_upper"].to_numpy(dtype=float)
        positions = np.arange(len(absolute_metrics)) + offsets[model_index]
        short_label = "Strict stack" if model_name == "Strict nested stack" else model_name.replace("Best single model: ", "")
        absolute_axis.errorbar(
            estimates,
            positions,
            xerr=np.vstack([
                np.maximum(0, estimates - lower),
                np.maximum(0, upper - estimates),
            ]),
            fmt=model_markers["Strict nested stack" if model_name == "Strict nested stack" else "single"],
            capsize=3,
            linewidth=1.3,
            markersize=6,
            label=short_label,
        )
    absolute_axis.set_yticks(np.arange(len(absolute_metrics)))
    absolute_axis.set_yticklabels([metric_labels[item] for item in absolute_metrics])
    absolute_axis.invert_yaxis()
    absolute_axis.set_xlim(0, 0.70)
    absolute_axis.set_xlabel("Estimate with component-bootstrap 95% CI")
    absolute_axis.set_title(f"{pathway.capitalize()} pathway: absolute performance")
    absolute_axis.grid(axis="x", alpha=0.25)
    absolute_axis.legend(frameon=False, loc="lower right")

    difference_axis = axes[row, 1]
    difference_subset = (
        paired_stack_vs_single_ci_df.loc[
            paired_stack_vs_single_ci_df["pathway"].eq(pathway)
            & paired_stack_vs_single_ci_df["metric"].isin(comparison_metrics)
        ]
        .set_index("metric").reindex(comparison_metrics)
    )
    differences = difference_subset["paired_difference"].to_numpy(dtype=float)
    lower = difference_subset["paired_ci_95_lower"].to_numpy(dtype=float)
    upper = difference_subset["paired_ci_95_upper"].to_numpy(dtype=float)
    positions = np.arange(len(comparison_metrics))
    difference_axis.errorbar(
        differences,
        positions,
        xerr=np.vstack([
            np.maximum(0, differences - lower),
            np.maximum(0, upper - differences),
        ]),
        fmt="o",
        color=pathway_colours[pathway],
        capsize=3,
        linewidth=1.3,
        markersize=6,
    )
    difference_axis.axvline(0, color="#111827", linestyle="--", linewidth=1)
    difference_axis.set_yticks(positions)
    difference_axis.set_yticklabels([metric_labels[item] for item in comparison_metrics])
    difference_axis.invert_yaxis()
    difference_axis.set_xlabel("Strict stack minus best single model (95% CI)")
    difference_axis.set_title(f"{pathway.capitalize()} pathway: paired differences")
    difference_axis.grid(axis="x", alpha=0.25)

figure.suptitle(
    "Primary internal-validation performance under dependency-component resampling",
    fontsize=14,
    fontweight="bold",
)
figure.savefig(
    V2_FIGURE_DIR / "results_primary_component_bootstrap.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()



### Prespecified corrected sensitivity analyses

Each sensitivity repeats the strict outer-fold stack procedure with the same dependency-grouped seeds. The stack and its fold-selected best-single comparator are retrained inside every outer fold. The antibiotic-summary sensitivity applies only to the patient pathway because the predictor is absent from the environmental pathway.


In [ ]:
SENSITIVITY_SCENARIOS = [
    "one_exact_copy",
    "adults_only",
    "exclude_irregular_matched_sets",
    "exclude_resolved_any_abx_summary",
    "exclude_largest_dependency_component",
]


def sensitivity_frame_and_features(pathway, scenario):
    frame = V2_ENGINEERED[pathway].loc[
        V2_PARTITIONS[pathway]["development"].index
    ].copy()
    feature_sets = copy.deepcopy(V2_FEATURE_SETS[pathway])
    detail = ""

    if scenario == "one_exact_copy":
        frame = frame.loc[frame["exact_duplicate_keep_flag"]].copy()
        detail = "Retained the first copy of each exact source-row duplicate."
    elif scenario == "adults_only":
        frame = frame.loc[frame["age_clean"].ge(18)].copy()
        detail = "Restricted to observations with numeric age >=18 years."
    elif scenario == "exclude_irregular_matched_sets":
        frame = frame.loc[frame["matched_set_design_valid"]].copy()
        detail = "Excluded all source-release matched sets outside the nominal design."
    elif scenario == "exclude_resolved_any_abx_summary":
        if pathway != "patient":
            return None, None, "Not applicable: any_abx is not an environmental-pathway predictor."
        for feature_set_name, features in feature_sets.items():
            feature_sets[feature_set_name] = [
                feature for feature in features
                if feature != "any_abx_0_60_resolved"
                and not feature.endswith("__any_abx_0_60_resolved")
            ]
        detail = "Removed the class-derived any-antibiotic summary and its pathogen interactions."
    elif scenario == "exclude_largest_dependency_component":
        largest_component = frame["dependency_component"].value_counts().idxmax()
        largest_n = int(frame["dependency_component"].eq(largest_component).sum())
        frame = frame.loc[~frame["dependency_component"].eq(largest_component)].copy()
        detail = f"Excluded largest development component ({largest_n} observations)."
    else:
        raise KeyError(scenario)

    return frame, feature_sets, detail


def sensitivity_point_metrics(predictions, score_column):
    averaged = average_repeated_oof(predictions, score_column)
    return bootstrap_metric_vector(averaged, "score", BOOTSTRAP_CAPACITY_FRACTION)


sensitivity_inclusion_rows = []
sensitivity_result_rows = []

for pathway in PATHWAY_DATA:
    primary_repeat_zero = V2_STACK_PREDICTIONS[pathway].loc[
        V2_STACK_PREDICTIONS[pathway]["repeat"].eq(0)
    ].copy()
    primary_stack_metrics = sensitivity_point_metrics(primary_repeat_zero, "probability")
    primary_single_metrics = sensitivity_point_metrics(primary_repeat_zero, "best_single_probability")

    for scenario in SENSITIVITY_SCENARIOS:
        frame, feature_sets, detail = sensitivity_frame_and_features(pathway, scenario)
        if frame is None:
            sensitivity_inclusion_rows.append({
                "pathway": pathway, "scenario": scenario, "status": "not_applicable",
                "observations": np.nan, "events": np.nan, "dependency_components": np.nan,
                "detail": detail,
            })
            continue
        if frame[TARGET].nunique() < 2 or frame["dependency_component"].nunique() < V2_SETTINGS["outer_splits"]:
            raise RuntimeError(f"Sensitivity {pathway}/{scenario} cannot support the configured grouped CV.")

        sensitivity_inclusion_rows.append({
            "pathway": pathway,
            "scenario": scenario,
            "status": "executed",
            "observations": len(frame),
            "events": int(frame[TARGET].sum()),
            "dependency_components": frame["dependency_component"].nunique(),
            "detail": detail,
        })
        predictions, folds, _ = strict_outer_stack_evaluation(
            pathway,
            frame,
            scenario=scenario,
            outer_repeats=V2_SETTINGS["sensitivity_outer_repeats"],
            feature_sets=feature_sets,
            build_final_bundle=False,
        )
        atomic_write_csv(V2_TABLE_DIR / f"sensitivity_oof_{pathway}_{scenario}.csv", predictions)
        atomic_write_csv(V2_TABLE_DIR / f"sensitivity_folds_{pathway}_{scenario}.csv", folds)

        stack_metrics = sensitivity_point_metrics(predictions, "probability")
        single_metrics = sensitivity_point_metrics(predictions, "best_single_probability")
        for model_label, metrics, reference in [
            ("Strict nested stack", stack_metrics, primary_stack_metrics),
            ("Fold-selected best single model", single_metrics, primary_single_metrics),
        ]:
            sensitivity_result_rows.append({
                "pathway": pathway,
                "scenario": scenario,
                "model": model_label,
                "observations": len(frame),
                "events": int(frame[TARGET].sum()),
                "dependency_components": frame["dependency_component"].nunique(),
                "capacity_fraction": BOOTSTRAP_CAPACITY_FRACTION,
                **metrics,
                **{f"delta_vs_primary_{metric}": metrics[metric] - reference[metric] for metric in BOOTSTRAP_METRICS},
            })

sensitivity_inclusion_audit_df = pd.DataFrame(sensitivity_inclusion_rows)
sensitivity_results_df = pd.DataFrame(sensitivity_result_rows)
atomic_write_csv(V2_TABLE_DIR / "sensitivity_inclusion_audit.csv", sensitivity_inclusion_audit_df)
atomic_write_csv(V2_TABLE_DIR / "strict_sensitivity_results.csv", sensitivity_results_df)
display(sensitivity_inclusion_audit_df)
display(sensitivity_results_df.round(4))

# Main-text figure: change from the corresponding primary outer repeat.
scenario_labels = {
    "one_exact_copy": "One exact copy",
    "adults_only": "Adults only",
    "exclude_irregular_matched_sets": "Exclude irregular sets",
    "exclude_resolved_any_abx_summary": "Remove antibiotic summary",
    "exclude_largest_dependency_component": "Exclude largest component",
}
sensitivity_metrics = [
    ("delta_vs_primary_AUROC", "Change in AUROC"),
    ("delta_vs_primary_AUPRC", "Change in AUPRC"),
    ("delta_vs_primary_precision", "Change in precision at 10%"),
]
sensitivity_plot_df = sensitivity_results_df.loc[
    sensitivity_results_df["model"].eq("Strict nested stack")
].copy()
sensitivity_plot_df["scenario_label"] = sensitivity_plot_df["scenario"].map(scenario_labels)
scenario_order = [scenario_labels[item] for item in SENSITIVITY_SCENARIOS]
pathway_colours = {"patient": "#D97706", "environmental": "#2563A6"}
pathway_offsets = {"patient": -0.10, "environmental": 0.10}

figure, axes = plt.subplots(1, 3, figsize=(12, 5.8), sharey=True, constrained_layout=True)
for axis, (metric, title) in zip(axes, sensitivity_metrics):
    for pathway in ["patient", "environmental"]:
        subset = sensitivity_plot_df.loc[
            sensitivity_plot_df["pathway"].eq(pathway)
        ].copy()
        subset["position"] = subset["scenario_label"].map(
            {label: index for index, label in enumerate(scenario_order)}
        )
        subset = subset.dropna(subset=["position", metric])
        axis.scatter(
            subset[metric],
            subset["position"] + pathway_offsets[pathway],
            color=pathway_colours[pathway],
            s=42,
            label=pathway.capitalize(),
            zorder=3,
        )
    axis.axvline(0, color="#111827", linestyle="--", linewidth=1)
    axis.set_title(title)
    axis.set_xlabel("Difference from primary repeat")
    axis.grid(axis="x", alpha=0.25)

axes[0].set_yticks(np.arange(len(scenario_order)))
axes[0].set_yticklabels(scenario_order)
axes[0].invert_yaxis()
axes[-1].legend(frameon=False, loc="best")
figure.suptitle(
    "Prespecified sensitivity analyses",
    fontsize=14,
    fontweight="bold",
)
figure.savefig(
    V2_FIGURE_DIR / "results_sensitivity_analyses.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()



In [ ]:
def pathogen_context_sensitivity(pathway):
    development = V2_ENGINEERED[pathway].loc[
        V2_PARTITIONS[pathway]["development"].index
    ].copy()
    rows = []
    for repeat in range(V2_SETTINGS["outer_repeats"]):
        splitter = BalancedComponentKFold(
            V2_SETTINGS["outer_splits"],
            random_state=RANDOM_STATE + repeat * 1000,
            n_trials=128,
        )
        split_indices = list(splitter.split(
            development, development[TARGET], development["dependency_component"]
        ))
        for feature_set in ["full", "without_pathogen_context"]:
            features = V2_FEATURE_SETS[pathway][feature_set]
            specification = V2ModelSpec(
                "Logistic regression", feature_set, {"C": 0.1}
            )
            predictions = np.full(len(development), np.nan)
            for fold, (training, validation) in enumerate(split_indices):
                estimator = make_v2_classifier(
                    specification, features, RANDOM_STATE + repeat * 1000 + fold
                )
                estimator.fit(development.iloc[training][features], development.iloc[training][TARGET])
                predictions[validation] = estimator.predict_proba(
                    development.iloc[validation][features]
                )[:, 1]
            repeat_frame = pd.DataFrame({
                "source_index": development.index,
                "y_true": development[TARGET].to_numpy(),
                "probability": predictions,
                "run": development["run"].to_numpy(),
            })
            metrics = pooled_repeat_metrics(
                pathway,
                feature_set,
                repeat_frame.assign(
                    repeat=repeat,
                    dependency_component=development["dependency_component"].to_numpy(),
                ),
            ).iloc[0].to_dict()
            rows.append({**metrics, "feature_set": feature_set})
    return pd.DataFrame(rows)


v2_pathogen_context_sensitivity_df = pd.concat([
    pathogen_context_sensitivity(pathway) for pathway in PATHWAY_DATA
], ignore_index=True)
v2_pathogen_context_summary_df = (
    v2_pathogen_context_sensitivity_df.groupby(["pathway", "feature_set"], as_index=False)
    .agg(
        mean_AUROC=("AUROC", "mean"),
        mean_AUPRC=("AUPRC", "mean"),
        mean_macro_pathogen_AUROC=("macro_pathogen_AUROC", "mean"),
        mean_macro_pathogen_AUPRC_lift=("macro_pathogen_AUPRC_lift", "mean"),
        mean_recall_at_5pct=("recall_at_capacity", "mean"),
    )
)
v2_pathogen_context_summary_df.to_csv(
    V2_TABLE_DIR / "pathogen_context_sensitivity.csv", index=False
)
figure, axis = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=v2_pathogen_context_summary_df,
    x="mean_macro_pathogen_AUPRC_lift",
    y="feature_set",
    hue="pathway",
    ax=axis,
)
axis.set_title("Sensitivity to pathogen context")
axis.set_xlabel("Mean pathogen-macro AUPRC lift")
axis.set_ylabel("")
axis.grid(axis="x", alpha=0.25)
figure.tight_layout()
figure.savefig(V2_FIGURE_DIR / "pathogen_context_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()
display(v2_pathogen_context_summary_df.round(4))


## 16. Locked-model refitting, calibration and operating policies


In [ ]:
def modal_spec_for_family(pathway, family):
    folds = V2_NESTED_FOLDS[(pathway, family)]
    serialized = folds["best_spec"].apply(
        lambda value: json.dumps(value, sort_keys=True) if isinstance(value, dict) else str(value)
    )
    chosen = serialized.value_counts().index[0]
    payload = json.loads(chosen)
    return V2ModelSpec(**payload)


V2_LOCKED_BASE_MODELS = {}
V2_LOCKED_PAIRWISE = {}
V2_LOCKED_MULTITASK = {}

for pathway in PATHWAY_DATA:
    development = V2_ENGINEERED[pathway].loc[V2_PARTITIONS[pathway]["development"].index]
    V2_LOCKED_BASE_MODELS[pathway] = {}
    for family in V2_MODEL_GRIDS:
        spec = modal_spec_for_family(pathway, family)
        features = V2_FEATURE_SETS[pathway][spec.feature_set]
        cache_key = strict_cache_key(
            "locked_base_model", pathway=pathway, family=family,
            spec=asdict(spec), row_hash=stable_hash(development.index.astype(str).tolist()),
        )
        cache_file = V2_CACHE_DIR / f"locked_base_{cache_key}.joblib"
        if cache_file.exists():
            bundle = joblib.load(cache_file)
        else:
            estimator = make_v2_classifier(spec, features, RANDOM_STATE)
            estimator.fit(development[features], development[TARGET])
            bundle = {"estimator": estimator, "features": features, "spec": spec}
            atomic_joblib_dump(cache_file, bundle)
        V2_LOCKED_BASE_MODELS[pathway][family] = bundle

    if V2_RUN_PAIRWISE_RANKER:
        selected_params = json.loads(V2_PAIRWISE_FOLDS[pathway]["best_params"].value_counts().index[0])
        cache_key = strict_cache_key(
            "locked_pairwise_model", pathway=pathway, params=selected_params,
            row_hash=stable_hash(development.index.astype(str).tolist()),
        )
        cache_file = V2_CACHE_DIR / f"locked_pairwise_{cache_key}.joblib"
        if cache_file.exists():
            V2_LOCKED_PAIRWISE[pathway] = joblib.load(cache_file)
        else:
            V2_LOCKED_PAIRWISE[pathway] = fit_xgb_pairwise_ranker(
                development, V2_FEATURE_SETS[pathway]["full"], selected_params, RANDOM_STATE
            )
            atomic_joblib_dump(cache_file, V2_LOCKED_PAIRWISE[pathway])
    if V2_RUN_MULTITASK:
        selected_config = json.loads(V2_MULTITASK_FOLDS[pathway]["best_config"].value_counts().index[0])
        selected_config["hidden"] = tuple(selected_config["hidden"])
        cache_key = strict_cache_key(
            "locked_multitask_model", pathway=pathway, config=selected_config,
            row_hash=stable_hash(development.index.astype(str).tolist()),
        )
        cache_file = V2_CACHE_DIR / f"locked_multitask_{cache_key}.joblib"
        if cache_file.exists():
            V2_LOCKED_MULTITASK[pathway] = joblib.load(cache_file)
        else:
            V2_LOCKED_MULTITASK[pathway] = fit_multitask_model(
                development, pathway, RANDOM_STATE, **selected_config
            )
            atomic_joblib_dump(cache_file, V2_LOCKED_MULTITASK[pathway])

print("Development-only base models and strict stack bundles refitted.")


In [ ]:
def v2_base_predictions(pathway, frame):
    predictions = {}
    for family, bundle in V2_LOCKED_BASE_MODELS[pathway].items():
        predictions[family] = bundle["estimator"].predict_proba(
            frame[bundle["features"]]
        )[:, 1]
    if pathway in V2_LOCKED_PAIRWISE:
        predictions["Pairwise ranker"] = predict_xgb_pairwise_ranker(
            V2_LOCKED_PAIRWISE[pathway], frame
        )
    if pathway in V2_LOCKED_MULTITASK:
        predictions["Multi-task neural network"] = predict_multitask_model(
            V2_LOCKED_MULTITASK[pathway], frame
        )
    return pd.DataFrame(predictions, index=frame.index)


def v2_stack_raw_score(pathway, frame):
    return predict_strict_stack_bundle(
        STRICT_STACK_FINAL_BUNDLES[pathway], frame, calibrated=False
    )


def v2_locked_raw_score(pathway, frame):
    """Dispatch to the candidate locked by development-only internal comparison."""
    candidate = V2_FINAL_CANDIDATES[pathway]
    if candidate == "Strict nested stack":
        return v2_stack_raw_score(pathway, frame)
    if candidate == "Pairwise ranker":
        return predict_xgb_pairwise_ranker(V2_LOCKED_PAIRWISE[pathway], frame)
    if candidate == "Multi-task neural network":
        return predict_multitask_model(V2_LOCKED_MULTITASK[pathway], frame)
    if candidate in V2_LOCKED_BASE_MODELS[pathway]:
        bundle = V2_LOCKED_BASE_MODELS[pathway][candidate]
        return bundle["estimator"].predict_proba(frame[bundle["features"]])[:, 1]
    raise KeyError(f"No locked scorer for {pathway}: {candidate}")


class RawScoreLogisticCalibrator:
    """Logistic calibration for unrestricted ranking or decision scores."""

    def __init__(self):
        self.location = 0.0
        self.scale = 1.0
        self.model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)

    def _transform(self, raw_score):
        values = np.asarray(raw_score, dtype=float)
        return ((values - self.location) / self.scale).reshape(-1, 1)

    def fit(self, raw_score, y_true):
        values = np.asarray(raw_score, dtype=float)
        self.location = float(np.nanmedian(values))
        scale = float(np.nanstd(values))
        self.scale = scale if np.isfinite(scale) and scale > 1e-12 else 1.0
        self.model.fit(self._transform(values), np.asarray(y_true, dtype=int))
        return self

    def predict(self, raw_score):
        return self.model.predict_proba(self._transform(raw_score))[:, 1]


def make_locked_calibrator(pathway, raw_score, y_true):
    candidate = V2_FINAL_CANDIDATES[pathway]
    if candidate == "Pairwise ranker":
        return (
            RawScoreLogisticCalibrator().fit(raw_score, y_true),
            "logistic calibration of standardised ranking score",
        )
    return (
        PlattLogitCalibrator().fit(raw_score, y_true),
        "Platt scaling on logit-transformed probability",
    )


V2_CALIBRATORS = {}
V2_CALIBRATION_METHODS = {}
V2_CALIBRATION_AUDIT = []
for pathway in PATHWAY_DATA:
    calibration = V2_ENGINEERED[pathway].loc[
        V2_PARTITIONS[pathway]["probability_calibration"].index
    ]
    raw = v2_locked_raw_score(pathway, calibration)
    calibrator, method = make_locked_calibrator(pathway, raw, calibration[TARGET])
    V2_CALIBRATORS[pathway] = calibrator
    V2_CALIBRATION_METHODS[pathway] = method
    calibrated = calibrator.predict(raw)
    V2_CALIBRATION_AUDIT.append({
        "pathway": pathway,
        "locked_candidate": V2_FINAL_CANDIDATES[pathway],
        "calibration_method": method,
        "partition": "probability_calibration",
        "evidence_role": "deployment calibrator fit only",
        "interpretation": (
            "Apparent fit diagnostic on the calibrator-fitting partition. The primary internal "
            "calibration evidence is the strict outer-fold cross-fitted table."
        ),
        **probability_metrics(calibration[TARGET], calibrated),
    })

v2_calibration_audit_df = pd.DataFrame(V2_CALIBRATION_AUDIT)
atomic_write_csv(V2_TABLE_DIR / "probability_calibrator_apparent_fit.csv", v2_calibration_audit_df)
display(v2_calibration_audit_df.round(4))


In [ ]:
V2_CAPACITY_GRID = [0.01, 0.02, 0.05, 0.10, 0.20]
TARGET_RECALL_AT_CAPACITY = 0.10
MAX_REVIEW_FRACTION = 0.10
SELECTIVE_TARGET_ERROR = 0.20
SELECTIVE_MIN_COVERAGE = 0.10


def pathogen_capacity_curve(y_true, score, pathogen, fractions=V2_CAPACITY_GRID):
    rows = []
    for fraction in fractions:
        metrics = within_pathogen_capacity(y_true, score, pathogen, fraction)
        rows.append({"capacity_fraction": fraction, **metrics})
    return pd.DataFrame(rows)


def choose_capacity_policy(y_true, score, pathogen):
    table = pathogen_capacity_curve(y_true, score, pathogen, V2_CAPACITY_GRID)
    feasible = table.loc[table["capacity_fraction"].le(MAX_REVIEW_FRACTION)]
    candidates = feasible.loc[feasible["recall_at_capacity"].ge(TARGET_RECALL_AT_CAPACITY)]
    if len(candidates):
        chosen = candidates.sort_values(["capacity_fraction", "precision_at_capacity"], ascending=[True, False]).iloc[0]
        rule = f"smallest pre-specified capacity with recall >= {TARGET_RECALL_AT_CAPACITY:.2f}"
        status = "target_met"
    else:
        chosen = feasible.sort_values(["recall_at_capacity", "precision_at_capacity"], ascending=False).iloc[0]
        rule = "descriptive highest recall within maximum review fraction"
        status = "target_not_met"
    return table, {**chosen.to_dict(), "selection_rule": rule, "policy_status": status}


def selective_risk_coverage(y_true, probability, thresholds=None):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    confidence = np.abs(probability - 0.5) * 2
    prediction = (probability >= 0.5).astype(int)
    if thresholds is None:
        thresholds = np.unique(np.r_[np.linspace(0, 0.95, 40), confidence])
    rows = []
    for threshold in thresholds:
        selected = confidence >= threshold
        if selected.sum() == 0:
            continue
        error = np.mean(prediction[selected] != y_true[selected])
        rows.append({
            "confidence_threshold": float(threshold),
            "coverage": float(selected.mean()),
            "selective_error": float(error),
            "selective_accuracy": float(1 - error),
            "selected_n": int(selected.sum()),
        })
    return pd.DataFrame(rows)


def choose_selective_policy(y_true, probability):
    table = selective_risk_coverage(y_true, probability)
    eligible = table.loc[
        table["selective_error"].le(SELECTIVE_TARGET_ERROR)
        & table["coverage"].ge(SELECTIVE_MIN_COVERAGE)
    ]
    if len(eligible):
        chosen = eligible.sort_values(["coverage", "selective_error"], ascending=[False, True]).iloc[0]
        return table, {
            **chosen.to_dict(),
            "selection_rule": "maximum coverage subject to target selective error",
            "policy_status": "target_met",
        }
    descriptive = table.sort_values(["selective_error", "coverage"], ascending=[True, False]).iloc[0]
    return table, {
        **descriptive.to_dict(),
        "confidence_threshold": np.nan,
        "selection_rule": "no threshold locked because the prespecified target was not met",
        "policy_status": "target_not_met",
    }


V2_CAPACITY_POLICIES = {}

V2_SELECTIVE_POLICIES = {}

V2_CAPACITY_CURVES = {}

V2_SELECTIVE_CURVES = {}
v2_operating_rows = []
for pathway in PATHWAY_DATA:
    operating = V2_ENGINEERED[pathway].loc[V2_PARTITIONS[pathway]["operating_point"].index]
    raw = v2_locked_raw_score(pathway, operating)
    calibrated = V2_CALIBRATORS[pathway].predict(raw)
    capacity_table, capacity_policy = choose_capacity_policy(
        operating[TARGET], raw, operating["run"]
    )
    selective_table, selective_policy = choose_selective_policy(operating[TARGET], calibrated)
    V2_CAPACITY_POLICIES[pathway] = capacity_policy

    V2_SELECTIVE_POLICIES[pathway] = selective_policy

    V2_CAPACITY_CURVES[pathway] = capacity_table.assign(pathway=pathway)

    V2_SELECTIVE_CURVES[pathway] = selective_table.assign(pathway=pathway)
    capacity_table.assign(pathway=pathway).to_csv(V2_TABLE_DIR / f"capacity_curve_operating_{pathway}.csv", index=False)
    selective_table.assign(pathway=pathway).to_csv(V2_TABLE_DIR / f"selective_risk_coverage_operating_{pathway}.csv", index=False)
    v2_operating_rows.append({"pathway": pathway, "policy_type": "capacity", **capacity_policy})
    v2_operating_rows.append({"pathway": pathway, "policy_type": "selective", **selective_policy})

v2_operating_policy_df = pd.DataFrame(v2_operating_rows)
v2_operating_policy_df.to_csv(V2_TABLE_DIR / "v2_locked_operating_policies.csv", index=False)
display(v2_operating_policy_df)

v2_capacity_curves_df = pd.concat(V2_CAPACITY_CURVES.values(), ignore_index=True)
v2_selective_curves_df = pd.concat(V2_SELECTIVE_CURVES.values(), ignore_index=True)
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
sns.lineplot(
    data=v2_capacity_curves_df, x="capacity_fraction", y="recall_at_capacity",
    hue="pathway", marker="o", ax=axes[0]
)
axes[0].axhline(TARGET_RECALL_AT_CAPACITY, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Recall by review capacity")
sns.lineplot(
    data=v2_selective_curves_df, x="coverage", y="selective_error",
    hue="pathway", ax=axes[1]
)
axes[1].axhline(SELECTIVE_TARGET_ERROR, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Selective risk-coverage")
for axis in axes:
    axis.grid(alpha=0.25)
figure.tight_layout()
figure.savefig(V2_FIGURE_DIR / "operating_policy_curves.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
def finite_sample_quantile(values, alpha=0.10):
    values = np.sort(np.asarray(values, dtype=float))
    if not len(values):
        raise ValueError("The conformal calibration pool is empty.")
    rank = int(np.ceil((len(values) + 1) * (1 - alpha)))
    return float(values[min(rank, len(values)) - 1])


def build_component_conformal_pool(frame, probability, alpha=0.10):
    probability = np.asarray(probability, dtype=float)
    true_label_score = np.where(frame[TARGET].to_numpy(int) == 1, 1 - probability, probability)
    component_scores = (
        pd.DataFrame({
            "component": frame["dependency_component"].to_numpy(),
            "score": true_label_score,
        })
        .groupby("component", sort=False)["score"]
        .max()
        .to_numpy()
    )
    return {
        "alpha": float(alpha),
        "component_scores": component_scores,
        "threshold": finite_sample_quantile(component_scores, alpha),
        "components": int(len(component_scores)),
        "coverage_target": "marginal complete-component coverage",
    }


def component_conformal_sets(frame, probability, pool, alpha=None):
    probability = np.asarray(probability, dtype=float)
    alpha = pool["alpha"] if alpha is None else float(alpha)
    component_scores = np.asarray(pool["component_scores"], dtype=float)
    threshold = finite_sample_quantile(component_scores, alpha)
    score_control = probability
    score_case = 1 - probability
    include_control = score_control <= threshold
    include_case = score_case <= threshold
    ordered_scores = np.sort(component_scores)
    p_control = (
        1 + len(ordered_scores) - np.searchsorted(ordered_scores, score_control, side="left")
    ) / (len(ordered_scores) + 1)
    p_case = (
        1 + len(ordered_scores) - np.searchsorted(ordered_scores, score_case, side="left")
    ) / (len(ordered_scores) + 1)
    labels = np.select(
        [include_control & include_case, include_control, include_case],
        ["{Control, Case}", "{Control}", "{Case}"],
        default="{}",
    )
    result = pd.DataFrame({
        "conformal_p_control": p_control,
        "conformal_p_case": p_case,
        "conformal_prediction_set": labels,
        "include_control": include_control,
        "include_case": include_case,
    })
    result["prediction_set_size"] = result[["include_control", "include_case"]].sum(axis=1)
    result["uncertainty_flag"] = np.select(
        [result["prediction_set_size"].eq(2), result["prediction_set_size"].eq(1)],
        ["Ambiguous", "Singleton"],
        default="Empty",
    )
    return result


V2_CONFORMAL_POOLS = {}
for pathway in PATHWAY_DATA:
    conformal = V2_ENGINEERED[pathway].loc[
        V2_PARTITIONS[pathway]["conformal_calibration"].index
    ]
    raw = v2_locked_raw_score(pathway, conformal)
    probability = V2_CALIBRATORS[pathway].predict(raw)
    V2_CONFORMAL_POOLS[pathway] = build_component_conformal_pool(
        conformal, probability, alpha=0.10
    )

pd.DataFrame([
    {
        "pathway": pathway,
        "calibration_components": pool["components"],
        "alpha": pool["alpha"],
        "threshold": pool["threshold"],
        "coverage_target": pool["coverage_target"],
    }
    for pathway, pool in V2_CONFORMAL_POOLS.items()
]).to_csv(V2_TABLE_DIR / "component_conformal_calibration.csv", index=False)
print("Pathway-level component conformal calibration completed.")


## 17. Non-confirmatory compatibility analysis

The locked HOI-TRUST pipeline is applied to a previously examined holdout solely as a compatibility analysis. Its labels are not used for model selection, calibration, capacity selection or conformal calibration, and its results are not described as independent external validation.


In [ ]:
def apply_capacity_policy(score, policy, pathogen):
    score = np.asarray(score, dtype=float)
    pathogen = np.asarray(pathogen)
    fraction = float(policy["capacity_fraction"])
    selected = np.zeros(len(score), dtype=bool)
    boundaries = {}
    for value in pd.unique(pathogen):
        positions = np.flatnonzero(pathogen == value)
        count = max(1, int(np.ceil(len(positions) * fraction)))
        boundary = np.sort(score[positions])[-count]
        selected[positions] = score[positions] >= boundary
        boundaries[str(value)] = float(boundary)
    return selected, json.dumps(boundaries, sort_keys=True)


V2_LEGACY_RESULTS = {}
legacy_metric_rows, legacy_conformal_rows, legacy_capacity_rows = [], [], []
if V2_RUN_LEGACY_COMPATIBILITY:
    for pathway in PATHWAY_DATA:
        test = V2_ENGINEERED[pathway].loc[V2_PARTITIONS[pathway]["legacy_final_test"].index].copy().reset_index(drop=True)
        raw = v2_locked_raw_score(pathway, test)
        probability = V2_CALIBRATORS[pathway].predict(raw)
        conformal = component_conformal_sets(test, probability, V2_CONFORMAL_POOLS[pathway])
        selected, boundary = apply_capacity_policy(
            raw, V2_CAPACITY_POLICIES[pathway], test["run"]
        )
        confidence = np.abs(probability - 0.5) * 2
        selective_policy = V2_SELECTIVE_POLICIES[pathway]
        if selective_policy["policy_status"] == "target_met":
            abstain = confidence < float(selective_policy["confidence_threshold"])
        else:
            abstain = np.ones(len(test), dtype=bool)
        result = test[[TARGET, "run", "sex", "age_clean", "matched_set_id", "dependency_component"]].copy()
        result["v2_raw_score"] = raw
        result["v2_calibrated_score"] = probability
        result["capacity_selected"] = selected
        result["selective_abstain"] = abstain
        result = pd.concat([result.reset_index(drop=True), conformal], axis=1)
        result["covered"] = np.where(result[TARGET].eq(1), result["include_case"], result["include_control"])
        V2_LEGACY_RESULTS[pathway] = result

        legacy_metric_rows.append({
            "pathway": pathway, "evaluation_status": "non-confirmatory legacy compatibility",
            **probability_metrics(result[TARGET], result["v2_calibrated_score"]),
            **pathogen_capacity_curve(
                result[TARGET], result["v2_raw_score"], result["run"], [0.05]
            ).iloc[0].to_dict(),
        })
        selected_subset = result.loc[result["capacity_selected"]]
        legacy_capacity_rows.append({
            "pathway": pathway,
            "locked_capacity_fraction": V2_CAPACITY_POLICIES[pathway]["capacity_fraction"],
            "score_boundary": boundary,
            "actual_selected_fraction": result["capacity_selected"].mean(),
            "precision": selected_subset[TARGET].mean(),
            "recall": selected_subset[TARGET].sum() / result[TARGET].sum(),
            "number_needed_to_review": 1 / selected_subset[TARGET].mean(),
        })
        legacy_conformal_rows.append({
            "pathway": pathway, "nominal_coverage": 0.90,
            "observation_coverage": result["covered"].mean(),
            "component_complete_coverage": result.groupby("dependency_component")["covered"].all().mean(),
            "singleton_fraction": result["uncertainty_flag"].eq("Singleton").mean(),
            "ambiguous_fraction": result["uncertainty_flag"].eq("Ambiguous").mean(),
            "empty_fraction": result["uncertainty_flag"].eq("Empty").mean(),
            "selective_nonabstention_fraction": (~result["selective_abstain"]).mean(),
            "selective_error_nonabstained": np.mean(
                (result.loc[~result["selective_abstain"], "v2_calibrated_score"] >= 0.5).astype(int)
                != result.loc[~result["selective_abstain"], TARGET]
            ) if (~result["selective_abstain"]).any() else np.nan,
        })

    v2_legacy_metrics_df = pd.DataFrame(legacy_metric_rows)
    v2_legacy_capacity_df = pd.DataFrame(legacy_capacity_rows)
    v2_legacy_conformal_df = pd.DataFrame(legacy_conformal_rows)
    v2_legacy_metrics_df.to_csv(V2_TABLE_DIR / "v2_legacy_holdout_metrics_NOT_CONFIRMATORY.csv", index=False)
    v2_legacy_capacity_df.to_csv(V2_TABLE_DIR / "v2_legacy_capacity_metrics_NOT_CONFIRMATORY.csv", index=False)
    v2_legacy_conformal_df.to_csv(V2_TABLE_DIR / "v2_legacy_uncertainty_metrics_NOT_CONFIRMATORY.csv", index=False)
    display(v2_legacy_metrics_df.round(4))
    display(v2_legacy_capacity_df.round(4))
    display(v2_legacy_conformal_df.round(4))


    pathway_order = [
        pathway for pathway in ["patient", "environmental"]
        if pathway in v2_legacy_conformal_df["pathway"].values
    ]

    coverage_plot = v2_legacy_conformal_df.melt(
        id_vars="pathway",
        value_vars=["observation_coverage", "component_complete_coverage"],
        var_name="coverage_level",
        value_name="coverage",
    )
    coverage_plot["coverage_level"] = coverage_plot["coverage_level"].map({
        "observation_coverage": "Observation",
        "component_complete_coverage": "Complete component",
    })

    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    sns.barplot(
        data=coverage_plot, x="pathway", y="coverage", hue="coverage_level",
        order=pathway_order, ax=axes[0],
    )
    axes[0].axhline(0.90, color="black", linestyle="--", linewidth=1, label="Nominal 90%")
    axes[0].set_title("Coverage")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Coverage")
    axes[0].set_ylim(0, 1)
    axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[0].legend(title="Level", loc="lower right")

    composition = (
        v2_legacy_conformal_df.set_index("pathway").reindex(pathway_order)[
            ["singleton_fraction", "ambiguous_fraction", "empty_fraction"]
        ]
    )
    bottom = np.zeros(len(composition))
    for column, label, colour in [
        ("singleton_fraction", "Singleton", "#4C78A8"),
        ("ambiguous_fraction", "Ambiguous", "#F2A541"),
        ("empty_fraction", "Empty", "#E45756"),
    ]:
        values = composition[column].fillna(0).to_numpy()
        axes[1].bar(composition.index, values, bottom=bottom, label=label, color=colour)
        bottom += values
    axes[1].set_title("Prediction-set composition")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Proportion")
    axes[1].set_ylim(0, 1)
    axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
    axes[1].legend(title="Set type", loc="lower right")

    figure.suptitle("Conformal uncertainty in the legacy compatibility set", fontsize=14)
    figure.savefig(
        V2_FIGURE_DIR / "results_conformal_compatibility.png",
        dpi=300, bbox_inches="tight", facecolor="white",
    )
    plt.show()


## 18. Heterogeneity, subgroup analysis and model reliance


In [ ]:
def subgroup_v2_metrics(result, subgroup, minimum_cases=30, minimum_controls=30):
    rows = []
    for value, subset in result.groupby(subgroup, dropna=False):
        cases = int(subset[TARGET].sum())
        controls = int((1 - subset[TARGET]).sum())
        row = {"subgroup": subgroup, "value": str(value), "n": len(subset), "cases": cases, "controls": controls}
        row["adequate_support"] = cases >= minimum_cases and controls >= minimum_controls
        if subset[TARGET].nunique() == 2:
            row.update(probability_metrics(subset[TARGET], subset["v2_calibrated_score"]))
            row.update(pathogen_capacity_curve(
                subset[TARGET], subset["v2_raw_score"], subset["run"], [0.05]
            ).iloc[0].to_dict())
            row["conformal_coverage"] = subset["covered"].mean()
            row["abstention_fraction"] = subset["selective_abstain"].mean()
        rows.append(row)
    return pd.DataFrame(rows)


if V2_LEGACY_RESULTS:
    v2_subgroup_frames = []
    for pathway, result in V2_LEGACY_RESULTS.items():
        working = result.copy()
        working["age_band"] = pd.cut(
            working["age_clean"], [-np.inf, 39, 59, 69, 79, np.inf],
            labels=["<40", "40-59", "60-69", "70-79", "80+"]
        ).astype("string").fillna("Missing")
        for subgroup in ["run", "sex", "age_band"]:
            v2_subgroup_frames.append(subgroup_v2_metrics(working, subgroup).assign(pathway=pathway))
    v2_subgroup_metrics_df = pd.concat(v2_subgroup_frames, ignore_index=True)
    v2_subgroup_metrics_df.to_csv(V2_TABLE_DIR / "v2_legacy_subgroup_robustness_NOT_CONFIRMATORY.csv", index=False)
    display(v2_subgroup_metrics_df.loc[v2_subgroup_metrics_df["adequate_support"]].round(4))

stack_coefficient_rows = []
for pathway, meta_model in V2_STACK_META_MODELS.items():
    coefficients = meta_model.named_steps["model"].coef_[0]
    for feature, coefficient in zip(V2_STACK_BASE_COLUMNS[pathway], coefficients):
        stack_coefficient_rows.append({"pathway": pathway, "base_model": feature, "meta_logit_coefficient": coefficient})
v2_stack_coefficients_df = pd.DataFrame(stack_coefficient_rows)
v2_stack_coefficients_df.to_csv(V2_TABLE_DIR / "v2_stack_meta_coefficients.csv", index=False)
display(v2_stack_coefficients_df.sort_values(["pathway", "meta_logit_coefficient"], ascending=[True, False]))

def v2_explanation_domains(pathway):
    full = V2_FEATURE_SETS[pathway]["full"]
    domains = {
        "Demographic and surgery": [
            c for c in full if c in {
                "age_clean", "age_squared_scaled", "age_missing", "age_band_v2",
                "sex", "any_surgery"
            }
        ],
        "Comorbidity": [c for c in full if c.startswith("elix_")],
        "Pathogen context and interactions": [
            c for c in full if c == "run" or c.startswith("INT__")
        ],
        "Prior antibiotic exposure": [
            c for c in full
            if pathway == "patient" and (
                c in model_antibiotic_cols or c.startswith("antibiotic_")
            )
        ],
        "Colonisation pressure": [
            c for c in full
            if pathway == "environmental" and (
                c in cp_cols or c.startswith("cp_") or c.startswith("target_cp")
            )
        ],
    }
    return {name: list(dict.fromkeys(columns)) for name, columns in domains.items() if columns}


def grouped_permutation_importance_locked_candidate(pathway, repeats=20, seed=RANDOM_STATE):
    """Model-agnostic domain importance on the independent operating partition."""
    evaluation = V2_ENGINEERED[pathway].loc[
        V2_PARTITIONS[pathway]["operating_point"].index
    ].copy()
    baseline_score = v2_locked_raw_score(pathway, evaluation)
    baseline_auprc = average_precision_score(evaluation[TARGET], baseline_score)
    rng = np.random.default_rng(seed + (0 if pathway == "patient" else 1000))
    rows = []
    for domain, columns in v2_explanation_domains(pathway).items():
        deltas = []
        for repetition in range(repeats):
            permuted = evaluation.copy()
            permutation = rng.permutation(len(permuted))
            # Joint permutation retains relationships within the domain while
            # breaking its association with the outcome and other domains.
            permuted.loc[:, columns] = evaluation.iloc[permutation][columns].to_numpy()
            permuted_score = v2_locked_raw_score(pathway, permuted)
            permuted_auprc = average_precision_score(evaluation[TARGET], permuted_score)
            deltas.append(baseline_auprc - permuted_auprc)
        rows.append({
            "pathway": pathway,
            "locked_candidate": V2_FINAL_CANDIDATES[pathway],
            "domain": domain,
            "n_columns": len(columns),
            "baseline_operating_AUPRC": baseline_auprc,
            "mean_AUPRC_decrease": float(np.mean(deltas)),
            "sd_AUPRC_decrease": float(np.std(deltas, ddof=1)) if len(deltas) > 1 else np.nan,
            "lower_2_5": float(np.quantile(deltas, 0.025)),
            "upper_97_5": float(np.quantile(deltas, 0.975)),
            "interpretation": "Model reliance/association; not causal importance",
        })
    return pd.DataFrame(rows)


importance_repeats = 5 if V2_RUN_PROFILE == "smoke" else 20
importance_key = strict_cache_key(
    "locked_grouped_permutation_importance",
    repeats=importance_repeats,
    final_candidates=V2_FINAL_CANDIDATES,
)
importance_cache_file = V2_CACHE_DIR / f"grouped_importance_{importance_key}.csv"
if importance_cache_file.exists():
    v2_locked_candidate_importance_df = pd.read_csv(importance_cache_file)
else:
    v2_locked_candidate_importance_df = pd.concat([
        grouped_permutation_importance_locked_candidate(
            pathway, repeats=importance_repeats
        )
        for pathway in PATHWAY_DATA
    ], ignore_index=True)
    atomic_write_csv(importance_cache_file, v2_locked_candidate_importance_df)
atomic_write_csv(
    V2_TABLE_DIR / "v2_locked_candidate_grouped_permutation_importance.csv",
    v2_locked_candidate_importance_df,
)
display(v2_locked_candidate_importance_df.sort_values(
    ["pathway", "mean_AUPRC_decrease"], ascending=[True, False]
).round(4))


# Publication figure: grouped permutation reliance
domain_order = (
    v2_locked_candidate_importance_df.groupby("domain")["mean_AUPRC_decrease"]
    .mean().sort_values().index.tolist()
)
pathway_order = [
    pathway for pathway in ["patient", "environmental"]
    if pathway in v2_locked_candidate_importance_df["pathway"].values
]
colours = {"patient": "#F58518", "environmental": "#4C78A8"}
offsets = {"patient": 0.12, "environmental": -0.12}

figure, axis = plt.subplots(figsize=(9, 5))
for pathway in pathway_order:
    subset = v2_locked_candidate_importance_df.loc[
        v2_locked_candidate_importance_df["pathway"].eq(pathway)
    ].copy()
    y_positions = np.array([domain_order.index(value) for value in subset["domain"]]) + offsets[pathway]
    means = subset["mean_AUPRC_decrease"].to_numpy()
    lower = subset["lower_2_5"].to_numpy()
    upper = subset["upper_97_5"].to_numpy()
    errors = np.vstack([np.maximum(0, means - lower), np.maximum(0, upper - means)])
    axis.errorbar(
        means, y_positions, xerr=errors, fmt="o", markersize=7, capsize=3,
        linewidth=1.5, color=colours[pathway], label=pathway,
    )

axis.axvline(0, color="black", linestyle="--", linewidth=1)
axis.set_yticks(range(len(domain_order)))
axis.set_yticklabels(domain_order)
axis.set_xlabel("Decrease in AUPRC after joint permutation")
axis.set_ylabel("")
axis.set_title("Grouped permutation reliance")
axis.legend(title="Pathway")
axis.grid(axis="x", alpha=0.25)
axis.grid(axis="y", visible=False)
figure.tight_layout()
figure.savefig(
    V2_FIGURE_DIR / "results_grouped_permutation_reliance.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()


In [ ]:
v2_within_pathogen_rows = []
for pathway, result in V2_LEGACY_RESULTS.items():
    for pathogen, subset in result.groupby("run"):
        row = {
            "pathway": pathway,
            "run": pathogen,
            "observations": len(subset),
            "events": int(subset[TARGET].sum()),
            "nonevents": int((1 - subset[TARGET]).sum()),
            "evaluation_status": "non-confirmatory legacy compatibility",
        }
        if subset[TARGET].nunique() == 2:
            row.update(probability_metrics(subset[TARGET], subset["v2_calibrated_score"]))
            row.update(capacity_curve(
                subset[TARGET], subset["v2_raw_score"], [0.05]
            ).iloc[0].to_dict())
            row["observation_conformal_coverage"] = subset["covered"].mean()
            row["component_complete_conformal_coverage"] = (
                subset.groupby("dependency_component")["covered"].all().mean()
            )
        v2_within_pathogen_rows.append(row)

v2_within_pathogen_metrics_df = pd.DataFrame(v2_within_pathogen_rows)
v2_within_pathogen_metrics_df.to_csv(
    V2_TABLE_DIR / "legacy_within_pathogen_metrics_NOT_CONFIRMATORY.csv", index=False
)

pathogen_order = (
    v2_within_pathogen_metrics_df.groupby("run")["AUPRC_lift"]
    .mean().sort_values(ascending=False).index
)
v2_within_pathogen_metrics_df["run"] = pd.Categorical(
    v2_within_pathogen_metrics_df["run"], categories=pathogen_order, ordered=True
)

figure, axis = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=v2_within_pathogen_metrics_df.sort_values("run"),
    x="AUPRC_lift", y="run", hue="pathway", size="observations",
    sizes=(45, 190), alpha=0.9, ax=axis,
)
axis.axvline(1.0, color="black", linestyle="--", linewidth=1, label="No-skill performance")
axis.set_title("Within-pathogen AUPRC lift in the legacy compatibility set")
axis.set_xlabel("AUPRC lift over pathogen-specific case proportion")
axis.set_ylabel("Pathogen target")
axis.grid(axis="x", alpha=0.25)
axis.grid(axis="y", visible=False)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
figure.tight_layout()
figure.savefig(
    V2_FIGURE_DIR / "results_within_pathogen_heterogeneity.png",
    dpi=300, bbox_inches="tight", facecolor="white",
)
plt.show()
display(v2_within_pathogen_metrics_df.round(4))


## 19. Independent validation protocol and executable adapter


In [ ]:
EXTERNAL_REQUIRED_COLUMNS = [
    "site_id", "admission_id", "patient_id", "prediction_timestamp", "analysis", "run",
    "outcome_binary", "outcome_window_start", "outcome_window_end",
    "age", "sex", "any_surgery",
]
EXTERNAL_OPTIONAL_OPERATIONAL_COLUMNS = [
    "ward_id", "hospital_id", "admission_timestamp", "discharge_timestamp",
    "culture_timestamp", "label_maturity_timestamp", "data_extract_timestamp",
    "race_ethnicity", "deprivation_group", "device_exposure", "length_of_stay_at_prediction",
]
EXTERNAL_FEATURE_COLUMNS = sorted(set(
    basic_cols + antibiotic_cols + cp_cols + elix_cols
))

external_data_contract_df = pd.DataFrame([
    {"column": column, "status": "required", "description": "Required external validation field"}
    for column in EXTERNAL_REQUIRED_COLUMNS
] + [
    {"column": column, "status": "model_feature_if_available", "description": "Map exactly to development definition and look-back window"}
    for column in EXTERNAL_FEATURE_COLUMNS
] + [
    {"column": column, "status": "recommended", "description": "Operational, transportability, fairness or monitoring field"}
    for column in EXTERNAL_OPTIONAL_OPERATIONAL_COLUMNS
]).drop_duplicates("column")
external_data_contract_df.to_csv(V2_PROTOCOL_DIR / "external_all_admission_data_contract.csv", index=False)

external_acquisition_checklist_df = pd.DataFrame([
    {"domain": "Governance", "requirement": "Research ethics/IRB approval and data-sharing agreement", "status": "Required before data access"},
    {"domain": "Population", "requirement": "All eligible adult admissions; no matched control sampling", "status": "Required"},
    {"domain": "Timing", "requirement": "Prediction timestamp and feature availability enforced before scoring", "status": "Required"},
    {"domain": "Outcome", "requirement": "Independent microbiology phenotype mapping and 3-30 day outcome definition", "status": "Required"},
    {"domain": "Transportability", "requirement": "Site, ward and calendar period retained under governance", "status": "Strongly recommended"},
    {"domain": "Fairness", "requirement": "Collect legally and ethically permissible sociodemographic variables", "status": "Strongly recommended"},
    {"domain": "Blinding", "requirement": "Freeze model, mapping and SAP before opening outcome labels", "status": "Required"},
    {"domain": "Clinical review", "requirement": "Infection prevention and microbiology sign-off on variable definitions", "status": "Required"},
])
external_acquisition_checklist_df.to_csv(V2_PROTOCOL_DIR / "external_cohort_acquisition_checklist.csv", index=False)
display(external_data_contract_df.head(25))
display(external_acquisition_checklist_df)


In [ ]:
MIMIC_IV_SQL_TEMPLATE = r"""
-- HOI-TRUST MIMIC-IV feasibility extraction skeleton.
-- Validate all table and item mappings against the installed MIMIC-IV version.
WITH admissions AS (
  SELECT subject_id, hadm_id, admittime, dischtime, admission_location, race
  FROM `physionet-data.mimiciv_hosp.admissions`
),
patients AS (
  SELECT subject_id, gender, anchor_age
  FROM `physionet-data.mimiciv_hosp.patients`
),
transfers AS (
  SELECT subject_id, hadm_id, transfer_id, careunit, intime, outtime
  FROM `physionet-data.mimiciv_hosp.transfers`
),
micro AS (
  SELECT subject_id, hadm_id, chartdate, charttime, spec_type_desc,
         org_name, ab_name, interpretation
  FROM `physionet-data.mimiciv_hosp.microbiologyevents`
),
prescriptions AS (
  SELECT subject_id, hadm_id, starttime, stoptime, drug, route
  FROM `physionet-data.mimiciv_hosp.prescriptions`
)
SELECT a.*, p.gender, p.anchor_age
FROM admissions a
JOIN patients p USING (subject_id);
"""
(V2_PROTOCOL_DIR / "mimic_iv_feasibility_extraction.sql").write_text(MIMIC_IV_SQL_TEMPLATE)
print("Generated:", V2_PROTOCOL_DIR / "mimic_iv_feasibility_extraction.sql")


In [ ]:
external_sap_df = pd.DataFrame([
    {"section": "Population", "primary_plan": "Consecutive all-eligible admissions at each external site"},
    {"section": "Index time", "primary_plan": "Prospectively reproducible prediction timestamp before outcome window"},
    {"section": "Model", "primary_plan": "Frozen v2 pipeline, feature mapping, capacity and uncertainty policies"},
    {"section": "Missing data", "primary_plan": "Report patterns; apply frozen preprocessing; no outcome-informed imputation"},
    {"section": "Primary performance", "primary_plan": "Calibration, discrimination and locked capacity metrics with clustered 95% CIs"},
    {"section": "Clinical utility", "primary_plan": "Decision curves only at clinician-approved thresholds in representative prevalence"},
    {"section": "Heterogeneity", "primary_plan": "Site, pathogen, sex, age, race/ethnicity and deprivation where adequately supported"},
    {"section": "Uncertainty", "primary_plan": "Coverage, set size and selective error overall and by pathogen"},
    {"section": "Recalibration", "primary_plan": "Frozen; intercept-only; intercept+slope; revision reported separately"},
    {"section": "Multiplicity", "primary_plan": "Primary estimands pre-specified; subgroup analyses labelled secondary/exploratory"},
    {"section": "Sample size", "primary_plan": "Determine from precision targets for calibration, discrimination and net benefit before extraction"},
    {"section": "Reporting", "primary_plan": "TRIPOD+AI and PROBAST+AI evaluation domains"},
])
external_sap_df.to_csv(V2_PROTOCOL_DIR / "external_validation_statistical_analysis_plan.csv", index=False)

EXTERNAL_SAP_MARKDOWN = (
    "# HOI-TRUST External Validation Statistical Analysis Plan\n\n"
    + "\n".join(
        f"## {row.section}\n{row.primary_plan}\n" for row in external_sap_df.itertuples()
    )
    + "\n## Sample-size determination\n"
    + "Use a dedicated external-validation sample-size calculation targeting precision of calibration-in-the-large, "
      "calibration slope, AUROC and net benefit. Inputs must include anticipated external prevalence, "
      "linear-predictor variance, plausible miscalibration and clinically relevant thresholds. The final calculation "
      "should be reviewed by a medical statistician and frozen before outcome-label access.\n"
)
(V2_PROTOCOL_DIR / "HOI_TRUST_external_validation_SAP.md").write_text(EXTERNAL_SAP_MARKDOWN)
display(external_sap_df)


In [ ]:
from statistics import NormalDist


def required_n_for_binomial_precision(probability, half_width, confidence=0.95):
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    p = float(probability)
    return int(np.ceil((z ** 2) * p * (1 - p) / (half_width ** 2)))


def hanley_mcneil_auc_se(auc, n_events, n_nonevents):
    """Large-sample planning approximation; final planning should use specialist software."""
    auc = float(auc)
    q1 = auc / (2 - auc)
    q2 = 2 * auc * auc / (1 + auc)
    variance = (
        auc * (1 - auc)
        + (n_events - 1) * (q1 - auc * auc)
        + (n_nonevents - 1) * (q2 - auc * auc)
    ) / (n_events * n_nonevents)
    return float(np.sqrt(max(variance, 0)))


def required_n_for_auc_precision(prevalence, auc, half_width=0.03, confidence=0.95):
    z = NormalDist().inv_cdf(0.5 + confidence / 2)
    low, high = 100, 10_000_000
    while low < high:
        mid = (low + high) // 2
        n_events = max(2, int(round(mid * prevalence)))
        n_nonevents = max(2, mid - n_events)
        achieved = z * hanley_mcneil_auc_se(auc, n_events, n_nonevents)
        if achieved <= half_width:
            high = mid
        else:
            low = mid + 1
    return int(low)


sample_size_rows = []
for prevalence in [0.01, 0.02, 0.05, 0.10]:
    prevalence_n = required_n_for_binomial_precision(prevalence, half_width=0.005)
    for anticipated_auc in [0.60, 0.65, 0.70]:
        auc_n = required_n_for_auc_precision(
            prevalence, anticipated_auc, half_width=0.03
        )
        sample_size_rows.append({
            "anticipated_prevalence": prevalence,
            "anticipated_AUROC": anticipated_auc,
            "prevalence_half_width_target": 0.005,
            "AUROC_half_width_target": 0.03,
            "n_for_prevalence_precision": prevalence_n,
            "n_for_AUROC_precision_approx": auc_n,
            "screening_minimum_n": max(prevalence_n, auc_n),
            "anticipated_events_at_screening_minimum": int(
                np.ceil(max(prevalence_n, auc_n) * prevalence)
            ),
            "status": "Scenario screen only — formal external-validation calculation required",
        })

external_sample_size_scenarios_df = pd.DataFrame(sample_size_rows)
external_sample_size_scenarios_df.to_csv(
    V2_PROTOCOL_DIR / "external_validation_sample_size_scenarios.csv", index=False
)
display(external_sample_size_scenarios_df)


In [ ]:
EXTERNAL_DIR = BASE_DIR / "external"
EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
EXTERNAL_COHORT_PATH = EXTERNAL_DIR / "hoi_trust_external_all_admissions.csv"


def validate_external_schema(frame):
    missing = sorted(set(EXTERNAL_REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise KeyError(f"External cohort missing required columns: {missing}")
    if frame["admission_id"].isna().any() or frame["patient_id"].isna().any():
        raise ValueError("External admission_id and patient_id must be complete.")
    if not set(pd.to_numeric(frame["outcome_binary"], errors="coerce").dropna().unique()).issubset({0, 1}):
        raise ValueError("outcome_binary must be 0/1 or missing for immature labels.")
    return True


def prepare_scoring_frame(frame, pathway):
    """Map an authorised external/shadow batch to the frozen development schema.

    Missing source predictors are never hidden: they are logged in an audit table,
    then represented as missing so the locked pipeline's pre-specified imputation
    can operate. This supports engineering tests, not a claim of transportability.
    """
    if pathway not in PATHWAY_DATA:
        raise ValueError(f"Unsupported analysis pathway: {pathway}")
    working = frame.copy()
    expected_source = set(elix_cols + ["age", "sex", "any_surgery", "run"])
    expected_source.update(antibiotic_cols if pathway == "patient" else cp_cols)
    missing_source = sorted(expected_source - set(working.columns))
    audit = pd.DataFrame([
        {
            "pathway": pathway,
            "column": column,
            "present_in_input": column in working.columns,
            "action": "mapped" if column in working.columns else "inserted_as_missing_for_locked_imputer",
        }
        for column in sorted(expected_source)
    ])
    for column in missing_source:
        working[column] = np.nan
    working["age_clean"] = pd.to_numeric(
        working["age"].astype("string").str.replace(">", "", regex=False).str.replace("+", "", regex=False),
        errors="coerce",
    )
    working["analysis"] = pathway
    if "site_id" not in working:
        working["site_id"] = "UNSPECIFIED_SITE"
    if "patient_id" not in working:
        raise KeyError("patient_id is required for dependency and audit control")
    if "admission_id" not in working:
        raise KeyError("admission_id is required for record linkage and audit control")
    working["dependency_component"] = (
        working["site_id"].astype(str) + "::" + working["patient_id"].astype(str)
    )
    working["matched_set_id"] = working["admission_id"].astype(str)
    working["matched_set_design_valid"] = False
    engineered = engineer_v2_features(working, pathway)
    return engineered, audit


def bootstrap_external_metrics(result, cluster="patient_id", repetitions=None):
    repetitions = repetitions or V2_SETTINGS["bootstrap"]
    rng = np.random.default_rng(RANDOM_STATE)
    clusters = result[cluster].dropna().unique()
    rows = []
    for _ in range(repetitions):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        sample = pd.concat([result.loc[result[cluster].eq(value)] for value in sampled], ignore_index=True)
        mature = sample.dropna(subset=["outcome_binary"])
        if mature["outcome_binary"].nunique() < 2:
            continue
        rows.append({
            **probability_metrics(mature["outcome_binary"], mature["calibrated_score"]),
            **pathogen_capacity_curve(
                mature["outcome_binary"], mature["raw_score"], mature["run"], [0.05]
            ).iloc[0].to_dict(),
        })
    boot = pd.DataFrame(rows)
    return pd.DataFrame([
        {"metric": c, "lower_95": boot[c].quantile(.025), "upper_95": boot[c].quantile(.975)}
        for c in boot.select_dtypes(include="number").columns
    ])


EXTERNAL_VALIDATION_RESULTS = {}
if V2_RUN_EXTERNAL_VALIDATION and EXTERNAL_COHORT_PATH.exists():
    external_raw = pd.read_csv(EXTERNAL_COHORT_PATH, low_memory=False)
    validate_external_schema(external_raw)
    external_provenance = {
        "path": str(EXTERNAL_COHORT_PATH), "bytes": EXTERNAL_COHORT_PATH.stat().st_size,
        "sha256": sha256_file(EXTERNAL_COHORT_PATH),
        "loaded_utc": datetime.now(timezone.utc).isoformat(),
    }
    (V2_PROTOCOL_DIR / "external_cohort_provenance.json").write_text(json.dumps(external_provenance, indent=2))
    external_raw["analysis"] = external_raw["analysis"].astype("string").str.strip().str.lower()
    unsupported = sorted(set(external_raw["analysis"].dropna()) - set(PATHWAY_DATA))
    if unsupported:
        raise ValueError(f"Unsupported external pathway labels: {unsupported}")

    mapping_audits = []
    for pathway, pathway_frame in external_raw.groupby("analysis"):
        pathway_frame = pathway_frame.copy().reset_index(drop=True)
        engineered, mapping_audit = prepare_scoring_frame(pathway_frame, pathway)
        mapping_audits.append(mapping_audit)
        raw_score = v2_locked_raw_score(pathway, engineered)
        calibrated = V2_CALIBRATORS[pathway].predict(raw_score)
        uncertainty = component_conformal_sets(engineered, calibrated, V2_CONFORMAL_POOLS[pathway])
        result = pathway_frame.copy().reset_index(drop=True)
        result["raw_score"] = raw_score
        result["calibrated_score"] = calibrated
        result = pd.concat([result, uncertainty], axis=1)
        result["model_version"] = PROJECT_VERSION
        result["locked_candidate"] = V2_FINAL_CANDIDATES[pathway]
        EXTERNAL_VALIDATION_RESULTS[pathway] = result
        result.to_csv(V2_TABLE_DIR / f"PRIVATE_external_scored_{pathway}.csv", index=False)

        mature = result.dropna(subset=["outcome_binary"]).copy()
        if mature["outcome_binary"].nunique() == 2:
            metrics = pd.DataFrame([{
                "pathway": pathway, **probability_metrics(mature["outcome_binary"], mature["calibrated_score"]),
                **pathogen_capacity_curve(
                mature["outcome_binary"], mature["raw_score"], mature["run"], [0.05]
            ).iloc[0].to_dict(),
            }])
            ci = bootstrap_external_metrics(mature)
            metrics.to_csv(V2_TABLE_DIR / f"external_validation_metrics_{pathway}.csv", index=False)
            ci.to_csv(V2_TABLE_DIR / f"external_validation_cluster_bootstrap_ci_{pathway}.csv", index=False)
            display(metrics.round(4)); display(ci.round(4))
    if mapping_audits:
        external_feature_mapping_audit_df = pd.concat(mapping_audits, ignore_index=True)
        external_feature_mapping_audit_df.to_csv(
            V2_TABLE_DIR / "external_feature_mapping_audit.csv", index=False
        )
        display(external_feature_mapping_audit_df)
else:
    print("No authorised external all-admission cohort found. Acquisition pack and executable adapter have been created.")


## 20. Prospective shadow-mode protocol


In [ ]:
shadow_event_schema_df = pd.DataFrame([
    {"field": "event_id", "required": True, "purpose": "Immutable prediction event identifier"},
    {"field": "model_version", "required": True, "purpose": "Traceability"},
    {"field": "site_id", "required": True, "purpose": "Transportability and monitoring"},
    {"field": "patient_hash", "required": True, "purpose": "Pseudonymous longitudinal linkage"},
    {"field": "admission_hash", "required": True, "purpose": "Prediction opportunity linkage"},
    {"field": "prediction_timestamp", "required": True, "purpose": "Latency and temporal integrity"},
    {"field": "run", "required": True, "purpose": "Target pathogen"},
    {"field": "raw_score", "required": True, "purpose": "Ranking"},
    {"field": "calibrated_score", "required": True, "purpose": "Probability monitoring after external calibration"},
    {"field": "capacity_selected", "required": True, "purpose": "Review burden"},
    {"field": "conformal_prediction_set", "required": True, "purpose": "Uncertainty"},
    {"field": "selective_abstain", "required": True, "purpose": "Safe non-recommendation"},
    {"field": "feature_completeness", "required": True, "purpose": "Data-quality monitoring"},
    {"field": "scoring_latency_ms", "required": True, "purpose": "Operational performance"},
    {"field": "outcome_binary", "required": False, "purpose": "Added only after label maturation"},
    {"field": "outcome_maturity_timestamp", "required": False, "purpose": "Prevent premature evaluation"},
])
shadow_event_schema_df.to_csv(V2_PROTOCOL_DIR / "shadow_mode_event_schema.csv", index=False)

shadow_acceptance_gates_df = pd.DataFrame([
    {"domain": "Availability", "metric": "Successful scoring opportunities", "gate": ">= 99%"},
    {"domain": "Latency", "metric": "Predictions delivered within agreed batch window", "gate": ">= 99%"},
    {"domain": "Completeness", "metric": "Critical feature completeness", "gate": ">= 98%"},
    {"domain": "Workload", "metric": "Observed review fraction versus locked capacity", "gate": "Within pre-specified tolerance"},
    {"domain": "Calibration", "metric": "Calibration-in-the-large after label maturity", "gate": "95% CI includes clinically acceptable range"},
    {"domain": "Discrimination", "metric": "External AUROC/AUPRC", "gate": "Meets pre-specified minimum with CI"},
    {"domain": "Uncertainty", "metric": "Pathogen-specific conformal coverage", "gate": "At or above nominal within precision tolerance"},
    {"domain": "Selective safety", "metric": "Error among non-abstained cases", "gate": "At or below locked target"},
    {"domain": "Fairness", "metric": "No materially unsafe subgroup degradation", "gate": "Clinical/statistical review"},
    {"domain": "Drift", "metric": "Feature and score drift", "gate": "No unresolved red alerts"},
])
shadow_acceptance_gates_df.to_csv(V2_PROTOCOL_DIR / "shadow_mode_acceptance_gates.csv", index=False)
display(shadow_event_schema_df)
display(shadow_acceptance_gates_df)


In [ ]:
def salted_hash(value, salt):
    return hashlib.sha256((str(salt) + "::" + str(value)).encode()).hexdigest()


def score_shadow_batch(batch, pathway, salt, model_version=PROJECT_VERSION):
    """Score a non-interventional shadow batch and return pseudonymised events.

    This function does not write to an EHR, send an alert, or influence care.
    """
    started = time.perf_counter()
    required = [
        "site_id", "patient_id", "admission_id", "prediction_timestamp",
        "run", "age", "sex", "any_surgery",
    ]
    missing = sorted(set(required) - set(batch.columns))
    if missing:
        raise KeyError(f"Shadow batch missing required fields: {missing}")
    working = batch.copy().reset_index(drop=True)
    engineered, feature_mapping_audit = prepare_scoring_frame(working, pathway)
    raw = v2_locked_raw_score(pathway, engineered)
    probability = V2_CALIBRATORS[pathway].predict(raw)
    uncertainty = component_conformal_sets(
        engineered, probability, V2_CONFORMAL_POOLS[pathway]
    )
    selected, _ = apply_capacity_policy(
        raw, V2_CAPACITY_POLICIES[pathway], working["run"]
    )
    confidence = np.abs(probability - 0.5) * 2
    selective_policy = V2_SELECTIVE_POLICIES[pathway]

    if selective_policy["policy_status"] == "target_met":

        abstain = confidence < float(selective_policy["confidence_threshold"])

    else:

        abstain = np.ones(len(working), dtype=bool)
    latency_ms = (time.perf_counter() - started) * 1000

    model_features = V2_FEATURE_SETS[pathway]["compact"]
    completeness = engineered[model_features].notna().mean(axis=1)
    missing_mapping_count = int((~feature_mapping_audit["present_in_input"]).sum())
    events = pd.DataFrame({
        "event_id": [
            hashlib.sha256(f"{model_version}:{i}:{time.time_ns()}".encode()).hexdigest()
            for i in range(len(working))
        ],
        "model_version": model_version,
        "locked_candidate": V2_FINAL_CANDIDATES[pathway],
        "site_id": working["site_id"].astype(str),
        "patient_hash": working["patient_id"].map(lambda x: salted_hash(x, salt)),
        "admission_hash": working["admission_id"].map(lambda x: salted_hash(x, salt)),
        "prediction_timestamp": working["prediction_timestamp"],
        "run": working["run"],
        "raw_score": raw,
        "calibrated_score": probability,
        "capacity_selected": selected,
        "conformal_prediction_set": uncertainty["conformal_prediction_set"],
        "selective_abstain": abstain,
        "feature_completeness": completeness,
        "missing_mapped_source_columns": missing_mapping_count,
        "scoring_latency_ms": latency_ms / max(len(working), 1),
    })
    return events, feature_mapping_audit


In [ ]:
def population_stability_index(reference, current, bins=10):
    reference = pd.Series(reference).dropna().astype(float)
    current = pd.Series(current).dropna().astype(float)
    if not len(reference) or not len(current):
        return np.nan
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    ref_counts, _ = np.histogram(reference, bins=edges)
    cur_counts, _ = np.histogram(current, bins=edges)
    ref_p = np.clip(ref_counts / max(ref_counts.sum(), 1), 1e-6, None)
    cur_p = np.clip(cur_counts / max(cur_counts.sum(), 1), 1e-6, None)
    return float(np.sum((cur_p - ref_p) * np.log(cur_p / ref_p)))


def shadow_drift_report(reference_events, current_events):
    rows = []
    for column in ["raw_score", "calibrated_score", "feature_completeness", "scoring_latency_ms"]:
        rows.append({
            "metric": column,
            "psi": population_stability_index(reference_events[column], current_events[column]),
            "reference_mean": pd.to_numeric(reference_events[column], errors="coerce").mean(),
            "current_mean": pd.to_numeric(current_events[column], errors="coerce").mean(),
        })
    report = pd.DataFrame(rows)
    report["drift_level"] = pd.cut(
        report["psi"], [-np.inf, 0.10, 0.25, np.inf], labels=["low", "investigate", "high"]
    ).astype("string")
    return report


def mature_shadow_performance(events):
    mature = events.dropna(subset=["outcome_binary"]).copy()
    if mature["outcome_binary"].nunique() < 2:
        return pd.DataFrame(), pd.DataFrame()
    overall = pd.DataFrame([{
        **probability_metrics(mature["outcome_binary"], mature["calibrated_score"]),
        **capacity_curve(mature["outcome_binary"], mature["raw_score"], [0.05]).iloc[0].to_dict(),
        "observed_alert_fraction": mature["capacity_selected"].mean(),
        "abstention_fraction": mature["selective_abstain"].mean(),
    }])
    by_pathogen = pd.DataFrame([
        {"run": run, "n": len(group), **probability_metrics(group["outcome_binary"], group["calibrated_score"])}
        for run, group in mature.groupby("run") if group["outcome_binary"].nunique() == 2
    ])
    return overall, by_pathogen


## 21. Human-factors and safety evaluation plan


In [ ]:
decide_ai_protocol_df = pd.DataFrame([
    {"domain": "Intended use", "measure": "Rank surveillance review queue; no autonomous diagnosis or treatment"},
    {"domain": "Users", "measure": "Infection prevention, microbiology and antimicrobial stewardship professionals"},
    {"domain": "Setting", "measure": "Specified hospitals, wards, hours and data refresh cycle"},
    {"domain": "Human-AI interaction", "measure": "What is shown, hidden, explained and editable"},
    {"domain": "Workflow", "measure": "Review time, records reviewed, interruptions and downstream actions"},
    {"domain": "Performance", "measure": "Human alone, AI alone and human-AI team where feasible"},
    {"domain": "Safety", "measure": "Missed cases, false alerts, inappropriate escalation and delay"},
    {"domain": "Trust", "measure": "Appropriate acceptance and override, not simple agreement"},
    {"domain": "Equity", "measure": "Differential alerts, overrides and actions across supported subgroups"},
    {"domain": "Usability", "measure": "Structured usability testing and qualitative interviews"},
    {"domain": "Adaptation", "measure": "No online learning during evaluation; all changes version controlled"},
    {"domain": "Reporting", "measure": "DECIDE-AI plus study-design-specific guidance"},
])
decide_ai_protocol_df.to_csv(V2_PROTOCOL_DIR / "early_live_clinical_evaluation_protocol.csv", index=False)
display(decide_ai_protocol_df)


In [ ]:
clinical_hazard_log_df = pd.DataFrame([
    {"hazard_id": "H01", "hazard": "High-risk acquisition not prioritised", "cause": "Model error, missing data or drift", "control": "Human review remains standard; abstention; monitoring; incident review", "residual_status": "Open until clinical validation"},
    {"hazard_id": "H02", "hazard": "Excessive false alerts", "cause": "Capacity mismatch or prevalence shift", "control": "Locked capacity policy; workload gate; no threshold changes without approval", "residual_status": "Open"},
    {"hazard_id": "H03", "hazard": "Stale or future information used", "cause": "Timestamp or ETL error", "control": "Feature-availability assertions and event-time audit", "residual_status": "Open"},
    {"hazard_id": "H04", "hazard": "Wrong patient/admission linkage", "cause": "Identifier mapping error", "control": "Pseudonymous linkage validation, reconciliation and immutable audit log", "residual_status": "Open"},
    {"hazard_id": "H05", "hazard": "Misleading explanation causes inappropriate action", "cause": "Causal interpretation of association", "control": "Explanation labelling, user training and action restrictions", "residual_status": "Open"},
    {"hazard_id": "H06", "hazard": "Subgroup performance degradation", "cause": "Representation or measurement differences", "control": "Pre-specified subgroup monitoring and stop rule", "residual_status": "Open"},
    {"hazard_id": "H07", "hazard": "Silent model or data pipeline change", "cause": "Uncontrolled update", "control": "Version pinning, hashes, change control and rollback", "residual_status": "Open"},
    {"hazard_id": "H08", "hazard": "Service unavailable", "cause": "Infrastructure failure", "control": "Fail-safe return to standard workflow; availability monitoring", "residual_status": "Open"},
])
clinical_hazard_log_df.to_csv(V2_PROTOCOL_DIR / "clinical_hazard_log_starter.csv", index=False)

model_change_policy_df = pd.DataFrame([
    {"change_class": "No model change", "example": "Infrastructure patch with identical predictions", "required_evidence": "Regression tests and hash comparison"},
    {"change_class": "Recalibration", "example": "Intercept update after external validation", "required_evidence": "Pre-specified recalibration analysis and safety review"},
    {"change_class": "Threshold/capacity change", "example": "Different review budget", "required_evidence": "Workflow approval, retrospective analysis and shadow validation"},
    {"change_class": "Feature mapping change", "example": "New antibiotic coding", "required_evidence": "Data validation, model impact analysis and versioned release"},
    {"change_class": "Model revision", "example": "Retraining or architecture change", "required_evidence": "New development, external validation and clinical safety case"},
])
model_change_policy_df.to_csv(V2_PROTOCOL_DIR / "model_change_control_policy.csv", index=False)
display(clinical_hazard_log_df)
display(model_change_policy_df)


In [ ]:
# Evidence status at submission

evidence_gate_df = pd.DataFrame([
    {
        "stage": "Retrospective model development",
        "status": "Completed",
        "evidence": "Frozen cohort, leakage controls, dependency-disjoint partitions, nested validation and reproducible artefacts",
    },
    {
        "stage": "Internal validation",
        "status": "Completed",
        "evidence": "Repeated grouped out-of-fold comparison, calibration, capacity, uncertainty, subgroup and sensitivity analyses",
    },
    {
        "stage": "Independent all-admission validation",
        "status": "Protocol prepared",
        "evidence": "External data contract, executable adapter and prespecified statistical analysis plan",
    },
    {
        "stage": "Prospective workflow evaluation",
        "status": "Protocol prepared",
        "evidence": "Shadow-mode event schema, monitoring plan and human-factors evaluation framework",
    },
])
evidence_gate_df.to_csv(V2_PROTOCOL_DIR / "evidence_gate_status.csv", index=False)
display(evidence_gate_df)
print("Evidence status: retrospective development and internal validation complete.")


## 22. Reproducibility manifest and model artefacts


In [ ]:
v2_model_card_df = pd.DataFrame([
    {
        "pathway": pathway,
        "locked_candidate": V2_FINAL_CANDIDATES[pathway],
        "calibration_method": V2_CALIBRATION_METHODS[pathway],
        "development_evidence": "Strict fold-contained dependency-component-grouped outer validation",
        "stacking_scope": "Base selection, level-one construction, meta-fit and internal calibration occur inside each outer training fold",
        "selection_interpretation": "Internal model-family choice; independent validation remains required",
        "operating_policy": V2_CAPACITY_POLICIES[pathway]["selection_rule"],
        "capacity_policy_status": V2_CAPACITY_POLICIES[pathway]["policy_status"],
        "selective_policy_status": V2_SELECTIVE_POLICIES[pathway]["policy_status"],
        "uncertainty": "Pathway-level component conformal calibration",
        "intended_use": INTENDED_USE,
        "absolute_risk_claim": False,
        "external_validation_status": "Required before clinical interpretation",
        "prospective_status": "Shadow-mode protocol only; not conducted",
    }
    for pathway in PATHWAY_DATA
])
atomic_write_csv(V2_PROTOCOL_DIR / "locked_model_cards.csv", v2_model_card_df)

# Save model objects. Torch state is stored separately from preprocessing objects.
for pathway in PATHWAY_DATA:
    for family, bundle in V2_LOCKED_BASE_MODELS.get(pathway, {}).items():
        joblib.dump(bundle, V2_MODEL_DIR / f"{pathway}_{family.replace(' ', '_')}_locked.joblib")
    if pathway in V2_LOCKED_PAIRWISE:
        joblib.dump(V2_LOCKED_PAIRWISE[pathway], V2_MODEL_DIR / f"{pathway}_pairwise_ranker_locked.joblib")
    if pathway in V2_LOCKED_MULTITASK:
        multitask_bundle = V2_LOCKED_MULTITASK[pathway].copy()
        model = multitask_bundle.pop("model")
        joblib.dump(multitask_bundle, V2_MODEL_DIR / f"{pathway}_multitask_preprocessing.joblib")
        torch.save(model.state_dict(), V2_MODEL_DIR / f"{pathway}_multitask_state_dict.pt")
    if pathway in STRICT_STACK_FINAL_BUNDLES:
        joblib.dump(
            STRICT_STACK_FINAL_BUNDLES[pathway],
            V2_MODEL_DIR / f"{pathway}_strict_nested_stack_bundle.joblib",
        )
    joblib.dump(V2_CALIBRATORS[pathway], V2_MODEL_DIR / f"{pathway}_locked_score_calibrator.joblib")
    joblib.dump(V2_CONFORMAL_POOLS[pathway], V2_MODEL_DIR / f"{pathway}_component_conformal_pool.joblib")

v2_model_artifact_manifest = [
    {
        "path": str(path.relative_to(V2_DIR)),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(V2_MODEL_DIR.glob("*")) if path.is_file()
]

v2_manifest = {
    "project": PROJECT_NAME,
    "version": PROJECT_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "run_profile": V2_RUN_PROFILE,
    "settings": V2_SETTINGS,
    "python_version": platform.python_version(),
    "python_build": sys.version.replace("\n", " "),
    "direct_package_lock_sha256": PACKAGE_LOCK_SHA256,
    "resolved_environment_sha256": FULL_ENVIRONMENT_SHA256,
    "code_version_sha256": EXPECTED_CODE_SHA256,
    "analysis_fingerprint": ANALYSIS_FINGERPRINT,
    "cache_schema_version": CACHE_SCHEMA_VERSION,
    "full_environment": FROZEN_ENVIRONMENT_LINES,
    "packages": {name: safe_version(name) for name in PINNED_REQUIREMENTS},
    "cache_policy": "Legacy caches archived; reuse allowed only for exact strict fingerprint matches",
    "development_policy": "Entire stack contained independently inside each outer dependency-component fold",
    "bootstrap_policy": f"{V2_SETTINGS['bootstrap']} whole-component replicates; percentile 95% intervals",
    "capacity_ci_fraction": BOOTSTRAP_CAPACITY_FRACTION,
    "sensitivity_scenarios": SENSITIVITY_SCENARIOS,
    "selection_caveat": "Internal selection and comparison; independent validation required",
    "final_candidates": V2_FINAL_CANDIDATES,
    "calibration_methods": V2_CALIBRATION_METHODS,
    "internal_calibration_evidence": "Cross-fitted inside strict outer validation",
    "conformal_method": "Pathway-level component conformal calibration",
    "capacity_policy": V2_CAPACITY_POLICIES,
    "selective_policy": V2_SELECTIVE_POLICIES,
    "external_cohort_status": "Adapter executed; review required" if EXTERNAL_VALIDATION_RESULTS else "Not supplied",
    "model_artifacts": v2_model_artifact_manifest,
    "artifact_loading_policy": "Load only verified artefacts from this strict run and confirm SHA-256",
    "evidence_status": "Retrospective model development and internal validation",
    "limitations": [
        "PhysioNet development data use matched case-control sampling",
        "Compatibility partition was previously viewed and is non-confirmatory",
        "No independent all-admission cohort is included",
        "No prospective shadow or live clinical evaluation has been conducted",
    ],
}
atomic_write_json(V2_PROTOCOL_DIR / "reproducibility_manifest.json", v2_manifest)

v2_output_index_df = pd.DataFrame([
    {"category": "Internal development", "output": "Candidate summary", "path": str(V2_TABLE_DIR / "candidate_summary.csv")},
    {"category": "Strict stack", "output": "Outer-fold stack predictions", "path": str(V2_TABLE_DIR / "strict_stack_oof_patient.csv")},
    {"category": "Uncertainty", "output": "Component bootstrap 95% CIs", "path": str(V2_TABLE_DIR / "component_bootstrap_95ci.csv")},
    {"category": "Comparison", "output": "Paired stack-minus-single 95% CIs", "path": str(V2_TABLE_DIR / "paired_stack_vs_best_single_95ci.csv")},
    {"category": "Calibration", "output": "Cross-fitted internal calibration", "path": str(V2_TABLE_DIR / "crossfitted_internal_calibration_by_repeat.csv")},
    {"category": "Calibration", "output": "Deployment calibrator apparent fit", "path": str(V2_TABLE_DIR / "probability_calibrator_apparent_fit.csv")},
    {"category": "Sensitivity", "output": "Sensitivity inclusion audit", "path": str(V2_TABLE_DIR / "sensitivity_inclusion_audit.csv")},
    {"category": "Sensitivity", "output": "Strict sensitivity results", "path": str(V2_TABLE_DIR / "strict_sensitivity_results.csv")},
    {"category": "Uncertainty", "output": "Pathway-level component conformal calibration", "path": str(V2_TABLE_DIR / "component_conformal_calibration.csv")},
    {"category": "Operations", "output": "Locked capacity and selective policies", "path": str(V2_TABLE_DIR / "v2_locked_operating_policies.csv")},
    {"category": "Compatibility", "output": "Non-confirmatory compatibility metrics", "path": str(V2_TABLE_DIR / "v2_legacy_holdout_metrics_NOT_CONFIRMATORY.csv")},
    {"category": "External validation", "output": "Data contract", "path": str(V2_PROTOCOL_DIR / "external_all_admission_data_contract.csv")},
    {"category": "External validation", "output": "Statistical analysis plan", "path": str(V2_PROTOCOL_DIR / "HOI_TRUST_external_validation_SAP.md")},
    {"category": "Prospective", "output": "Shadow-mode schema", "path": str(V2_PROTOCOL_DIR / "shadow_mode_event_schema.csv")},
    {"category": "Safety", "output": "Clinical hazard log starter", "path": str(V2_PROTOCOL_DIR / "clinical_hazard_log_starter.csv")},
    {"category": "Manifest", "output": "Reproducibility manifest", "path": str(V2_PROTOCOL_DIR / "reproducibility_manifest.json")},
    {"category": "Manifest", "output": "Full resolved environment", "path": str(V2_PROTOCOL_DIR / "requirements_frozen_full.txt")},
])
atomic_write_csv(V2_DIR / "output_index.csv", v2_output_index_df)
display(v2_output_index_df)
print("HOI-TRUST analysis complete.")
print("Evidence status: retrospective model development and internal validation")


## 23. Retrospective surveillance dashboard

The dashboard is constructed at the end of the analytical workflow. It uses actual de-identified v2 scores from the compatibility set, assigns display-only record identifiers and excludes the source outcome, patient identifiers, matched-set identifiers and dependency-component identifiers. Only aggregate dashboard summaries are written to disk.


In [ ]:
# Retrospective dashboard built from de-identified model outputs

if not V2_LEGACY_RESULTS:
    raise RuntimeError(
        "Compatibility analysis outputs are unavailable. Run the preceding evaluation "
        "cells before creating the dashboard."
    )


PATHOGEN_DISPLAY_NAMES = {
    "DS_E_coli": "Drug-susceptible E. coli",
    "ESBL_E_coli": "ESBL E. coli",
    "DS_K_pneumoniae": "Drug-susceptible K. pneumoniae",
    "ESBL_K_pneumoniae": "ESBL K. pneumoniae",
    "C_diff": "C. difficile",
    "VSE_faecalis": "Vancomycin-susceptible E. faecalis",
    "VRE_faecium": "Vancomycin-resistant E. faecium",
    "MSSA": "MSSA",
    "MRSA": "MRSA",
    "DS_P_aeruginosa": "Drug-susceptible P. aeruginosa",
    "DR_P_aeruginosa": "Drug-resistant P. aeruginosa",
}


def review_band(percentile):
    if percentile <= 1:
        return "Top 1%"
    if percentile <= 5:
        return "Top 5%"
    if percentile <= 10:
        return "Top 10%"
    if percentile <= 20:
        return "Top 20%"
    return "Outside top 20%"


def model_reliance_context(pathway_value):
    reliance = v2_locked_candidate_importance_df.copy()
    if pathway_value != "All":
        reliance = reliance.loc[
            reliance["pathway"].eq(pathway_value.casefold())
        ]
    reliance = (
        reliance.groupby("domain", as_index=False)["mean_AUPRC_decrease"]
        .mean().sort_values("mean_AUPRC_decrease", ascending=False)
        .head(3)
    )
    return ", ".join(
        f"{row.domain} ({row.mean_AUPRC_decrease:.3f})"
        for row in reliance.itertuples(index=False)
    )


dashboard_frames = []
for pathway, result in V2_LEGACY_RESULTS.items():
    safe = pd.DataFrame({
        "pathway_key": pathway,
        "pathway": "Patient" if pathway == "patient" else "Environmental",
        "pathogen_code": result["run"].astype(str),
        "ranking_score": result["v2_raw_score"].astype(float),
        "model_score": result["v2_calibrated_score"].astype(float),
        "capacity_selected": result["capacity_selected"].astype(bool),
        "selective_status": np.where(result["selective_abstain"], "Deferred", "Retained"),
        "uncertainty": result["uncertainty_flag"].astype(str),
        "conformal_set": result["conformal_prediction_set"].astype(str),
    })
    safe["pathogen"] = safe["pathogen_code"].map(PATHOGEN_DISPLAY_NAMES).fillna(
        safe["pathogen_code"]
    )
    safe["review_rank"] = (
        safe.groupby("pathogen_code")["ranking_score"]
        .rank(method="min", ascending=False).astype(int)
    )
    group_size = safe.groupby("pathogen_code")["ranking_score"].transform("size")
    safe["review_percentile"] = safe["review_rank"] / group_size * 100
    safe["review_band"] = safe["review_percentile"].map(review_band)
    safe = safe.sort_values(
        ["ranking_score", "pathogen_code"], ascending=[False, True]
    ).reset_index(drop=True)
    prefix = "PAT" if pathway == "patient" else "ENV"
    safe["record_id"] = [f"{prefix}-{number:05d}" for number in range(1, len(safe) + 1)]
    dashboard_frames.append(safe)

V2_DASHBOARD_DATA = pd.concat(dashboard_frames, ignore_index=True)

# The dashboard contains real model outputs but excludes labels and source identifiers.
forbidden_dashboard_columns = {
    TARGET,
    "deidentified_patient_id",
    "pathway_patient_id",
    "matched_set_id",
    "dependency_component",
    "group_index",
    "covered",
}
assert not (forbidden_dashboard_columns & set(V2_DASHBOARD_DATA.columns))
assert len(V2_DASHBOARD_DATA) == sum(len(frame) for frame in V2_LEGACY_RESULTS.values())

dashboard_aggregate_df = (
    V2_DASHBOARD_DATA.groupby(
        ["pathway", "pathogen_code", "pathogen", "uncertainty"], as_index=False
    )
    .agg(
        observations=("record_id", "size"),
        mean_model_score=("model_score", "mean"),
        selected_for_review=("capacity_selected", "sum"),
    )
)
atomic_write_csv(V2_TABLE_DIR / "dashboard_aggregate_summary.csv", dashboard_aggregate_df)

# Static manuscript figure generated from the same dashboard source.
figure = plt.figure(figsize=(12, 9.2), facecolor="white")
grid = figure.add_gridspec(
    4, 2, height_ratios=[0.45, 0.72, 2.25, 1.55], hspace=0.48, wspace=0.28
)
title_axis = figure.add_subplot(grid[0, :])
title_axis.axis("off")
title_axis.text(
    0, 0.72, "Retrospective surveillance review", fontsize=20,
    weight="bold", color="#102A43",
)
title_axis.text(
    0, 0.10,
    "De-identified, non-confirmatory compatibility outputs; scores are not prospective absolute risks",
    fontsize=10.5, color="#486581",
)

card_axis = figure.add_subplot(grid[1, :])
card_axis.axis("off")
cards = [
    ("Records", f"{len(V2_DASHBOARD_DATA):,}"),
    ("Selected for review", f"{int(V2_DASHBOARD_DATA['capacity_selected'].sum()):,}"),
    ("Ambiguous outputs", f"{V2_DASHBOARD_DATA['uncertainty'].eq('Ambiguous').mean():.1%}"),
    ("Pathogen targets", f"{V2_DASHBOARD_DATA['pathogen_code'].nunique()}"),
]
for index, (label, value) in enumerate(cards):
    left = index * 0.245
    card_axis.add_patch(
        plt.Rectangle(
            (left, 0.05), 0.225, 0.88,
            facecolor="#F5F7FA", edgecolor="#D9E2EC",
        )
    )
    card_axis.text(left + 0.018, 0.66, label, fontsize=9.5, color="#627D98")
    card_axis.text(left + 0.018, 0.25, value, fontsize=17, weight="bold", color="#102A43")

score_axis = figure.add_subplot(grid[2, 0])
sns.histplot(
    data=V2_DASHBOARD_DATA,
    x="model_score",
    hue="pathway",
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=score_axis,
)
score_axis.set_title("Calibrated-score distribution")
score_axis.set_xlabel("Calibrated score in matched compatibility data")
score_axis.set_ylabel("Density")

uncertainty_axis = figure.add_subplot(grid[2, 1])
uncertainty_plot = (
    V2_DASHBOARD_DATA.groupby(["pathway", "uncertainty"])
    .size().rename("observations").reset_index()
)
uncertainty_plot["proportion"] = (
    uncertainty_plot["observations"]
    / uncertainty_plot.groupby("pathway")["observations"].transform("sum")
)
sns.barplot(
    data=uncertainty_plot,
    x="pathway",
    y="proportion",
    hue="uncertainty",
    ax=uncertainty_axis,
)
uncertainty_axis.set_title("Uncertainty composition")
uncertainty_axis.set_xlabel("")
uncertainty_axis.set_ylabel("Proportion")
uncertainty_axis.set_ylim(0, 1)
uncertainty_axis.yaxis.set_major_formatter(PercentFormatter(1.0))

queue_axis = figure.add_subplot(grid[3, :])
queue_axis.axis("off")
queue_preview = (
    V2_DASHBOARD_DATA.loc[V2_DASHBOARD_DATA["capacity_selected"]]
    .sort_values(["ranking_score", "record_id"], ascending=[False, True])
    .head(8)
    [["record_id", "pathway", "pathogen", "model_score", "review_percentile", "uncertainty"]]
    .copy()
)
queue_preview["model_score"] = queue_preview["model_score"].map(lambda value: f"{value:.3f}")
queue_preview["review_percentile"] = queue_preview["review_percentile"].map(
    lambda value: f"{value:.2f}%"
)
queue_axis.set_title("Illustrative ranked review queue", loc="left", fontsize=11, weight="bold")
table = queue_axis.table(
    cellText=queue_preview.values,
    colLabels=["Record", "Pathway", "Pathogen target", "Score", "Review percentile", "Uncertainty"],
    cellLoc="left",
    colLoc="left",
    loc="center",
    colWidths=[0.12, 0.12, 0.31, 0.10, 0.17, 0.14],
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.25)

figure.text(
    0.01,
    0.005,
    "Research prototype. Global grouped model reliance is descriptive; individual causal explanations are not provided.",
    fontsize=8.5,
    color="#627D98",
)
figure.savefig(
    V2_FIGURE_DIR / "results_dashboard_demonstrator.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()

print(
    f"Dashboard source prepared: {len(V2_DASHBOARD_DATA):,} "
    "real de-identified scored observations"
)
print("Only aggregate dashboard data were written to disk.")

if RUN_DASHBOARD:
    import ipywidgets as widgets
    import plotly.express as px
    from IPython.display import clear_output

    pathway_widget = widgets.Dropdown(
        options=["All", "Patient", "Environmental"],
        value="All",
        description="Pathway:",
    )
    pathogen_widget = widgets.Dropdown(
        options=[("All", "All")] + [
            (PATHOGEN_DISPLAY_NAMES.get(code, code), code)
            for code in sorted(V2_DASHBOARD_DATA["pathogen_code"].unique())
        ],
        value="All",
        description="Pathogen:",
    )
    uncertainty_widget = widgets.Dropdown(
        options=["All"] + sorted(V2_DASHBOARD_DATA["uncertainty"].unique().tolist()),
        value="All",
        description="Uncertainty:",
    )
    review_widget = widgets.Dropdown(
        options=["All records", "Selected queue", "Top 20%", "Top 10%", "Top 5%", "Top 1%"],
        value="Selected queue",
        description="Review view:",
    )
    search_widget = widgets.Text(
        value="", placeholder="Record ID", description="Find:"
    )
    output_widget = widgets.Output()

    def render_dashboard(*_):
        with output_widget:
            clear_output(wait=True)
            filtered = V2_DASHBOARD_DATA.copy()
            if pathway_widget.value != "All":
                filtered = filtered.loc[filtered["pathway"].eq(pathway_widget.value)]
            if pathogen_widget.value != "All":
                filtered = filtered.loc[
                    filtered["pathogen_code"].eq(pathogen_widget.value)
                ]
            if uncertainty_widget.value != "All":
                filtered = filtered.loc[
                    filtered["uncertainty"].eq(uncertainty_widget.value)
                ]
            if review_widget.value == "Selected queue":
                filtered = filtered.loc[filtered["capacity_selected"]]
            elif review_widget.value.startswith("Top"):
                limit = float(review_widget.value.split()[1].rstrip("%"))
                filtered = filtered.loc[filtered["review_percentile"].le(limit)]
            if search_widget.value.strip():
                filtered = filtered.loc[
                    filtered["record_id"].str.contains(
                        search_widget.value.strip(), case=False, regex=False
                    )
                ]
            filtered = filtered.sort_values(
                ["ranking_score", "record_id"], ascending=[False, True]
            )

            total = len(filtered)
            mean_score = filtered["model_score"].mean() if total else np.nan
            ambiguous = filtered["uncertainty"].eq("Ambiguous").mean() if total else np.nan
            selected_count = int(filtered["capacity_selected"].sum()) if total else 0
            pathogens = filtered["pathogen_code"].nunique() if total else 0
            reliance_text = model_reliance_context(pathway_widget.value)

            display(HTML(f"""
            <div style='font-family:Arial,sans-serif;border:1px solid #d9e2ec;border-radius:14px;padding:20px;background:#f8fafc'>
              <div style='display:flex;justify-content:space-between;gap:16px'>
                <div>
                  <div style='font-size:12px;letter-spacing:.10em;color:#486581;font-weight:700'>HOI-TRUST</div>
                  <div style='font-size:25px;font-weight:750;color:#102a43'>Retrospective Surveillance Review</div>
                  <div style='color:#627d98;margin-top:4px'>De-identified, non-confirmatory compatibility outputs</div>
                </div>
                <div style='background:#e6f0ff;color:#174ea6;padding:8px 12px;border-radius:10px;font-size:12px;height:max-content'>Research prototype</div>
              </div>
              <div style='display:grid;grid-template-columns:repeat(4,minmax(130px,1fr));gap:12px;margin-top:18px'>
                <div style='background:white;padding:14px;border-radius:12px'><div style='color:#627d98;font-size:12px'>Visible records</div><div style='font-size:24px;font-weight:700;color:#102a43'>{total:,}</div></div>
                <div style='background:white;padding:14px;border-radius:12px'><div style='color:#627d98;font-size:12px'>Mean calibrated score</div><div style='font-size:24px;font-weight:700;color:#102a43'>{mean_score:.3f}</div></div>
                <div style='background:white;padding:14px;border-radius:12px'><div style='color:#627d98;font-size:12px'>Ambiguous outputs</div><div style='font-size:24px;font-weight:700;color:#102a43'>{ambiguous:.1%}</div></div>
                <div style='background:white;padding:14px;border-radius:12px'><div style='color:#627d98;font-size:12px'>Selected / pathogens</div><div style='font-size:24px;font-weight:700;color:#102a43'>{selected_count:,} / {pathogens}</div></div>
              </div>
              <div style='background:#fff7ed;border-left:4px solid #d97706;padding:10px 12px;margin-top:14px;color:#7c2d12;font-size:12px'>Scores support ranking within matched retrospective data and are not prospective absolute infection risks.</div>
              <div style='background:white;padding:10px 12px;margin-top:10px;color:#486581;font-size:12px'><b>Global grouped reliance:</b> {reliance_text}. This is associative model context, not an individual or causal explanation.</div>
            </div>
            """))

            if not total:
                display(HTML("<p style='padding:16px'>No records match the selected filters.</p>"))
                return

            chart = px.scatter(
                filtered,
                x="review_percentile",
                y="model_score",
                color="uncertainty",
                symbol="pathway",
                hover_data=["record_id", "pathogen", "review_band", "conformal_set"],
                labels={
                    "review_percentile": "Within-pathogen review percentile (lower = higher priority)",
                    "model_score": "Calibrated model score",
                },
                title="Priority landscape",
            )
            chart.update_layout(height=420, margin=dict(l=30, r=20, t=60, b=30))
            chart.update_xaxes(autorange="reversed")
            chart.show()

            columns = [
                "record_id",
                "pathway",
                "pathogen",
                "model_score",
                "review_percentile",
                "review_band",
                "capacity_selected",
                "uncertainty",
                "conformal_set",
                "selective_status",
            ]
            display(
                filtered[columns].head(30).style.format({
                    "model_score": "{:.3f}",
                    "review_percentile": "{:.2f}",
                })
            )

    for widget in [
        pathway_widget,
        pathogen_widget,
        uncertainty_widget,
        review_widget,
        search_widget,
    ]:
        widget.observe(render_dashboard, names="value")
    controls = widgets.HBox(
        [pathway_widget, pathogen_widget, uncertainty_widget, review_widget, search_widget],
        layout=widgets.Layout(flex_flow="row wrap", gap="8px"),
    )
    display(controls, output_widget)
    render_dashboard()
else:
    print("RUN_DASHBOARD is False; dashboard rendering skipped.")



## 24. Submission artefact audit


In [ ]:
# Final submission audit

required_figures = [
    "results_primary_component_bootstrap.png",
    "results_sensitivity_analyses.png",
    "candidate_comparison.png",
    "pathogen_context_sensitivity.png",
    "operating_policy_curves.png",
    "results_conformal_compatibility.png",
    "results_within_pathogen_heterogeneity.png",
    "results_grouped_permutation_reliance.png",
    "results_dashboard_demonstrator.png",
]
required_tables = [
    "candidate_summary.csv",
    "component_bootstrap_95ci.csv",
    "paired_stack_vs_best_single_95ci.csv",
    "crossfitted_internal_calibration_by_repeat.csv",
    "sensitivity_inclusion_audit.csv",
    "strict_sensitivity_results.csv",
    "probability_calibrator_apparent_fit.csv",
    "v2_locked_operating_policies.csv",
    "v2_legacy_holdout_metrics_NOT_CONFIRMATORY.csv",
    "v2_legacy_uncertainty_metrics_NOT_CONFIRMATORY.csv",
    "legacy_within_pathogen_metrics_NOT_CONFIRMATORY.csv",
    "v2_locked_candidate_grouped_permutation_importance.csv",
    "v2_stack_meta_coefficients.csv",
    "dashboard_aggregate_summary.csv",
]

submission_audit_df = pd.DataFrame(
    [
        {"artifact_type": "figure", "artifact": name, "exists": (V2_FIGURE_DIR / name).exists()}
        for name in required_figures
    ]
    + [
        {"artifact_type": "table", "artifact": name, "exists": (V2_TABLE_DIR / name).exists()}
        for name in required_tables
    ]
)

if not submission_audit_df["exists"].all():
    missing = submission_audit_df.loc[~submission_audit_df["exists"], "artifact"].tolist()
    raise RuntimeError("Submission artefacts are missing: " + ", ".join(missing))

if not environment_audit_df["matches_pin"].all():
    raise RuntimeError("Environment audit is not clean.")
if platform.python_version() != PINNED_PYTHON_VERSION:
    raise RuntimeError("Python changed during execution.")
if not CACHE_INITIALISED_MARKER.exists():
    raise RuntimeError("Legacy-cache archive policy was not initialised.")
if (component_bootstrap_ci_df["bootstrap_replicates"] < V2_SETTINGS["bootstrap"]).any():
    raise RuntimeError("A component-bootstrap interval has too few replicates.")

executed_scenarios = set(
    sensitivity_inclusion_audit_df.loc[
        sensitivity_inclusion_audit_df["status"].eq("executed"), "scenario"
    ]
)
required_executed = set(SENSITIVITY_SCENARIOS) - {"exclude_resolved_any_abx_summary"}
if not required_executed.issubset(executed_scenarios):
    raise RuntimeError("One or more required sensitivity scenarios did not execute.")
if not (
    (sensitivity_inclusion_audit_df["pathway"].eq("patient"))
    & (sensitivity_inclusion_audit_df["scenario"].eq("exclude_resolved_any_abx_summary"))
    & (sensitivity_inclusion_audit_df["status"].eq("executed"))
).any():
    raise RuntimeError("Patient any_abx sensitivity did not execute.")

for pathway, predictions in V2_STACK_PREDICTIONS.items():
    expected_rows = len(V2_PARTITIONS[pathway]["development"]) * V2_SETTINGS["outer_repeats"]
    if len(predictions) != expected_rows:
        raise RuntimeError(f"Strict stack OOF row count failed for {pathway}.")
    if predictions[["probability", "calibrated_probability", "best_single_probability"]].isna().any().any():
        raise RuntimeError(f"Strict stack contains missing predictions for {pathway}.")
    if V2_STACK_FOLDS[pathway]["component_overlap"].astype(int).ne(0).any():
        raise RuntimeError(f"Outer-fold component leakage found for {pathway}.")

if "V2_DASHBOARD_DATA" in globals():
    assert TARGET not in V2_DASHBOARD_DATA.columns
    assert not ({"deidentified_patient_id", "matched_set_id", "dependency_component"} & set(V2_DASHBOARD_DATA.columns))

EXECUTION_COMPLETED_UTC = datetime.now(timezone.utc).isoformat()
final_run_record = {
    "project": PROJECT_NAME,
    "project_version": PROJECT_VERSION,
    "execution_completed_utc": EXECUTION_COMPLETED_UTC,
    "python_version": platform.python_version(),
    "python_build": sys.version.replace("\n", " "),
    "direct_package_lock_sha256": PACKAGE_LOCK_SHA256,
    "resolved_environment_sha256": FULL_ENVIRONMENT_SHA256,
    "code_version_sha256": EXPECTED_CODE_SHA256,
    "analysis_fingerprint": ANALYSIS_FINGERPRINT,
    "cache_schema_version": CACHE_SCHEMA_VERSION,
    "notebook_expected_name": "HOI_TRUST_submission_ready.ipynb",
    "notebook_sha256": "Pending post-save verification cell",
    "all_cells_reached_final_audit": True,
    "environment_warnings": 0,
    "submission_artifacts_complete": True,
}
atomic_write_json(V2_PROTOCOL_DIR / "final_run_record.json", final_run_record)
atomic_write_csv(V2_DIR / "submission_artifact_audit.csv", submission_audit_df)
display(submission_audit_df)
display(pd.DataFrame([final_run_record]))
print("FINAL AUDIT PASSED: all required analyses and artefacts completed in the pinned environment.")
print("Save the notebook now, then follow the manual post-save hash cell.")


### Manual post-save SHA-256 finalisation

After the final audit passes:

1. Save the completed notebook under the exact name `HOI_TRUST_submission_ready.ipynb`.
2. Confirm `HOI_TRUST_NOTEBOOK_PATH` points to that saved file.
3. Set `FINALISE_SAVED_NOTEBOOK_HASH = True` and run only the next cell.
4. Do **not** save the notebook again after running it. The external `final_submission_manifest.json` and `final_notebook_sha256.txt` then describe the saved bytes exactly.

An executable notebook cannot embed its own final hash in its saved outputs without changing the bytes being hashed; the external signed-off manifest is the reproducible solution.


In [ ]:
FINALISE_SAVED_NOTEBOOK_HASH = False

if not FINALISE_SAVED_NOTEBOOK_HASH:
    print("Post-save hash not requested. Save the completed notebook, set the flag to True, and run only this cell.")
else:
    if NOTEBOOK_PATH.name != "HOI_TRUST_submission_ready.ipynb":
        raise RuntimeError(f"Unexpected notebook filename: {NOTEBOOK_PATH.name}")
    if not NOTEBOOK_PATH.exists():
        raise FileNotFoundError(
            f"Saved notebook not found at {NOTEBOOK_PATH}. Set HOI_TRUST_NOTEBOOK_PATH correctly."
        )
    final_notebook_sha256 = sha256_file(NOTEBOOK_PATH)
    final_manifest = json.loads(
        (V2_PROTOCOL_DIR / "final_run_record.json").read_text(encoding="utf-8")
    )
    final_manifest.update({
        "notebook_path": str(NOTEBOOK_PATH),
        "notebook_bytes": NOTEBOOK_PATH.stat().st_size,
        "notebook_sha256": final_notebook_sha256,
        "hash_recorded_utc": datetime.now(timezone.utc).isoformat(),
        "instruction": "Do not save or modify the notebook after this hash was calculated.",
    })
    atomic_write_json(V2_PROTOCOL_DIR / "final_submission_manifest.json", final_manifest)
    (V2_PROTOCOL_DIR / "final_notebook_sha256.txt").write_text(
        f"{final_notebook_sha256}  {NOTEBOOK_PATH.name}\n", encoding="utf-8"
    )
    print("Final saved-notebook SHA-256:", final_notebook_sha256)
    print("Recorded in:", V2_PROTOCOL_DIR / "final_submission_manifest.json")


## Reporting boundary

The completed evidence supports retrospective model development and dependency-aware internal validation. The compatibility set is non-confirmatory, and the matched analytical distribution does not provide hospital-population absolute-risk calibration. External all-admission, temporal and prospective evaluation remain future work.
